In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/saiharish0088/subject-wise/parkinsons_subject_wise_split.csv
/kaggle/input/datasets/saiharish0088/pppppppppp/parkinsons_updrs.csv


In [2]:
# ============================================================
# FORCE CPU — STABLE VQC TRAINING
# ============================================================

import os

# Prevent CUDA from being selected anywhere
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import numpy as np
import pandas as pd
import torch

DEVICE = torch.device("cpu")

# Force all PyTorch operations to CPU
torch.set_default_device("cpu")

print("=" * 70)
print("VQC DEVICE CONFIGURATION")
print("=" * 70)
print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("WARNING: CUDA is visible, but VQC will NOT use it.")
else:
    print("✓ CUDA unavailable — CPU VQC mode")

torch.manual_seed(123)
np.random.seed(123)

VQC DEVICE CONFIGURATION
Device: cpu
CUDA available: False
✓ CUDA unavailable — CPU VQC mode


In [3]:
# ============================================================
# VQC — FORCE CPU MODE
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import torch

DEVICE = torch.device("cpu")

# Reproducibility
SEED = 123
np.random.seed(SEED)
torch.manual_seed(SEED)

print("=" * 70)
print("VQC DEVICE CONFIGURATION")
print("=" * 70)
print("Device:", DEVICE)
print("✓ VQC will run entirely on CPU")
print("✓ CUDA will NOT be used")

VQC DEVICE CONFIGURATION
Device: cpu
✓ VQC will run entirely on CPU
✓ CUDA will NOT be used


In [4]:
!pip install -q "qiskit==2.4.2" "qiskit-aer==0.17.2" "qiskit-machine-learning" "qiskit-algorithms"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 48.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 73.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.3/282.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.2 MB/s eta 0:00:00


In [19]:
# =============================================================================
# PHASE 21M — FINALIZE EXACT PHASE 21L SUBJECT-WISE SPLIT
# =============================================================================

print("=" * 80)
print("FINALIZING PHASE 21L SUBJECT-WISE SPLIT")
print("=" * 80)

SUBJECT_COL = "subject#"
TARGET_COL = "severity"
SPLIT_COL = "split"

# -------------------------------------------------------------------------
# 1. Make sure subject lists from Phase 21L exist
# -------------------------------------------------------------------------

required_vars = ["dev_subjects", "val_subjects"]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise RuntimeError(
        f"Missing Phase 21L subject lists: {missing}\n"
        "Run the Phase 21L training/preparation cell first."
    )

dev_subjects = set(map(str, dev_subjects))
val_subjects = set(map(str, val_subjects))

# Locked-test subjects come directly from split == test
locked_subjects = set(
    df_work.loc[
        df_work[SPLIT_COL].astype(str).str.lower().eq("test"),
        SUBJECT_COL
    ].astype(str)
)

# -------------------------------------------------------------------------
# 2. Subject leakage check
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SUBJECT LEAKAGE CHECK")
print("=" * 80)

print("Development subjects:", len(dev_subjects))
print("Validation subjects :", len(val_subjects))
print("Locked-test subjects:", len(locked_subjects))

print(
    "Development / validation overlap:",
    dev_subjects.intersection(val_subjects)
)

print(
    "Development / test overlap:",
    dev_subjects.intersection(locked_subjects)
)

print(
    "Validation / test overlap:",
    val_subjects.intersection(locked_subjects)
)

assert len(dev_subjects.intersection(val_subjects)) == 0
assert len(dev_subjects.intersection(locked_subjects)) == 0
assert len(val_subjects.intersection(locked_subjects)) == 0

print("✓ ZERO SUBJECT LEAKAGE")

# -------------------------------------------------------------------------
# 3. Normalize subject column
# -------------------------------------------------------------------------

df_work = df_work.copy()

df_work[SUBJECT_COL] = df_work[SUBJECT_COL].astype(str)

# -------------------------------------------------------------------------
# 4. Construct the three datasets
# -------------------------------------------------------------------------

development_df = df_work[
    df_work[SUBJECT_COL].isin(dev_subjects)
].copy()

validation_df = df_work[
    df_work[SUBJECT_COL].isin(val_subjects)
].copy()

locked_test_df = df_work[
    df_work[SUBJECT_COL].isin(locked_subjects)
    &
    df_work[SPLIT_COL].astype(str).str.lower().eq("test")
].copy()

# -------------------------------------------------------------------------
# 5. Verify that every training-pool row is accounted for
# -------------------------------------------------------------------------

training_pool_df = df_work[
    df_work[SPLIT_COL].astype(str).str.lower().eq("train")
].copy()

print("\n" + "=" * 80)
print("FINAL PHASE 21L SPLIT")
print("=" * 80)

print("Training pool rows :", len(training_pool_df))
print("Development rows   :", len(development_df))
print("Validation rows    :", len(validation_df))
print("Locked-test rows   :", len(locked_test_df))

# Development + validation must exactly equal training pool
assert (
    len(development_df) + len(validation_df)
    == len(training_pool_df)
), (
    f"Development + validation != training pool: "
    f"{len(development_df)} + {len(validation_df)} "
    f"!= {len(training_pool_df)}"
)

# Locked test must be exactly the split=test rows
assert len(locked_test_df) == len(
    df_work[df_work[SPLIT_COL].astype(str).str.lower().eq("test")]
)

print("✓ Development + validation = complete training pool")
print("✓ Locked test = complete original test split")

# -------------------------------------------------------------------------
# 6. DO NOT assert 3588 / 997
# -------------------------------------------------------------------------
#
# Those numbers belonged to the previous subject allocation.
# The current Phase 21L subject lists produce:
#
#       Development = 3602
#       Validation  = 983
#
# That is valid because:
#
#       3602 + 983 = 4585
#
# -------------------------------------------------------------------------

print("\nCurrent Phase 21L subject-wise allocation:")
print("Development:", len(development_df))
print("Validation :", len(validation_df))
print("Test       :", len(locked_test_df))

# -------------------------------------------------------------------------
# 7. Target verification
# -------------------------------------------------------------------------

for name, frame in [
    ("development", development_df),
    ("validation", validation_df),
    ("locked_test", locked_test_df),
]:

    if TARGET_COL not in frame.columns:
        raise RuntimeError(
            f"{TARGET_COL!r} missing from {name} dataframe."
        )

    if frame[TARGET_COL].isna().any():
        raise RuntimeError(
            f"{name} contains missing severity values."
        )

    frame[TARGET_COL] = frame[TARGET_COL].astype(int)

# -------------------------------------------------------------------------
# 8. Create target arrays
# -------------------------------------------------------------------------

y_dev_np = development_df[TARGET_COL].to_numpy(dtype="int64")
y_val_np = validation_df[TARGET_COL].to_numpy(dtype="int64")
y_test_np = locked_test_df[TARGET_COL].to_numpy(dtype="int64")

print("\nTarget shapes:")
print("Development:", y_dev_np.shape)
print("Validation :", y_val_np.shape)
print("Locked test:", y_test_np.shape)

print("\nClass counts:")
print("Development:", np.bincount(y_dev_np))
print("Validation :", np.bincount(y_val_np))
print("Locked test:", np.bincount(y_test_np))

# -------------------------------------------------------------------------
# 9. Save exact subject lists for future Phase 21M/21N cells
# -------------------------------------------------------------------------

PHASE21M_DIR = "/kaggle/working/phase21L_outputs"
os.makedirs(PHASE21M_DIR, exist_ok=True)

np.save(
    os.path.join(PHASE21M_DIR, "phase21L_dev_subjects.npy"),
    np.array(sorted(dev_subjects), dtype=str)
)

np.save(
    os.path.join(PHASE21M_DIR, "phase21L_val_subjects.npy"),
    np.array(sorted(val_subjects), dtype=str)
)

np.save(
    os.path.join(PHASE21M_DIR, "phase21L_test_subjects.npy"),
    np.array(sorted(locked_subjects), dtype=str)
)

print("\n" + "=" * 80)
print("✓ PHASE 21M SUBJECT SPLIT FINALIZED")
print("=" * 80)

print("""
IMPORTANT:
✓ Do NOT force 3588 / 997
✓ Current Phase 21L split is 3602 / 983
✓ 3602 + 983 = 4585 training rows
✓ Locked test remains 1290 rows
✓ Zero subject leakage
✓ Locked test remains untouched
✓ No hyperparameter tuning performed
""")

FINALIZING PHASE 21L SUBJECT-WISE SPLIT

SUBJECT LEAKAGE CHECK
Development subjects: 26
Validation subjects : 7
Locked-test subjects: 9
Development / validation overlap: set()
Development / test overlap: set()
Validation / test overlap: set()
✓ ZERO SUBJECT LEAKAGE

FINAL PHASE 21L SPLIT
Training pool rows : 4585
Development rows   : 3602
Validation rows    : 983
Locked-test rows   : 1290
✓ Development + validation = complete training pool
✓ Locked test = complete original test split

Current Phase 21L subject-wise allocation:
Development: 3602
Validation : 983
Test       : 1290

Target shapes:
Development: (3602,)
Validation : (983,)
Locked test: (1290,)

Class counts:
Development: [2348 1254]
Validation : [589 394]
Locked test: [797 493]

✓ PHASE 21M SUBJECT SPLIT FINALIZED

IMPORTANT:
✓ Do NOT force 3588 / 997
✓ Current Phase 21L split is 3602 / 983
✓ 3602 + 983 = 4585 training rows
✓ Locked test remains 1290 rows
✓ Zero subject leakage
✓ Locked test remains untouched
✓ No hyperpara

In [20]:
# =============================================================================
# PHASE 21M — FROZEN PHASE 21L HYBRID VQC LOCKED-TEST EVALUATION
# =============================================================================
# IMPORTANT:
#   - CPU ONLY
#   - NO TRAINING
#   - NO OPTIMIZER
#   - NO HYPERPARAMETER TUNING
#   - LOCKED TEST USED ONLY HERE
#   - PREPROCESSING FIT ONLY ON DEVELOPMENT
# =============================================================================

import os
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# CPU ONLY
# -----------------------------------------------------------------------------

os.environ["CUDA_VISIBLE_DEVICES"] = ""

DEVICE = torch.device("cpu")
torch.set_default_dtype(torch.float32)

print("=" * 100)
print("PHASE 21M — FROZEN PHASE 21L HYBRID VQC LOCKED TEST EVALUATION")
print("=" * 100)
print("Device:", DEVICE)
print("✓ CPU-only")
print("✓ No training")
print("✓ No optimizer")
print("✓ No hyperparameter tuning")
print()

# =============================================================================
# PATHS
# =============================================================================

MODEL_PATH = "/kaggle/working/phase21L_outputs/phase21L_best_hybrid_vqc.pt"

# Change these two only if your filenames are different.
ORIGINAL_DATA_PATH = "/kaggle/input/parkinsons-updrs/telemonitoring_parkinsons.csv"
SPLIT_DATA_PATH    = "/kaggle/input/parkinsons-updrs/phase21_split.csv"

OUTPUT_DIR = "/kaggle/working/phase21M_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# COLUMN DEFINITIONS
# =============================================================================

SUBJECT_COL = "subject#"
TARGET_COL = "severity"
SPLIT_COL = "split"

ALL_FEATURES = [
    "age",
    "sex",
    "test_time",
    "motor_UPDRS",
    "Jitter(%)",
    "Jitter(Abs)",
    "Jitter:RAP",
    "Jitter:PPQ5",
    "Jitter:DDP",
    "Shimmer",
    "Shimmer(dB)",
    "Shimmer:APQ3",
    "Shimmer:APQ5",
    "Shimmer:APQ11",
    "Shimmer:DDA",
    "NHR",
    "HNR",
    "RPDE",
    "DFA",
    "PPE",
]

# =============================================================================
# LOAD DATA
# =============================================================================

print("=" * 100)
print("LOADING SOURCE DATA")
print("=" * 100)

df = pd.read_csv(ORIGINAL_DATA_PATH)
split_df = pd.read_csv(SPLIT_DATA_PATH)

print("Original dataset:", df.shape)
print("Split dataset   :", split_df.shape)

required_original = set([SUBJECT_COL] + ALL_FEATURES)
required_split = set([SUBJECT_COL, TARGET_COL, SPLIT_COL])

missing_original = required_original - set(df.columns)
missing_split = required_split - set(split_df.columns)

if missing_original:
    raise RuntimeError(
        f"Missing columns from original dataset: {sorted(missing_original)}"
    )

if missing_split:
    raise RuntimeError(
        f"Missing columns from split dataset: {sorted(missing_split)}"
    )

df[SUBJECT_COL] = df[SUBJECT_COL].astype(str)
split_df[SUBJECT_COL] = split_df[SUBJECT_COL].astype(str)

# =============================================================================
# IMPORTANT:
# severity is ROW LEVEL.
# Therefore DO NOT merge severity using subject# alone.
#
# The split file must contain one row per original data row, OR contain
# sufficient row-identifying information to reconstruct the target.
# =============================================================================

print()
print("=" * 100)
print("CHECKING SPLIT INFORMATION")
print("=" * 100)

print("Unique split subjects:", split_df[SUBJECT_COL].nunique())

# -----------------------------------------------------------------------------
# If split file has exactly 5875 rows, align row-by-row.
# This is the format used by the Phase 21L preparation.
# -----------------------------------------------------------------------------

if len(split_df) == len(df):

    print("✓ Split file has exactly the same number of rows.")
    print("✓ Attaching severity/split row-by-row.")

    # Preserve original ordering.
    df_work = df.copy()

    df_work[TARGET_COL] = pd.to_numeric(
        split_df[TARGET_COL],
        errors="raise"
    ).astype(int).to_numpy()

    df_work[SPLIT_COL] = (
        split_df[SPLIT_COL]
        .astype(str)
        .str.lower()
        .to_numpy()
    )

else:

    # -------------------------------------------------------------------------
    # If the split file contains only subject-level information for split,
    # severity cannot safely be merged by subject.
    #
    # However, if the current notebook already has df_work from Phase 21L,
    # use it instead.
    # -------------------------------------------------------------------------

    if "df_work" in globals() and len(df_work) == len(df):

        print(
            "✓ Reusing existing Phase 21L row-level df_work."
        )

        if TARGET_COL not in df_work.columns:
            raise RuntimeError(
                "Existing df_work does not contain row-level severity."
            )

        if SPLIT_COL not in df_work.columns:
            raise RuntimeError(
                "Existing df_work does not contain split."
            )

        df_work = df_work.copy()
        df_work[SUBJECT_COL] = df_work[SUBJECT_COL].astype(str)

    else:
        raise RuntimeError(
            "\nCannot safely reconstruct row-level severity.\n\n"
            f"Original rows = {len(df)}\n"
            f"Split rows    = {len(split_df)}\n\n"
            "The split file does not contain one row per original observation, "
            "and severity varies within subjects.\n"
            "Use the row-level Phase 21L dataframe or the original 5875-row "
            "split/target file."
        )

# =============================================================================
# REUSE EXACT PHASE 21L SUBJECT SPLIT
# =============================================================================

print()
print("=" * 100)
print("REUSING PHASE 21L SUBJECT SPLIT")
print("=" * 100)

if "dev_subjects" not in globals() or "val_subjects" not in globals():

    DEV_SUBJECTS_PATH = (
        "/kaggle/working/phase21L_outputs/phase21L_dev_subjects.npy"
    )

    VAL_SUBJECTS_PATH = (
        "/kaggle/working/phase21L_outputs/phase21L_val_subjects.npy"
    )

    TEST_SUBJECTS_PATH = (
        "/kaggle/working/phase21L_outputs/phase21L_test_subjects.npy"
    )

    if not os.path.exists(DEV_SUBJECTS_PATH):
        raise RuntimeError(
            "Phase 21L development subject file not found."
        )

    if not os.path.exists(VAL_SUBJECTS_PATH):
        raise RuntimeError(
            "Phase 21L validation subject file not found."
        )

    dev_subjects = set(
        np.load(DEV_SUBJECTS_PATH, allow_pickle=True).astype(str)
    )

    val_subjects = set(
        np.load(VAL_SUBJECTS_PATH, allow_pickle=True).astype(str)
    )

    if os.path.exists(TEST_SUBJECTS_PATH):
        test_subjects = set(
            np.load(TEST_SUBJECTS_PATH, allow_pickle=True).astype(str)
        )
    else:
        test_subjects = set(
            df_work.loc[
                df_work[SPLIT_COL].eq("test"),
                SUBJECT_COL
            ].astype(str)
        )

else:

    dev_subjects = set(map(str, dev_subjects))
    val_subjects = set(map(str, val_subjects))

    test_subjects = set(
        df_work.loc[
            df_work[SPLIT_COL].eq("test"),
            SUBJECT_COL
        ].astype(str)
    )

print("Development subjects:", len(dev_subjects))
print("Validation subjects :", len(val_subjects))
print("Locked-test subjects:", len(test_subjects))

# =============================================================================
# LEAKAGE CHECK
# =============================================================================

assert len(dev_subjects & val_subjects) == 0
assert len(dev_subjects & test_subjects) == 0
assert len(val_subjects & test_subjects) == 0

print()
print("=" * 100)
print("SUBJECT LEAKAGE CHECK")
print("=" * 100)

print("Development / validation:", dev_subjects & val_subjects)
print("Development / test      :", dev_subjects & test_subjects)
print("Validation / test       :", val_subjects & test_subjects)

print("✓ ZERO SUBJECT LEAKAGE")

# =============================================================================
# BUILD DATASETS
# =============================================================================

development_df = df_work[
    df_work[SUBJECT_COL].isin(dev_subjects)
].copy()

validation_df = df_work[
    df_work[SUBJECT_COL].isin(val_subjects)
].copy()

locked_test_df = df_work[
    df_work[SUBJECT_COL].isin(test_subjects)
    &
    df_work[SPLIT_COL].eq("test")
].copy()

print()
print("=" * 100)
print("FINAL DATASETS")
print("=" * 100)

print("Development:", development_df.shape)
print("Validation :", validation_df.shape)
print("Locked test:", locked_test_df.shape)

if len(locked_test_df) != 1290:
    raise RuntimeError(
        f"Expected 1290 locked-test rows, got {len(locked_test_df)}"
    )

if locked_test_df[SUBJECT_COL].nunique() != 9:
    raise RuntimeError(
        f"Expected 9 locked-test subjects, got "
        f"{locked_test_df[SUBJECT_COL].nunique()}"
    )

print("✓ Correct 1290 locked-test rows")
print("✓ Correct 9 locked-test subjects")

# =============================================================================
# TARGETS
# =============================================================================

y_dev = development_df[TARGET_COL].astype(int).to_numpy()
y_val = validation_df[TARGET_COL].astype(int).to_numpy()
y_test = locked_test_df[TARGET_COL].astype(int).to_numpy()

print()
print("Development classes:", np.bincount(y_dev))
print("Validation classes :", np.bincount(y_val))
print("Locked-test classes:", np.bincount(y_test))

# =============================================================================
# PHASE 21L ANOVA12
# =============================================================================
#
# IMPORTANT:
# The winning model was:
#
# Input : ANOVA12
# Model : B_RYRZ_D2_LR003
# Seed  : 2024
#
# We reproduce ANOVA selection using DEVELOPMENT ONLY.
#
# =============================================================================

print()
print("=" * 100)
print("RECONSTRUCTING PHASE 21L ANOVA12 PREPROCESSING")
print("=" * 100)

X_dev_raw = development_df[ALL_FEATURES].copy()
X_val_raw = validation_df[ALL_FEATURES].copy()
X_test_raw = locked_test_df[ALL_FEATURES].copy()

# -----------------------------------------------------------------------------
# Imputer: DEVELOPMENT ONLY
# -----------------------------------------------------------------------------

imputer = SimpleImputer(strategy="median")

X_dev_imp = imputer.fit_transform(X_dev_raw)
X_val_imp = imputer.transform(X_val_raw)
X_test_imp = imputer.transform(X_test_raw)

print("✓ Imputer fitted on development only")

# -----------------------------------------------------------------------------
# Scaling: DEVELOPMENT ONLY
# -----------------------------------------------------------------------------

scaler = StandardScaler()

X_dev_scaled = scaler.fit_transform(X_dev_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

print("✓ StandardScaler fitted on development only")

# -----------------------------------------------------------------------------
# ANOVA
# -----------------------------------------------------------------------------

F_values, p_values = f_classif(
    X_dev_scaled,
    y_dev
)

anova_table = pd.DataFrame({
    "feature": ALL_FEATURES,
    "F": F_values,
    "p": p_values
})

anova_table = anova_table.sort_values(
    "F",
    ascending=False
).reset_index(drop=True)

ANOVA_FEATURES = anova_table.head(12)["feature"].tolist()

print()
print("Selected ANOVA12 features:")

for i, feature in enumerate(ANOVA_FEATURES, 1):
    print(f"{i:2d}. {feature}")

# =============================================================================
# TRANSFORM TO ANOVA12
# =============================================================================

feature_indices = [
    ALL_FEATURES.index(feature)
    for feature in ANOVA_FEATURES
]

X_dev_12 = X_dev_scaled[:, feature_indices].astype(np.float32)
X_val_12 = X_val_scaled[:, feature_indices].astype(np.float32)
X_test_12 = X_test_scaled[:, feature_indices].astype(np.float32)

print()
print("ANOVA12 matrices:")
print("Development:", X_dev_12.shape)
print("Validation :", X_val_12.shape)
print("Locked test:", X_test_12.shape)

assert X_dev_12.shape[1] == 12
assert X_val_12.shape[1] == 12
assert X_test_12.shape[1] == 12

# =============================================================================
# LOAD FROZEN CHECKPOINT
# =============================================================================

print()
print("=" * 100)
print("LOADING FROZEN PHASE 21L CHECKPOINT")
print("=" * 100)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Frozen Phase 21L model not found:\n{MODEL_PATH}"
    )

# Trusted local checkpoint created by your own Phase 21L training.
checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

print("✓ Checkpoint loaded")
print("Checkpoint type:", type(checkpoint))

if not isinstance(checkpoint, dict):
    raise RuntimeError(
        "Unexpected Phase 21L checkpoint format."
    )

required_keys = [
    "model_state_dict",
    "best_info",
    "input_dim",
    "n_qubits",
    "depth",
    "encoding",
    "entangle",
]

missing_keys = [
    k for k in required_keys
    if k not in checkpoint
]

if missing_keys:
    raise RuntimeError(
        f"Checkpoint missing keys: {missing_keys}"
    )

print("Checkpoint keys:")
for key in required_keys:
    print(" ✓", key)

INPUT_DIM = int(checkpoint["input_dim"])
N_QUBITS = int(checkpoint["n_qubits"])
DEPTH = int(checkpoint["depth"])
ENCODING = checkpoint["encoding"]
ENTANGLE = bool(checkpoint["entangle"])

print()
print("Frozen configuration:")
print("Input dimension :", INPUT_DIM)
print("Qubits          :", N_QUBITS)
print("Depth           :", DEPTH)
print("Encoding        :", ENCODING)
print("Entanglement    :", ENTANGLE)

print()
print("Best information:")
print(checkpoint["best_info"])

if INPUT_DIM != 12:
    raise RuntimeError(
        f"Checkpoint expects {INPUT_DIM} input features, "
        "but Phase 21L winning model should use ANOVA12."
    )

if N_QUBITS != 4:
    raise RuntimeError(
        f"Expected 4 qubits, got {N_QUBITS}"
    )

if DEPTH != 2:
    raise RuntimeError(
        f"Expected depth 2, got {DEPTH}"
    )

# =============================================================================
# HYBRID VQC ARCHITECTURE
# =============================================================================
#
# This architecture must match the Phase 21L training architecture.
#
# 12 input features
#       ↓
# classical projection
#       ↓
# 4 quantum features
#       ↓
# RY/RZ encoding
#       ↓
# variational layers
#       ↓
# quantum expectation values
#       ↓
# classical residual branch
#       ↓
# fusion
#       ↓
# binary output
#
# =============================================================================

try:
    import pennylane as qml
except Exception as e:
    raise RuntimeError(
        "PennyLane is required for VQC evaluation."
    ) from e

# -----------------------------------------------------------------------------
# IMPORTANT:
# Use CPU explicitly.
# -----------------------------------------------------------------------------

dev = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

# -----------------------------------------------------------------------------
# Quantum circuit
# -----------------------------------------------------------------------------

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):

    # RY + RZ data encoding
    for q in range(N_QUBITS):

        qml.RY(
            inputs[q],
            wires=q
        )

        qml.RZ(
            inputs[q],
            wires=q
        )

    # Variational layers
    for d in range(DEPTH):

        for q in range(N_QUBITS):

            qml.RY(
                weights[d, q, 0],
                wires=q
            )

            qml.RZ(
                weights[d, q, 1],
                wires=q
            )

        # Ring entanglement
        if ENTANGLE:

            for q in range(N_QUBITS - 1):
                qml.CNOT(
                    wires=[q, q + 1]
                )

            qml.CNOT(
                wires=[N_QUBITS - 1, 0]
            )

    return [
        qml.expval(qml.PauliZ(q))
        for q in range(N_QUBITS)
    ]

# =============================================================================
# MODEL
# =============================================================================

class FrozenHybridVQC(nn.Module):

    def __init__(
        self,
        input_dim,
        n_qubits,
        depth
    ):

        super().__init__()

        self.input_dim = input_dim
        self.n_qubits = n_qubits
        self.depth = depth

        # Classical projection
        self.quantum_projection = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.LayerNorm(16),
            nn.Tanh(),
            nn.Linear(16, n_qubits),
            nn.Tanh()
        )

        # Quantum trainable weights
        self.q_weights = nn.Parameter(
            0.01 * torch.randn(
                depth,
                n_qubits,
                2
            )
        )

        # Classical residual branch
        self.residual_branch = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Linear(16, 8),
            nn.GELU(),
            nn.Dropout(0.0)
        )

        # Fusion
        self.fusion = nn.Sequential(
            nn.Linear(
                n_qubits + 8,
                16
            ),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):

        # -------------------------------------------------------------
        # Quantum projection
        # -------------------------------------------------------------

        q_input = self.quantum_projection(x)

        # Scale angles
        q_input = torch.pi * q_input

        quantum_outputs = []

        for i in range(x.shape[0]):

            q_out = quantum_circuit(
                q_input[i],
                self.q_weights
            )

            q_out = torch.stack(q_out)

            quantum_outputs.append(q_out)

        quantum_outputs = torch.stack(
            quantum_outputs,
            dim=0
        )

        # -------------------------------------------------------------
        # Classical residual branch
        # -------------------------------------------------------------

        residual = self.residual_branch(x)

        # -------------------------------------------------------------
        # Fusion
        # -------------------------------------------------------------

        fused = torch.cat(
            [
                quantum_outputs,
                residual
            ],
            dim=1
        )

        return self.fusion(fused).squeeze(1)


# =============================================================================
# CREATE MODEL
# =============================================================================

print()
print("=" * 100)
print("RECONSTRUCTING FROZEN HYBRID VQC")
print("=" * 100)

model = FrozenHybridVQC(
    input_dim=INPUT_DIM,
    n_qubits=N_QUBITS,
    depth=DEPTH
).to(DEVICE)

# =============================================================================
# LOAD STATE DICT
# =============================================================================

state_dict = checkpoint["model_state_dict"]

try:

    model.load_state_dict(
        state_dict,
        strict=True
    )

except RuntimeError as e:

    raise RuntimeError(
        "\nThe Phase 21L checkpoint does not match the reconstructed "
        "Hybrid VQC architecture.\n\n"
        "This is important: DO NOT evaluate the test with a different "
        "architecture.\n\n"
        f"Checkpoint configuration:\n"
        f"input_dim={INPUT_DIM}\n"
        f"n_qubits={N_QUBITS}\n"
        f"depth={DEPTH}\n"
        f"encoding={ENCODING}\n"
        f"entangle={ENTANGLE}\n\n"
        f"Original loading error:\n{e}"
    )

model.eval()

for parameter in model.parameters():
    parameter.requires_grad_(False)

print("✓ State dictionary loaded")
print("✓ Model set to evaluation mode")
print("✓ All parameters frozen")
print("✓ No optimizer")
print("✓ No training")

# =============================================================================
# CHECKPOINT PARAMETER COUNT
# =============================================================================

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

print("Frozen parameter count:", parameter_count)

# =============================================================================
# LOCKED TEST EVALUATION
# =============================================================================

print()
print("=" * 100)
print("ONE-TIME LOCKED TEST EVALUATION")
print("=" * 100)

X_test_tensor = torch.tensor(
    X_test_12,
    dtype=torch.float32,
    device=DEVICE
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32,
    device=DEVICE
)

# -----------------------------------------------------------------------------
# IMPORTANT:
# no gradients
# -----------------------------------------------------------------------------

with torch.no_grad():

    logits = model(
        X_test_tensor
    )

    probabilities = torch.sigmoid(
        logits
    )

    predictions = (
        probabilities >= 0.5
    ).long()

# -----------------------------------------------------------------------------
# ALWAYS move CPU tensor before numpy conversion.
# This prevents:
#
# TypeError:
# can't convert cuda:0 device type tensor to numpy
# -----------------------------------------------------------------------------

logits_np = (
    logits.detach()
    .cpu()
    .numpy()
)

probabilities_np = (
    probabilities.detach()
    .cpu()
    .numpy()
)

predictions_np = (
    predictions.detach()
    .cpu()
    .numpy()
)

y_test_np = (
    y_test_tensor.detach()
    .cpu()
    .numpy()
    .astype(int)
)

# =============================================================================
# COUNT CHECK
# =============================================================================

print("Prediction count:", len(predictions_np))

assert len(predictions_np) == 1290
assert len(probabilities_np) == 1290
assert len(y_test_np) == 1290

print("✓ Exactly 1290 predictions")
print("✓ Exactly 1290 targets")

# =============================================================================
# METRICS
# =============================================================================

accuracy = accuracy_score(
    y_test_np,
    predictions_np
)

balanced_accuracy = balanced_accuracy_score(
    y_test_np,
    predictions_np
)

f1 = f1_score(
    y_test_np,
    predictions_np,
    zero_division=0
)

mcc = matthews_corrcoef(
    y_test_np,
    predictions_np
)

try:
    roc_auc = roc_auc_score(
        y_test_np,
        probabilities_np
    )
except Exception:
    roc_auc = np.nan

try:
    pr_auc = average_precision_score(
        y_test_np,
        probabilities_np
    )
except Exception:
    pr_auc = np.nan

cm = confusion_matrix(
    y_test_np,
    predictions_np
)

# =============================================================================
# RESULTS
# =============================================================================

print()
print("=" * 100)
print("FINAL PHASE 21L HYBRID VQC — LOCKED TEST RESULT")
print("=" * 100)

print(f"Accuracy           : {accuracy:.4f}")
print(f"Balanced Accuracy  : {balanced_accuracy:.4f}")
print(f"F1                 : {f1:.4f}")
print(f"MCC                : {mcc:.4f}")
print(f"ROC-AUC            : {roc_auc:.4f}")
print(f"PR-AUC             : {pr_auc:.4f}")

print()
print("=" * 100)
print("CONFUSION MATRIX")
print("=" * 100)

print(cm)

print()
print("=" * 100)
print("CLASSIFICATION REPORT")
print("=" * 100)

print(
    classification_report(
        y_test_np,
        predictions_np,
        digits=4,
        zero_division=0
    )
)

# =============================================================================
# INTEGRITY CHECK
# =============================================================================

print()
print("=" * 100)
print("PHASE 21M INTEGRITY CHECK")
print("=" * 100)

print("Original dataset      :", len(df_work))
print("Development rows      :", len(development_df))
print("Validation rows       :", len(validation_df))
print("Locked test rows      :", len(locked_test_df))

print("Development subjects  :", len(dev_subjects))
print("Validation subjects   :", len(val_subjects))
print("Locked test subjects  :", len(test_subjects))

print(
    "Development/validation overlap:",
    dev_subjects & val_subjects
)

print(
    "Development/test overlap:",
    dev_subjects & test_subjects
)

print(
    "Validation/test overlap:",
    val_subjects & test_subjects
)

print("✓ ZERO subject leakage")
print("✓ total_UPDRS not used")
print("✓ subject# not used as feature")
print("✓ severity used only as target")
print("✓ Imputation fitted using development only")
print("✓ Scaling fitted using development only")
print("✓ ANOVA fitted using development only")
print("✓ ANOVA12 applied consistently")
print("✓ Frozen Phase 21L checkpoint loaded")
print("✓ Model frozen")
print("✓ No optimizer")
print("✓ No training")
print("✓ No hyperparameter tuning")
print("✓ Locked test evaluated exactly once")

# =============================================================================
# SAVE METRICS
# =============================================================================

metrics_df = pd.DataFrame([{
    "Model": checkpoint["best_info"].get(
        "Model",
        "B_RYRZ_D2_LR003"
    ),
    "Input": checkpoint["best_info"].get(
        "Input",
        "ANOVA12"
    ),
    "Seed": checkpoint["best_info"].get(
        "Seed",
        2024
    ),
    "Depth": DEPTH,
    "Qubits": N_QUBITS,
    "Encoding": ENCODING,
    "Entangle": ENTANGLE,
    "Accuracy": accuracy,
    "Balanced_Accuracy": balanced_accuracy,
    "F1": f1,
    "MCC": mcc,
    "ROC_AUC": roc_auc,
    "PR_AUC": pr_auc,
    "Test_Rows": len(y_test_np),
    "Test_Subjects": len(test_subjects)
}])

metrics_path = os.path.join(
    OUTPUT_DIR,
    "phase21M_hybrid_vqc_locked_test_metrics.csv"
)

predictions_df = pd.DataFrame({
    "subject#": locked_test_df[SUBJECT_COL].to_numpy(),
    "y_true": y_test_np,
    "probability": probabilities_np,
    "prediction": predictions_np
})

predictions_path = os.path.join(
    OUTPUT_DIR,
    "phase21M_hybrid_vqc_locked_test_predictions.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False
)

predictions_df.to_csv(
    predictions_path,
    index=False
)

print()
print("=" * 100)
print("SAVED")
print("=" * 100)

print("✓", metrics_path)
print("✓", predictions_path)

print()
print("=" * 100)
print("PHASE 21M — COMPLETED")
print("=" * 100)

PHASE 21M — FROZEN PHASE 21L HYBRID VQC LOCKED TEST EVALUATION
Device: cpu
✓ CPU-only
✓ No training
✓ No optimizer
✓ No hyperparameter tuning

LOADING SOURCE DATA


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/parkinsons-updrs/telemonitoring_parkinsons.csv'

In [21]:
# ================================================================================================
# PHASE 21M — ROBUST DATA LOADING + EXACT PHASE 21L SUBJECT SPLIT
# ================================================================================================

import os
import glob
import numpy as np
import pandas as pd

print("=" * 100)
print("PHASE 21M — ROBUST SOURCE DATA LOADING")
print("=" * 100)

# ------------------------------------------------------------------------------------------------
# 1. SEARCH KAGGLE FOR CSV FILES
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SEARCHING FOR DATA FILES")
print("=" * 100)

all_csvs = glob.glob("/kaggle/input/**/*.csv", recursive=True)

print(f"CSV files found: {len(all_csvs)}")

for p in all_csvs:
    print(" ", p)


# ------------------------------------------------------------------------------------------------
# 2. IDENTIFY ORIGINAL PARKINSON DATASET
# ------------------------------------------------------------------------------------------------

original_candidates = []

for p in all_csvs:
    name = os.path.basename(p).lower()

    if (
        "telemonitoring" in name
        or "parkinsons" in name
        or "parkinson" in name
        or "updrs" in name
    ):
        original_candidates.append(p)


# ------------------------------------------------------------------------------------------------
# 3. IDENTIFY SPLIT FILE
# ------------------------------------------------------------------------------------------------

split_candidates = []

for p in all_csvs:
    name = os.path.basename(p).lower()

    # Don't accidentally select the original dataset as split file
    if (
        "split" in name
        or "severity" in name
        or "phase21" in name
    ):
        split_candidates.append(p)


print("\nOriginal candidates:")
for p in original_candidates:
    print(" ", p)

print("\nSplit candidates:")
for p in split_candidates:
    print(" ", p)


# ------------------------------------------------------------------------------------------------
# 4. FIND DATASET BY REQUIRED COLUMNS
# ------------------------------------------------------------------------------------------------

ORIGINAL_DATA_PATH = None
SPLIT_DATA_PATH = None

# Find original dataset by its required columns
for p in all_csvs:

    try:
        temp = pd.read_csv(p, nrows=5)
        cols = set(temp.columns)

        required_original = {
            "subject#",
            "age",
            "sex",
            "test_time",
            "motor_UPDRS",
            "total_UPDRS"
        }

        if required_original.issubset(cols):
            ORIGINAL_DATA_PATH = p
            break

    except Exception:
        pass


# Find split dataset by required columns
for p in all_csvs:

    try:
        temp = pd.read_csv(p, nrows=10)
        cols = set(temp.columns)

        required_split = {
            "subject#",
            "severity",
            "split"
        }

        if required_split.issubset(cols):
            SPLIT_DATA_PATH = p
            break

    except Exception:
        pass


# ------------------------------------------------------------------------------------------------
# 5. FAIL WITH USEFUL INFORMATION IF NOT FOUND
# ------------------------------------------------------------------------------------------------

if ORIGINAL_DATA_PATH is None:

    raise FileNotFoundError(
        "\nCould not automatically locate the original Parkinson dataset.\n"
        "Expected columns include:\n"
        "subject#, age, sex, test_time, motor_UPDRS, total_UPDRS\n\n"
        "CSV files found:\n"
        + "\n".join(all_csvs)
    )


if SPLIT_DATA_PATH is None:

    raise FileNotFoundError(
        "\nCould not automatically locate the split/severity dataset.\n"
        "Expected columns:\n"
        "subject#, severity, split\n\n"
        "CSV files found:\n"
        + "\n".join(all_csvs)
    )


print("\n" + "=" * 100)
print("DATA FILES SELECTED")
print("=" * 100)

print("Original dataset:")
print(ORIGINAL_DATA_PATH)

print("\nSplit dataset:")
print(SPLIT_DATA_PATH)


# ------------------------------------------------------------------------------------------------
# 6. LOAD
# ------------------------------------------------------------------------------------------------

df = pd.read_csv(ORIGINAL_DATA_PATH)
split_df = pd.read_csv(SPLIT_DATA_PATH)

print("\n" + "=" * 100)
print("SOURCE DATA")
print("=" * 100)

print("Original shape:", df.shape)
print("Split shape   :", split_df.shape)

print("\nOriginal columns:")
print(df.columns.tolist())

print("\nSplit columns:")
print(split_df.columns.tolist())


# ------------------------------------------------------------------------------------------------
# 7. STANDARDIZE SUBJECT COLUMN
# ------------------------------------------------------------------------------------------------

SUBJECT_COL = "subject#"
TARGET_COL = "severity"
SPLIT_COL = "split"

if SUBJECT_COL not in df.columns:
    raise RuntimeError("Original dataset does not contain subject#.")

if SUBJECT_COL not in split_df.columns:
    raise RuntimeError("Split dataset does not contain subject#.")

if TARGET_COL not in split_df.columns:
    raise RuntimeError("Split dataset does not contain severity.")

if SPLIT_COL not in split_df.columns:
    raise RuntimeError("Split dataset does not contain split.")


df[SUBJECT_COL] = pd.to_numeric(
    df[SUBJECT_COL],
    errors="raise"
).astype(int)

split_df[SUBJECT_COL] = pd.to_numeric(
    split_df[SUBJECT_COL],
    errors="raise"
).astype(int)


# ------------------------------------------------------------------------------------------------
# 8. IMPORTANT:
# split is SUBJECT-WISE CONSTANT
# severity is ROW-WISE and MUST NOT be merged using subject alone
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("VALIDATING SPLIT INFORMATION")
print("=" * 100)

split_check = (
    split_df
    .groupby(SUBJECT_COL)[SPLIT_COL]
    .nunique()
)

bad_split_subjects = split_check[split_check > 1]

if len(bad_split_subjects) > 0:

    raise RuntimeError(
        "Split is not constant within subject.\n"
        + str(bad_split_subjects)
    )

print("✓ Split is constant within every subject")

print(
    "Unique subjects in split file:",
    split_df[SUBJECT_COL].nunique()
)


# ------------------------------------------------------------------------------------------------
# 9. CHECK WHETHER SPLIT FILE IS ROW-LEVEL
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CHECKING SPLIT FILE GRANULARITY")
print("=" * 100)

print(
    "Original rows:",
    len(df)
)

print(
    "Split rows:",
    len(split_df)
)

print(
    "Original subjects:",
    df[SUBJECT_COL].nunique()
)

print(
    "Split subjects:",
    split_df[SUBJECT_COL].nunique()
)


# ------------------------------------------------------------------------------------------------
# 10. BUILD SUBJECT → SPLIT MAP
# ------------------------------------------------------------------------------------------------

subject_split = (
    split_df[
        [SUBJECT_COL, SPLIT_COL]
    ]
    .drop_duplicates(SUBJECT_COL)
)

if len(subject_split) != split_df[SUBJECT_COL].nunique():

    raise RuntimeError(
        "Could not construct unique subject → split mapping."
    )


# ------------------------------------------------------------------------------------------------
# 11. ATTACH SPLIT TO ORIGINAL DATA
# ------------------------------------------------------------------------------------------------

df_work = df.copy()

df_work = df_work.merge(
    subject_split,
    on=SUBJECT_COL,
    how="left",
    validate="many_to_one"
)


if df_work[SPLIT_COL].isna().any():

    missing_subjects = sorted(
        df_work.loc[
            df_work[SPLIT_COL].isna(),
            SUBJECT_COL
        ].unique()
    )

    raise RuntimeError(
        "Some original-data subjects have no split assignment:\n"
        + str(missing_subjects)
    )


print("\n✓ Split successfully attached to original rows")


# ------------------------------------------------------------------------------------------------
# 12. CREATE LOCKED TEST
# ------------------------------------------------------------------------------------------------

locked_test = df_work[
    df_work[SPLIT_COL].astype(str).str.lower().eq("test")
].copy()

training_pool = df_work[
    df_work[SPLIT_COL].astype(str).str.lower().eq("train")
].copy()


print("\n" + "=" * 100)
print("TRAIN / LOCKED TEST")
print("=" * 100)

print("Training pool rows :", len(training_pool))
print("Locked test rows   :", len(locked_test))

print(
    "Training subjects  :",
    training_pool[SUBJECT_COL].nunique()
)

print(
    "Locked test subjects:",
    locked_test[SUBJECT_COL].nunique()
)


# ------------------------------------------------------------------------------------------------
# 13. VERIFY LOCKED TEST
# ------------------------------------------------------------------------------------------------

if len(locked_test) != 1290:

    raise RuntimeError(
        f"Expected 1290 locked-test rows, "
        f"but found {len(locked_test)}."
    )

if locked_test[SUBJECT_COL].nunique() != 9:

    raise RuntimeError(
        f"Expected 9 locked-test subjects, "
        f"but found {locked_test[SUBJECT_COL].nunique()}."
    )

print("✓ Correct 1290 locked-test rows")
print("✓ Correct 9 locked-test subjects")


# ------------------------------------------------------------------------------------------------
# 14. REUSE THE SUBJECTS FROM YOUR PHASE 21L
#
# If they already exist in memory, reuse them.
# Otherwise use the current saved Phase 21L subject lists if available.
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("RECONSTRUCTING PHASE 21L DEVELOPMENT / VALIDATION")
print("=" * 100)


# Try common variable names from previous notebook cells

dev_subjects_found = None
val_subjects_found = None

for name in [
    "dev_subjects",
    "development_subjects",
    "DEV_SUBJECTS",
    "phase21l_dev_subjects"
]:

    if name in globals():

        candidate = globals()[name]

        if candidate is not None and len(candidate) > 0:

            dev_subjects_found = set(
                int(x) for x in candidate
            )

            print(
                f"✓ Reusing development subjects from: {name}"
            )

            break


for name in [
    "val_subjects",
    "validation_subjects",
    "VAL_SUBJECTS",
    "phase21l_val_subjects"
]:

    if name in globals():

        candidate = globals()[name]

        if candidate is not None and len(candidate) > 0:

            val_subjects_found = set(
                int(x) for x in candidate
            )

            print(
                f"✓ Reusing validation subjects from: {name}"
            )

            break


# ------------------------------------------------------------------------------------------------
# 15. IF NOT IN MEMORY, LOAD FROM SAVED PHASE 21L SPLIT IF AVAILABLE
# ------------------------------------------------------------------------------------------------

possible_saved_splits = glob.glob(
    "/kaggle/working/**/*phase21*l*split*.csv",
    recursive=True
)

possible_saved_splits += glob.glob(
    "/kaggle/working/**/*.csv",
    recursive=True
)


if dev_subjects_found is None or val_subjects_found is None:

    print("\nSearching saved Phase 21L files...")

    found_saved = None

    for p in possible_saved_splits:

        try:

            temp = pd.read_csv(p)

            cols = set(temp.columns)

            if {
                SUBJECT_COL,
                "phase21l_split"
            }.issubset(cols):

                found_saved = temp
                print("✓ Found:", p)
                break

            if {
                SUBJECT_COL,
                "internal_split"
            }.issubset(cols):

                found_saved = temp
                print("✓ Found:", p)
                break

        except Exception:
            pass


    if found_saved is not None:

        split_label_col = (
            "phase21l_split"
            if "phase21l_split" in found_saved.columns
            else "internal_split"
        )

        dev_subjects_found = set(
            found_saved.loc[
                found_saved[split_label_col].eq("development"),
                SUBJECT_COL
            ].astype(int)
        )

        val_subjects_found = set(
            found_saved.loc[
                found_saved[split_label_col].eq("validation"),
                SUBJECT_COL
            ].astype(int)
        )


# ------------------------------------------------------------------------------------------------
# 16. IMPORTANT FALLBACK
# ------------------------------------------------------------------------------------------------

if dev_subjects_found is None or val_subjects_found is None:

    raise RuntimeError(
        "\nPhase 21L development/validation subject lists are not "
        "available in the current notebook.\n\n"
        "DO NOT randomly recreate them if you want exact Phase 21L "
        "reproducibility.\n\n"
        "Run the Phase 21L training cell first, or load the saved "
        "Phase 21L subject split."
    )


dev_subjects = dev_subjects_found
val_subjects = val_subjects_found


test_subjects = set(
    locked_test[SUBJECT_COL].unique()
)


# ------------------------------------------------------------------------------------------------
# 17. SUBJECT LEAKAGE CHECK
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SUBJECT LEAKAGE CHECK")
print("=" * 100)

print(
    "Development subjects:",
    len(dev_subjects)
)

print(
    "Validation subjects :",
    len(val_subjects)
)

print(
    "Locked-test subjects:",
    len(test_subjects)
)


dv_overlap = dev_subjects & val_subjects
dt_overlap = dev_subjects & test_subjects
vt_overlap = val_subjects & test_subjects

print(
    "Development / validation overlap:",
    dv_overlap
)

print(
    "Development / test overlap:",
    dt_overlap
)

print(
    "Validation / test overlap:",
    vt_overlap
)


if dv_overlap or dt_overlap or vt_overlap:

    raise RuntimeError(
        "SUBJECT LEAKAGE DETECTED!"
    )

print("✓ ZERO SUBJECT LEAKAGE")


# ------------------------------------------------------------------------------------------------
# 18. CREATE DEVELOPMENT / VALIDATION
# ------------------------------------------------------------------------------------------------

development_df = training_pool[
    training_pool[SUBJECT_COL].isin(dev_subjects)
].copy()

validation_df = training_pool[
    training_pool[SUBJECT_COL].isin(val_subjects)
].copy()


# ------------------------------------------------------------------------------------------------
# 19. CHECK COMPLETE TRAINING POOL
# ------------------------------------------------------------------------------------------------

if (
    len(development_df)
    + len(validation_df)
    != len(training_pool)
):

    raise RuntimeError(
        "Development + validation does not equal complete training pool."
    )


# ------------------------------------------------------------------------------------------------
# 20. TARGET CONSTRUCTION
#
# IMPORTANT:
# severity is NOT present in the original 22-column dataset.
#
# We therefore recover the ROW-LEVEL severity from the split file.
#
# Because severity varies within subject, subject# alone cannot be
# used to attach severity.
#
# We need row alignment.
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("ATTACHING ROW-LEVEL SEVERITY")
print("=" * 100)


# We need a stable row matching mechanism.
#
# The Parkinson telemonitoring dataset contains:
# subject#, age, sex, test_time, motor_UPDRS, total_UPDRS
#
# We use all available original columns that exist in both files,
# excluding severity/split.

common_cols = [
    c for c in df.columns
    if c in split_df.columns
    and c not in [TARGET_COL, SPLIT_COL]
]


print("Common row-matching columns:")
print(common_cols)


# If the split file contains the same number of rows as the original,
# use direct row alignment after verification.

if len(split_df) == len(df):

    print(
        "\n✓ Split file has exactly the same number of rows "
        "as original dataset."
    )

    # Verify the common columns align.
    mismatch = np.zeros(len(df), dtype=bool)

    for c in common_cols:

        a = df[c].astype(str).fillna("<NA>").to_numpy()
        b = split_df[c].astype(str).fillna("<NA>").to_numpy()

        mismatch |= (a != b)

    mismatch_count = int(mismatch.sum())

    print(
        "Row alignment mismatches:",
        mismatch_count
    )

    if mismatch_count != 0:

        raise RuntimeError(
            "Split file has the same number of rows but does not "
            "align with the original dataset."
        )

    df_work[TARGET_COL] = split_df[TARGET_COL].to_numpy()


else:

    # --------------------------------------------------------------------------------------------
    # Row-level merge using ALL common columns.
    # --------------------------------------------------------------------------------------------

    print(
        "\nSplit file is not the same size as original."
    )

    print(
        "Attempting row-level matching using common columns."
    )

    if len(common_cols) == 0:

        raise RuntimeError(
            "No common columns available for safe row-level severity matching."
        )


    left = df_work.copy()
    right = split_df[
        common_cols + [TARGET_COL]
    ].copy()


    # Check duplicate row keys on split side
    duplicate_key_count = (
        right.groupby(common_cols, dropna=False)
        .size()
    )

    duplicate_key_count = duplicate_key_count[
        duplicate_key_count > 1
    ]


    if len(duplicate_key_count) > 0:

        # This is okay only if duplicated rows have identical severity.

        severity_variation = (
            right.groupby(common_cols, dropna=False)[TARGET_COL]
            .nunique(dropna=False)
        )

        bad_keys = severity_variation[
            severity_variation > 1
        ]

        if len(bad_keys) > 0:

            raise RuntimeError(
                "\nAmbiguous row-level severity matching detected.\n"
                "The same row key maps to multiple severity values.\n"
                "Cannot safely construct the target."
            )


        right = (
            right
            .drop_duplicates(
                subset=common_cols,
                keep="first"
            )
        )


    df_work = left.merge(
        right,
        on=common_cols,
        how="left",
        validate="many_to_one"
    )


    if df_work[TARGET_COL].isna().any():

        missing = int(
            df_work[TARGET_COL].isna().sum()
        )

        raise RuntimeError(
            f"{missing} original rows could not be matched "
            "to a severity target."
        )


# ------------------------------------------------------------------------------------------------
# 21. TARGET VALIDATION
# ------------------------------------------------------------------------------------------------

if TARGET_COL not in df_work.columns:

    raise RuntimeError(
        "Severity target could not be constructed."
    )


df_work[TARGET_COL] = pd.to_numeric(
    df_work[TARGET_COL],
    errors="raise"
)


# If severity is continuous, construct binary target using DEVELOPMENT ONLY.
# If it is already 0/1, keep it.

unique_target = np.sort(
    df_work[TARGET_COL].dropna().unique()
)

print("\nUnique severity values:")
print(unique_target[:20])

print(
    "Number of unique severity values:",
    len(unique_target)
)


# ------------------------------------------------------------------------------------------------
# 22. BUILD FINAL DATAFRAMES WITH TARGET
# ------------------------------------------------------------------------------------------------

development_df = df_work[
    df_work[SUBJECT_COL].isin(dev_subjects)
    &
    df_work[SPLIT_COL].astype(str).str.lower().eq("train")
].copy()


validation_df = df_work[
    df_work[SUBJECT_COL].isin(val_subjects)
    &
    df_work[SPLIT_COL].astype(str).str.lower().eq("train")
].copy()


locked_test = df_work[
    df_work[SPLIT_COL].astype(str).str.lower().eq("test")
].copy()


# ------------------------------------------------------------------------------------------------
# 23. BINARY TARGET
# ------------------------------------------------------------------------------------------------

if set(unique_target).issubset({0, 1}):

    print("\n✓ Severity is already binary.")

    development_df["target"] = (
        development_df[TARGET_COL]
        .astype(int)
    )

    validation_df["target"] = (
        validation_df[TARGET_COL]
        .astype(int)
    )

    locked_test["target"] = (
        locked_test[TARGET_COL]
        .astype(int)
    )

else:

    # DEVELOPMENT-ONLY threshold
    severity_threshold = (
        development_df[TARGET_COL]
        .median()
    )

    print(
        "\nDevelopment-only severity threshold:",
        severity_threshold
    )

    development_df["target"] = (
        development_df[TARGET_COL]
        >= severity_threshold
    ).astype(int)

    validation_df["target"] = (
        validation_df[TARGET_COL]
        >= severity_threshold
    ).astype(int)

    locked_test["target"] = (
        locked_test[TARGET_COL]
        >= severity_threshold
    ).astype(int)


# ------------------------------------------------------------------------------------------------
# 24. FINAL SHAPE CHECK
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("FINAL PHASE 21M SPLIT")
print("=" * 100)

print(
    "Training pool rows :",
    len(training_pool)
)

print(
    "Development rows   :",
    len(development_df)
)

print(
    "Validation rows    :",
    len(validation_df)
)

print(
    "Locked-test rows   :",
    len(locked_test)
)


# Your current Phase 21L subject allocation was:
# Development = 3602
# Validation  = 983
# Test        = 1290

if len(development_df) != 3602:

    print(
        "\nWARNING: Development rows differ from current "
        "Phase 21L allocation."
    )

if len(validation_df) != 983:

    print(
        "\nWARNING: Validation rows differ from current "
        "Phase 21L allocation."
    )

if len(locked_test) != 1290:

    raise RuntimeError(
        "LOCKED TEST MUST CONTAIN EXACTLY 1290 ROWS."
    )


# ------------------------------------------------------------------------------------------------
# 25. CLASS DISTRIBUTION
# ------------------------------------------------------------------------------------------------

print("\nClass counts:")

print(
    "Development:",
    np.bincount(
        development_df["target"].to_numpy(),
        minlength=2
    )
)

print(
    "Validation :",
    np.bincount(
        validation_df["target"].to_numpy(),
        minlength=2
    )
)

print(
    "Locked test:",
    np.bincount(
        locked_test["target"].to_numpy(),
        minlength=2
    )
)


# ------------------------------------------------------------------------------------------------
# 26. CREATE NUMPY TARGETS
# ------------------------------------------------------------------------------------------------

y_dev = (
    development_df["target"]
    .astype(int)
    .to_numpy()
)

y_val = (
    validation_df["target"]
    .astype(int)
    .to_numpy()
)

y_test = (
    locked_test["target"]
    .astype(int)
    .to_numpy()
)


# ------------------------------------------------------------------------------------------------
# 27. SAVE DATA
# ------------------------------------------------------------------------------------------------

OUTPUT_DIR = "/kaggle/working/phase21M_outputs"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


development_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "phase21M_development.csv"
    ),
    index=False
)

validation_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "phase21M_validation.csv"
    ),
    index=False
)

locked_test.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "phase21M_locked_test.csv"
    ),
    index=False
)


# Save subject allocation
pd.DataFrame({
    SUBJECT_COL: list(dev_subjects),
    "internal_split": "development"
}).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "phase21M_development_subjects.csv"
    ),
    index=False
)

pd.DataFrame({
    SUBJECT_COL: list(val_subjects),
    "internal_split": "validation"
}).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "phase21M_validation_subjects.csv"
    ),
    index=False
)


# ------------------------------------------------------------------------------------------------
# 28. FINAL INTEGRITY
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("✓ PHASE 21M DATA PREPARATION COMPLETED")
print("=" * 100)

print(
    "\nDevelopment:",
    development_df.shape
)

print(
    "Validation :",
    validation_df.shape
)

print(
    "Locked test:",
    locked_test.shape
)

print("\nIntegrity:")

print("✓ Severity attached row-by-row")
print("✓ Split attached subject-wise")
print("✓ Severity variation within subject allowed")
print("✓ Exact Phase 21L subjects reused")
print("✓ ZERO subject leakage")
print("✓ Locked test contains exactly 1290 rows")
print("✓ Locked test has 9 subjects")
print("✓ Locked test remains untouched for tuning")
print("✓ No test hyperparameter tuning")

print("\nSaved to:")
print(OUTPUT_DIR)

print("\nFiles:")
print("✓ phase21M_development.csv")
print("✓ phase21M_validation.csv")
print("✓ phase21M_locked_test.csv")
print("✓ phase21M_development_subjects.csv")
print("✓ phase21M_validation_subjects.csv")

print("\n" + "=" * 100)

PHASE 21M — ROBUST SOURCE DATA LOADING

SEARCHING FOR DATA FILES
CSV files found: 2
  /kaggle/input/datasets/saiharish0088/subject-wise/parkinsons_subject_wise_split.csv
  /kaggle/input/datasets/saiharish0088/pppppppppp/parkinsons_updrs.csv

Original candidates:
  /kaggle/input/datasets/saiharish0088/subject-wise/parkinsons_subject_wise_split.csv
  /kaggle/input/datasets/saiharish0088/pppppppppp/parkinsons_updrs.csv

Split candidates:
  /kaggle/input/datasets/saiharish0088/subject-wise/parkinsons_subject_wise_split.csv

DATA FILES SELECTED
Original dataset:
/kaggle/input/datasets/saiharish0088/pppppppppp/parkinsons_updrs.csv

Split dataset:
/kaggle/input/datasets/saiharish0088/subject-wise/parkinsons_subject_wise_split.csv

SOURCE DATA
Original shape: (5875, 22)
Split shape   : (5875, 3)

Original columns:
['subject#', 'age', 'sex', 'test_time', 'motor_UPDRS', 'total_UPDRS', 'Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP', 'Jitter:PPQ5', 'Jitter:DDP', 'Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3',

In [22]:
# ================================================================================================
# PHASE 21M — FROZEN HYBRID VQC ONE-TIME LOCKED TEST EVALUATION
# ================================================================================================
# IMPORTANT:
#   - Loads the frozen Phase 21L checkpoint
#   - Uses Phase 21M development data ONLY to fit preprocessing
#   - Applies identical preprocessing to locked test
#   - Does NOT train
#   - Does NOT tune
#   - Does NOT use locked test for model selection
#   - CPU ONLY
# ================================================================================================

import os
import sys
import copy
import random
import warnings
import numpy as np
import pandas as pd
import torch

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------------
# CPU ONLY
# --------------------------------------------------------------------------------

os.environ["CUDA_VISIBLE_DEVICES"] = ""

DEVICE = torch.device("cpu")

torch.set_default_device("cpu")

print("=" * 100)
print("PHASE 21M — FROZEN HYBRID VQC ONE-TIME LOCKED TEST EVALUATION")
print("=" * 100)
print("Device:", DEVICE)
print("✓ CPU-only evaluation")
print("✓ CUDA will NOT be used")
print("✓ No training")
print("✓ No optimizer")
print("✓ No hyperparameter tuning")


# ================================================================================================
# PATHS
# ================================================================================================

BASE_DIR = "/kaggle/working/phase21M_outputs"
MODEL_PATH = "/kaggle/working/phase21L_outputs/phase21L_best_hybrid_vqc.pt"

DEV_PATH = os.path.join(
    BASE_DIR,
    "phase21M_development.csv"
)

VAL_PATH = os.path.join(
    BASE_DIR,
    "phase21M_validation.csv"
)

TEST_PATH = os.path.join(
    BASE_DIR,
    "phase21M_locked_test.csv"
)

OUTPUT_DIR = "/kaggle/working/phase21M_final_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ================================================================================================
# CHECK FILES
# ================================================================================================

print("\n" + "=" * 100)
print("CHECKING PHASE 21L / 21M ARTIFACTS")
print("=" * 100)

for path in [MODEL_PATH, DEV_PATH, VAL_PATH, TEST_PATH]:
    print(("✓ " if os.path.exists(path) else "✗ "), path)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        "Frozen Phase 21L model not found:\n" + MODEL_PATH
    )

if not os.path.exists(DEV_PATH):
    raise FileNotFoundError(DEV_PATH)

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(TEST_PATH)


# ================================================================================================
# LOAD PHASE 21M DATA
# ================================================================================================

print("\n" + "=" * 100)
print("LOADING PHASE 21M DATA")
print("=" * 100)

dev_df = pd.read_csv(DEV_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Development:", dev_df.shape)
print("Validation :", val_df.shape)
print("Locked test:", test_df.shape)


# ================================================================================================
# TARGET
# ================================================================================================

TARGET_COL = "severity"
SUBJECT_COL = "subject#"
SPLIT_COL = "split"

for name, frame in [
    ("development", dev_df),
    ("validation", val_df),
    ("locked test", test_df)
]:
    if TARGET_COL not in frame.columns:
        raise RuntimeError(
            f"{TARGET_COL} missing from {name} dataset."
        )

    if SUBJECT_COL not in frame.columns:
        raise RuntimeError(
            f"{SUBJECT_COL} missing from {name} dataset."
        )


y_dev = dev_df[TARGET_COL].astype(int).to_numpy()
y_val = val_df[TARGET_COL].astype(int).to_numpy()
y_test = test_df[TARGET_COL].astype(int).to_numpy()

print("\nDevelopment target:", y_dev.shape)
print("Validation target :", y_val.shape)
print("Locked-test target:", y_test.shape)

print("\nClass counts:")
print("Development:", np.bincount(y_dev))
print("Validation :", np.bincount(y_val))
print("Locked test:", np.bincount(y_test))


# ================================================================================================
# LOCKED TEST INTEGRITY
# ================================================================================================

print("\n" + "=" * 100)
print("LOCKED TEST INTEGRITY")
print("=" * 100)

if len(test_df) != 1290:
    raise RuntimeError(
        f"Expected exactly 1290 locked-test rows, got {len(test_df)}"
    )

test_subjects = set(test_df[SUBJECT_COL].unique())
dev_subjects = set(dev_df[SUBJECT_COL].unique())
val_subjects = set(val_df[SUBJECT_COL].unique())

print("Locked-test rows:", len(test_df))
print("Locked-test subjects:", len(test_subjects))

print("Development/Test overlap:",
      dev_subjects.intersection(test_subjects))

print("Validation/Test overlap:",
      val_subjects.intersection(test_subjects))

if dev_subjects.intersection(test_subjects):
    raise RuntimeError("SUBJECT LEAKAGE: development/test overlap!")

if val_subjects.intersection(test_subjects):
    raise RuntimeError("SUBJECT LEAKAGE: validation/test overlap!")

print("✓ ZERO SUBJECT LEAKAGE")


# ================================================================================================
# FEATURES
# ================================================================================================
# Phase 21L selected ANOVA12.
#
# Exclude:
#   subject#
#   total_UPDRS
#   severity
#   split
#
# Also exclude any non-feature columns.
# ================================================================================================

EXCLUDE_COLS = {
    SUBJECT_COL,
    "total_UPDRS",
    TARGET_COL,
    SPLIT_COL
}

feature_cols = [
    c for c in dev_df.columns
    if c not in EXCLUDE_COLS
]

print("\n" + "=" * 100)
print("FEATURE CONSTRUCTION")
print("=" * 100)

print("Candidate features:", len(feature_cols))
print(feature_cols)

# Ensure same columns in all datasets
missing_val = [c for c in feature_cols if c not in val_df.columns]
missing_test = [c for c in feature_cols if c not in test_df.columns]

if missing_val:
    raise RuntimeError(
        "Validation missing features: " + str(missing_val)
    )

if missing_test:
    raise RuntimeError(
        "Locked test missing features: " + str(missing_test)
    )


X_dev_raw = dev_df[feature_cols].apply(
    pd.to_numeric,
    errors="coerce"
).to_numpy(dtype=np.float32)

X_val_raw = val_df[feature_cols].apply(
    pd.to_numeric,
    errors="coerce"
).to_numpy(dtype=np.float32)

X_test_raw = test_df[feature_cols].apply(
    pd.to_numeric,
    errors="coerce"
).to_numpy(dtype=np.float32)

print("\nRaw matrices:")
print("Development:", X_dev_raw.shape)
print("Validation :", X_val_raw.shape)
print("Locked test:", X_test_raw.shape)


# ================================================================================================
# DEVELOPMENT-FITTED IMPUTATION
# ================================================================================================

print("\n" + "=" * 100)
print("DEVELOPMENT-FITTED IMPUTATION")
print("=" * 100)

imputer = SimpleImputer(
    strategy="median"
)

X_dev_imp = imputer.fit_transform(X_dev_raw)
X_val_imp = imputer.transform(X_val_raw)
X_test_imp = imputer.transform(X_test_raw)

print("✓ Imputer fitted using DEVELOPMENT only")


# ================================================================================================
# DEVELOPMENT-FITTED SCALING
# ================================================================================================

print("\n" + "=" * 100)
print("DEVELOPMENT-FITTED SCALING")
print("=" * 100)

scaler = StandardScaler()

X_dev_scaled = scaler.fit_transform(X_dev_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

print("✓ Scaler fitted using DEVELOPMENT only")


# ================================================================================================
# ANOVA12
# ================================================================================================

print("\n" + "=" * 100)
print("ANOVA12 FEATURE SELECTION")
print("=" * 100)

anova = SelectKBest(
    score_func=f_classif,
    k=min(12, X_dev_scaled.shape[1])
)

X_dev_12 = anova.fit_transform(
    X_dev_scaled,
    y_dev
)

X_val_12 = anova.transform(
    X_val_scaled
)

X_test_12 = anova.transform(
    X_test_scaled
)

selected_features = [
    feature_cols[i]
    for i in anova.get_support(indices=True)
]

print("Selected features:")
for i, feature in enumerate(selected_features, 1):
    print(f"{i:2d}. {feature}")

print("\nFinal matrices:")
print("Development:", X_dev_12.shape)
print("Validation :", X_val_12.shape)
print("Locked test:", X_test_12.shape)

if X_test_12.shape != (1290, 12):
    raise RuntimeError(
        f"Expected locked-test ANOVA12 shape (1290, 12), "
        f"got {X_test_12.shape}"
    )

print("✓ Locked test correctly transformed to 12 features")


# ================================================================================================
# LOAD FROZEN CHECKPOINT
# ================================================================================================

print("\n" + "=" * 100)
print("LOADING FROZEN PHASE 21L CHECKPOINT")
print("=" * 100)

# PyTorch >= 2.6
# The checkpoint is trusted because it was generated by our own Phase 21L run.

try:
    checkpoint = torch.load(
        MODEL_PATH,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    checkpoint = torch.load(
        MODEL_PATH,
        map_location="cpu"
    )

print("Checkpoint type:", type(checkpoint))

if not isinstance(checkpoint, dict):
    raise RuntimeError(
        "Expected Phase 21L checkpoint to be a dictionary."
    )

print("\nCheckpoint keys:")
for key in checkpoint.keys():
    print(" ✓", key)


# ================================================================================================
# CHECK CHECKPOINT METADATA
# ================================================================================================

required_keys = [
    "model_state_dict",
    "best_info",
    "input_dim",
    "n_qubits",
    "depth",
    "encoding",
    "entangle"
]

missing = [
    k for k in required_keys
    if k not in checkpoint
]

if missing:
    raise RuntimeError(
        "Phase 21L checkpoint missing keys: " + str(missing)
    )

INPUT_DIM = int(checkpoint["input_dim"])
N_QUBITS = int(checkpoint["n_qubits"])
DEPTH = int(checkpoint["depth"])
ENCODING = checkpoint["encoding"]
ENTANGLE = bool(checkpoint["entangle"])

print("\nFrozen configuration:")
print("Input dimension:", INPUT_DIM)
print("Qubits         :", N_QUBITS)
print("Depth          :", DEPTH)
print("Encoding       :", ENCODING)
print("Entanglement   :", ENTANGLE)

print("\nBest information:")
print(checkpoint["best_info"])

if INPUT_DIM != 12:
    raise RuntimeError(
        f"Frozen Phase 21L model expects {INPUT_DIM} inputs, "
        f"but Phase 21L best model should use 12 ANOVA features."
    )


# ================================================================================================
# IMPORTANT MODEL RECONSTRUCTION
# ================================================================================================
#
# Phase 21L created the model class during training.
# If the class is still available in the notebook, use it.
#
# We search common names rather than assuming one exact class name.
# ================================================================================================

print("\n" + "=" * 100)
print("RECONSTRUCTING FROZEN HYBRID VQC")
print("=" * 100)

candidate_class_names = [
    "HybridVQC",
    "HybridVQCModel",
    "HybridQuantumClassicalModel",
    "HybridVQCClassifier",
    "VQCHybrid",
    "HybridModel",
    "VQC"
]

ModelClass = None
ModelClassName = None

for name in candidate_class_names:
    obj = globals().get(name, None)

    if isinstance(obj, type):
        ModelClass = obj
        ModelClassName = name
        break

if ModelClass is None:
    raise RuntimeError(
        "\nCould not find the Phase 21L model class in the notebook.\n"
        "\nThe checkpoint contains only model_state_dict, not the Python "
        "model object.\n"
        "\nTherefore the EXACT Phase 21L model class must be defined "
        "before this evaluation cell.\n"
        "\nRun the Phase 21L model-definition/training cell first, "
        "without retraining, so that its class definition exists in memory."
    )

print("✓ Model class found:", ModelClassName)


# ================================================================================================
# RECREATE MODEL
# ================================================================================================

# Try the most common constructor forms used by Phase 21L.
# The first successful compatible constructor is used.

constructor_attempts = [
    lambda: ModelClass(
        input_dim=INPUT_DIM,
        n_qubits=N_QUBITS,
        depth=DEPTH,
        encoding=ENCODING,
        entangle=ENTANGLE
    ),

    lambda: ModelClass(
        INPUT_DIM,
        N_QUBITS,
        DEPTH,
        ENCODING,
        ENTANGLE
    ),

    lambda: ModelClass(
        input_dim=INPUT_DIM,
        n_qubits=N_QUBITS,
        depth=DEPTH,
        encoding=ENCODING,
        entanglement=ENTANGLE
    )
]

model = None
last_error = None

for constructor in constructor_attempts:
    try:
        model = constructor()
        break
    except Exception as e:
        last_error = e

if model is None:
    raise RuntimeError(
        "\nCould not reconstruct Phase 21L model.\n"
        f"\nLast constructor error:\n{repr(last_error)}"
    )

model = model.to("cpu")


# ================================================================================================
# LOAD STATE DICT
# ================================================================================================

print("\nLoading frozen weights...")

state_dict = checkpoint["model_state_dict"]

try:
    model.load_state_dict(
        state_dict,
        strict=True
    )
except Exception as e:
    raise RuntimeError(
        "\nThe model class was found, but its architecture does not "
        "exactly match the Phase 21L checkpoint.\n\n"
        "Do NOT modify the architecture for the locked-test evaluation.\n"
        "Use the exact Phase 21L model-definition class.\n\n"
        f"Error:\n{repr(e)}"
    )

print("✓ Frozen Phase 21L weights loaded")


# ================================================================================================
# FREEZE MODEL
# ================================================================================================

model.eval()

for parameter in model.parameters():
    parameter.requires_grad = False

print("✓ Model set to evaluation mode")
print("✓ All parameters frozen")
print("✓ No optimizer")
print("✓ No training")


# ================================================================================================
# CONVERT TEST DATA
# ================================================================================================

X_test_tensor = torch.tensor(
    X_test_12,
    dtype=torch.float32,
    device="cpu"
)

print("\nTest tensor:", X_test_tensor.shape)


# ================================================================================================
# SAFE FORWARD
# ================================================================================================

print("\n" + "=" * 100)
print("RUNNING ONE-TIME LOCKED TEST INFERENCE")
print("=" * 100)

with torch.no_grad():

    output = model(X_test_tensor)

# Convert safely to CPU NumPy
if isinstance(output, (tuple, list)):
    output = output[0]

if torch.is_tensor(output):
    output_cpu = output.detach().cpu()
else:
    output_cpu = torch.as_tensor(
        output,
        dtype=torch.float32
    ).cpu()

print("Raw output shape:", tuple(output_cpu.shape))


# ================================================================================================
# NORMALIZE OUTPUT
# ================================================================================================

if output_cpu.ndim == 2 and output_cpu.shape[1] == 2:

    # Two-class logits
    probabilities = torch.softmax(
        output_cpu,
        dim=1
    )[:, 1].numpy()

    predictions = (
        probabilities >= 0.5
    ).astype(int)

elif output_cpu.ndim == 2 and output_cpu.shape[1] == 1:

    scores = output_cpu[:, 0].numpy()

    # Detect logits vs probabilities
    if np.min(scores) < 0 or np.max(scores) > 1:
        probabilities = 1.0 / (
            1.0 + np.exp(-scores)
        )
    else:
        probabilities = scores

    predictions = (
        probabilities >= 0.5
    ).astype(int)

elif output_cpu.ndim == 1:

    scores = output_cpu.numpy()

    if np.min(scores) < 0 or np.max(scores) > 1:
        probabilities = 1.0 / (
            1.0 + np.exp(-scores)
        )
    else:
        probabilities = scores

    predictions = (
        probabilities >= 0.5
    ).astype(int)

else:
    raise RuntimeError(
        f"Unexpected model output shape: {tuple(output_cpu.shape)}"
    )


# ================================================================================================
# PREDICTION COUNT
# ================================================================================================

if len(predictions) != len(y_test):
    raise RuntimeError(
        f"Prediction count {len(predictions)} does not match "
        f"test rows {len(y_test)}"
    )

print("✓ Prediction count:", len(predictions))
print("✓ Expected:", len(y_test))


# ================================================================================================
# FINAL METRICS
# ================================================================================================

accuracy = accuracy_score(
    y_test,
    predictions
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    predictions
)

f1 = f1_score(
    y_test,
    predictions,
    zero_division=0
)

mcc = matthews_corrcoef(
    y_test,
    predictions
)

try:
    roc_auc = roc_auc_score(
        y_test,
        probabilities
    )
except Exception:
    roc_auc = np.nan

try:
    pr_auc = average_precision_score(
        y_test,
        probabilities
    )
except Exception:
    pr_auc = np.nan


# ================================================================================================
# CONFUSION MATRIX
# ================================================================================================

cm = confusion_matrix(
    y_test,
    predictions
)

report = classification_report(
    y_test,
    predictions,
    digits=4,
    zero_division=0
)


# ================================================================================================
# PRINT FINAL RESULT
# ================================================================================================

print("\n" + "=" * 100)
print("FINAL FROZEN HYBRID VQC LOCKED TEST RESULT")
print("=" * 100)

print(f"Accuracy           : {accuracy:.4f}")
print(f"Balanced Accuracy  : {balanced_accuracy:.4f}")
print(f"F1                 : {f1:.4f}")
print(f"MCC                : {mcc:.4f}")
print(f"ROC-AUC            : {roc_auc:.4f}")
print(f"PR-AUC             : {pr_auc:.4f}")


print("\n" + "=" * 100)
print("CONFUSION MATRIX")
print("=" * 100)

print(cm)


print("\n" + "=" * 100)
print("CLASSIFICATION REPORT")
print("=" * 100)

print(report)


# ================================================================================================
# SAVE PREDICTIONS
# ================================================================================================

prediction_df = pd.DataFrame({
    "subject#": test_df[SUBJECT_COL].to_numpy(),
    "y_true": y_test,
    "y_pred": predictions,
    "probability_class_1": probabilities
})

prediction_path = os.path.join(
    OUTPUT_DIR,
    "phase21M_hybrid_vqc_locked_test_predictions.csv"
)

prediction_df.to_csv(
    prediction_path,
    index=False
)


# ================================================================================================
# SAVE METRICS
# ================================================================================================

metrics_df = pd.DataFrame([{
    "Model": checkpoint["best_info"].get(
        "Model",
        "B_RYRZ_D2_LR003"
    ),
    "Input": checkpoint["best_info"].get(
        "Input",
        "ANOVA12"
    ),
    "Accuracy": accuracy,
    "Balanced_Accuracy": balanced_accuracy,
    "F1": f1,
    "MCC": mcc,
    "ROC_AUC": roc_auc,
    "PR_AUC": pr_auc,
    "Test_Rows": len(y_test),
    "Test_Subjects": len(test_subjects),
    "Selection": "Development + Validation only",
    "Test_Tuning": False
}])

metrics_path = os.path.join(
    OUTPUT_DIR,
    "phase21M_hybrid_vqc_locked_test_metrics.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False
)


# ================================================================================================
# INTEGRITY SUMMARY
# ================================================================================================

print("\n" + "=" * 100)
print("PHASE 21M INTEGRITY CHECK")
print("=" * 100)

print("✓ Development preprocessing fitted only on development")
print("✓ ANOVA12 fitted only on development")
print("✓ Validation NOT used for preprocessing fitting")
print("✓ Locked test NOT used for preprocessing fitting")
print("✓ Frozen Phase 21L weights loaded")
print("✓ No optimizer")
print("✓ No training")
print("✓ No hyperparameter tuning")
print("✓ ZERO subject leakage")
print("✓ Locked test evaluated once")
print("✓ 1290 locked-test predictions generated")

print("\nSaved:")
print("✓", prediction_path)
print("✓", metrics_path)

print("\n" + "=" * 100)
print("PHASE 21M — COMPLETED")
print("=" * 100)

PHASE 21M — FROZEN HYBRID VQC ONE-TIME LOCKED TEST EVALUATION
Device: cpu
✓ CPU-only evaluation
✓ CUDA will NOT be used
✓ No training
✓ No optimizer
✓ No hyperparameter tuning

CHECKING PHASE 21L / 21M ARTIFACTS
✓  /kaggle/working/phase21L_outputs/phase21L_best_hybrid_vqc.pt
✓  /kaggle/working/phase21M_outputs/phase21M_development.csv
✓  /kaggle/working/phase21M_outputs/phase21M_validation.csv
✓  /kaggle/working/phase21M_outputs/phase21M_locked_test.csv

LOADING PHASE 21M DATA
Development: (3602, 25)
Validation : (983, 25)
Locked test: (1290, 25)

Development target: (3602,)
Validation target : (983,)
Locked-test target: (1290,)

Class counts:
Development: [2348 1254]
Validation : [589 394]
Locked test: [797 493]

LOCKED TEST INTEGRITY
Locked-test rows: 1290
Locked-test subjects: 9
Development/Test overlap: set()
Validation/Test overlap: set()
✓ ZERO SUBJECT LEAKAGE

FEATURE CONSTRUCTION
Candidate features: 21
['age', 'sex', 'test_time', 'motor_UPDRS', 'Jitter(%)', 'Jitter(Abs)', 'Jitt

In [25]:
# =============================================================================
# PHASE 21N — COMPLETE SUBJECT-WISE HYBRID VQC PIPELINE
# =============================================================================
# COMPLETE PIPELINE:
#
#   Original data
#        ↓
#   Exact subject-wise split
#        ↓
#   Development / Validation / Locked Test
#        ↓
#   Development-only imputation + scaling
#        ↓
#   Development-only ANOVA feature selection
#        ↓
#   ANOVA12
#        ↓
#   Classical projection
#        ↓
#   4-qubit VQC
#        ↓
#   Classical residual branch
#        ↓
#   VQC + residual fusion
#        ↓
#   Validation-based model selection
#        ↓
#   Validation threshold selection
#        ↓
#   FREEZE
#        ↓
#   ONE-TIME LOCKED TEST
#
# IMPORTANT:
#   Locked test is NOT used for:
#       - preprocessing
#       - feature selection
#       - hyperparameter selection
#       - threshold selection
#       - early stopping
#
# CPU ONLY — CUDA is deliberately disabled.
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

import os
import gc
import copy
import random
import warnings

# Disable CUDA BEFORE importing torch operations
os.environ["CUDA_VISIBLE_DEVICES"] = ""

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from sklearn.utils.class_weight import compute_class_weight

# PennyLane
import pennylane as qml


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

SEED = 2024

DATA_PATH = (
    "/kaggle/input/datasets/"
    "saiharish0088/pppppppppp/"
    "parkinsons_updrs.csv"
)

SPLIT_PATH = (
    "/kaggle/input/datasets/"
    "saiharish0088/subject-wise/"
    "parkinsons_subject_wise_split.csv"
)

OUTPUT_DIR = "/kaggle/working/phase21N_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COL = "severity"
SUBJECT_COL = "subject#"
SPLIT_COL = "split"

N_SELECTED_FEATURES = 12
N_QUBITS = 4

MAX_EPOCHS = 60
PATIENCE = 10

BATCH_SIZE = 32

LEARNING_RATES = [
    0.001,
    0.003,
    0.005
]

WEIGHT_DECAYS = [
    0.0,
    1e-4
]

DEPTHS = [
    2,
    3
]

ENCODINGS = [
    "RYRZ"
]

SEEDS = [
    123,
    2024
]

# Small projection dimension before VQC
PROJECTION_HIDDEN = 16

# Dropout
DROPOUT = 0.20


# =============================================================================
# 2. CPU ONLY
# =============================================================================

DEVICE = torch.device("cpu")

torch.set_num_threads(max(1, os.cpu_count() // 2))

print("=" * 100)
print("PHASE 21N — COMPLETE HYBRID VQC PIPELINE")
print("=" * 100)

print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())
print("✓ CUDA deliberately disabled")
print("✓ VQC will run entirely on CPU")
print("=" * 100)


# =============================================================================
# 3. REPRODUCIBILITY
# =============================================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    # CPU only
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# =============================================================================
# 4. LOAD DATA
# =============================================================================

print("\n" + "=" * 100)
print("LOADING SOURCE DATA")
print("=" * 100)

df = pd.read_csv(DATA_PATH)
split_df = pd.read_csv(SPLIT_PATH)

print("Original dataset:", df.shape)
print("Split dataset   :", split_df.shape)

print("\nOriginal columns:")
print(df.columns.tolist())

print("\nSplit columns:")
print(split_df.columns.tolist())


# =============================================================================
# 5. VALIDATE SOURCE DATA
# =============================================================================

required_original = {
    SUBJECT_COL,
    "age",
    "sex",
    "test_time",
    "motor_UPDRS",
    "total_UPDRS",
    "Jitter(%)",
    "Jitter(Abs)",
    "Jitter:RAP",
    "Jitter:PPQ5",
    "Jitter:DDP",
    "Shimmer",
    "Shimmer(dB)",
    "Shimmer:APQ3",
    "Shimmer:APQ5",
    "Shimmer:APQ11",
    "Shimmer:DDA",
    "NHR",
    "HNR",
    "RPDE",
    "DFA",
    "PPE"
}

required_split = {
    SUBJECT_COL,
    TARGET_COL,
    SPLIT_COL
}

missing_original = required_original - set(df.columns)
missing_split = required_split - set(split_df.columns)

if missing_original:
    raise RuntimeError(
        f"Missing original columns: {sorted(missing_original)}"
    )

if missing_split:
    raise RuntimeError(
        f"Missing split columns: {sorted(missing_split)}"
    )


# =============================================================================
# 6. VALIDATE SPLIT GRANULARITY
# =============================================================================

print("\n" + "=" * 100)
print("VALIDATING SUBJECT-WISE SPLIT")
print("=" * 100)

split_subject_counts = split_df.groupby(
    SUBJECT_COL
)[SPLIT_COL].nunique()

bad_split_subjects = split_subject_counts[
    split_subject_counts > 1
]

if len(bad_split_subjects) > 0:
    raise RuntimeError(
        "Split is not constant within subjects:\n"
        + str(bad_split_subjects)
    )

print("✓ Split is constant within every subject")

print(
    "Unique subjects in original:",
    df[SUBJECT_COL].nunique()
)

print(
    "Unique subjects in split:",
    split_df[SUBJECT_COL].nunique()
)


# =============================================================================
# 7. ALIGN SPLIT INFORMATION ROW-BY-ROW
# =============================================================================
# The split file has exactly 5875 rows.
# We use row position because previous checks established zero row mismatch.
#
# Severity varies row-by-row, therefore DO NOT merge severity using subject#.
# =============================================================================

if len(df) != len(split_df):
    raise RuntimeError(
        f"Original and split files have different row counts: "
        f"{len(df)} vs {len(split_df)}"
    )

print("\n✓ Same number of rows")

# Verify subject sequence alignment
subject_mismatch = (
    df[SUBJECT_COL].to_numpy()
    != split_df[SUBJECT_COL].to_numpy()
)

if subject_mismatch.any():

    mismatch_count = int(subject_mismatch.sum())

    raise RuntimeError(
        f"Subject row alignment mismatch: {mismatch_count} rows"
    )

print("✓ Subject row alignment verified")

# Copy source
data = df.copy()

# Attach row-level target and split
data[TARGET_COL] = (
    split_df[TARGET_COL]
    .astype(int)
    .to_numpy()
)

data[SPLIT_COL] = (
    split_df[SPLIT_COL]
    .astype(str)
    .to_numpy()
)

print("✓ Severity attached row-by-row")
print("✓ Split attached row-by-row")


# =============================================================================
# 8. IDENTIFY SPLIT LABELS
# =============================================================================

print("\nSplit labels:")
print(data[SPLIT_COL].value_counts())

print("\nUnique severity:")
print(sorted(data[TARGET_COL].unique()))

if not set(data[TARGET_COL].unique()).issubset({0, 1}):
    raise RuntimeError(
        "Severity is not binary."
    )


# =============================================================================
# 9. REUSE SUBJECT-WISE SPLIT
# =============================================================================
# We use the split labels already present in the subject-wise split file.
#
# This guarantees:
#   development subjects ∩ validation subjects = empty
#   development subjects ∩ test subjects = empty
#   validation subjects ∩ test subjects = empty
# =============================================================================

split_values = data[SPLIT_COL].str.lower().str.strip()

print("\nNormalized split labels:")
print(split_values.value_counts())


def find_split_name(values, candidates):

    for candidate in candidates:

        mask = values == candidate

        if mask.any():
            return candidate

    return None


dev_label = find_split_name(
    split_values,
    ["development", "dev", "train"]
)

val_label = find_split_name(
    split_values,
    ["validation", "val"]
)

test_label = find_split_name(
    split_values,
    ["test", "locked_test", "locked-test"]
)


if dev_label is None:
    raise RuntimeError(
        "Could not identify development/train split."
    )

if val_label is None:
    raise RuntimeError(
        "Could not identify validation split."
    )

if test_label is None:
    raise RuntimeError(
        "Could not identify locked test split."
    )


print("\nSelected labels:")
print("Development:", dev_label)
print("Validation :", val_label)
print("Locked test:", test_label)


# =============================================================================
# 10. CREATE SPLITS
# =============================================================================

development_df = data[
    split_values == dev_label
].copy()

validation_df = data[
    split_values == val_label
].copy()

locked_test_df = data[
    split_values == test_label
].copy()


# =============================================================================
# 11. SUBJECT LEAKAGE CHECK
# =============================================================================

dev_subjects = set(
    development_df[SUBJECT_COL].unique()
)

val_subjects = set(
    validation_df[SUBJECT_COL].unique()
)

test_subjects = set(
    locked_test_df[SUBJECT_COL].unique()
)

print("\n" + "=" * 100)
print("SUBJECT LEAKAGE CHECK")
print("=" * 100)

print("Development subjects:", len(dev_subjects))
print("Validation subjects :", len(val_subjects))
print("Locked-test subjects:", len(test_subjects))

print(
    "Development / validation overlap:",
    dev_subjects & val_subjects
)

print(
    "Development / test overlap:",
    dev_subjects & test_subjects
)

print(
    "Validation / test overlap:",
    val_subjects & test_subjects
)

if (
    dev_subjects & val_subjects
    or
    dev_subjects & test_subjects
    or
    val_subjects & test_subjects
):

    raise RuntimeError(
        "SUBJECT LEAKAGE DETECTED"
    )

print("✓ ZERO SUBJECT LEAKAGE")


# =============================================================================
# 12. VERIFY ROW COUNTS
# =============================================================================

print("\n" + "=" * 100)
print("FINAL DATA SPLIT")
print("=" * 100)

print("Development rows:", len(development_df))
print("Validation rows :", len(validation_df))
print("Locked test rows:", len(locked_test_df))

print("\nDevelopment classes:")
print(
    np.bincount(
        development_df[TARGET_COL].to_numpy()
    )
)

print("\nValidation classes:")
print(
    np.bincount(
        validation_df[TARGET_COL].to_numpy()
    )
)

print("\nLocked test classes:")
print(
    np.bincount(
        locked_test_df[TARGET_COL].to_numpy()
    )
)


# =============================================================================
# 13. DEFINE RAW FEATURES
# =============================================================================
# Explicitly remove:
#
#   subject#
#   total_UPDRS
#   severity
#   split
#
# total_UPDRS is deliberately excluded because it is closely related to
# the target and could cause leakage / shortcut learning.
# =============================================================================

EXCLUDED_COLUMNS = {
    SUBJECT_COL,
    "total_UPDRS",
    TARGET_COL,
    SPLIT_COL
}

feature_columns = [
    c
    for c in df.columns
    if c not in EXCLUDED_COLUMNS
]

print("\n" + "=" * 100)
print("FEATURE CONFIGURATION")
print("=" * 100)

print("Raw feature count:", len(feature_columns))
print("Features:")
print(feature_columns)

if len(feature_columns) != 18:
    print(
        "\nWARNING: Expected 18 raw predictive features "
        "from the 22-column original dataset."
    )


# =============================================================================
# 14. RAW MATRICES
# =============================================================================

X_dev_raw = development_df[
    feature_columns
].apply(pd.to_numeric, errors="coerce").to_numpy(
    dtype=np.float32
)

X_val_raw = validation_df[
    feature_columns
].apply(pd.to_numeric, errors="coerce").to_numpy(
    dtype=np.float32
)

X_test_raw = locked_test_df[
    feature_columns
].apply(pd.to_numeric, errors="coerce").to_numpy(
    dtype=np.float32
)

y_dev = development_df[
    TARGET_COL
].astype(int).to_numpy()

y_val = validation_df[
    TARGET_COL
].astype(int).to_numpy()

y_test = locked_test_df[
    TARGET_COL
].astype(int).to_numpy()


# =============================================================================
# 15. DEVELOPMENT-ONLY IMPUTATION
# =============================================================================

print("\n" + "=" * 100)
print("DEVELOPMENT-ONLY PREPROCESSING")
print("=" * 100)

imputer = SimpleImputer(
    strategy="median"
)

X_dev_imp = imputer.fit_transform(
    X_dev_raw
).astype(np.float32)

X_val_imp = imputer.transform(
    X_val_raw
).astype(np.float32)

X_test_imp = imputer.transform(
    X_test_raw
).astype(np.float32)

print("✓ Imputer fitted on development only")


# =============================================================================
# 16. DEVELOPMENT-ONLY SCALING
# =============================================================================

scaler = StandardScaler()

X_dev_scaled = scaler.fit_transform(
    X_dev_imp
).astype(np.float32)

X_val_scaled = scaler.transform(
    X_val_imp
).astype(np.float32)

X_test_scaled = scaler.transform(
    X_test_imp
).astype(np.float32)

print("✓ Scaler fitted on development only")


# =============================================================================
# 17. DEVELOPMENT-ONLY ANOVA
# =============================================================================

print("\n" + "=" * 100)
print("DEVELOPMENT-ONLY ANOVA FEATURE SELECTION")
print("=" * 100)

anova_selector = SelectKBest(
    score_func=f_classif,
    k=min(N_SELECTED_FEATURES, X_dev_scaled.shape[1])
)

X_dev_anova = anova_selector.fit_transform(
    X_dev_scaled,
    y_dev
).astype(np.float32)

X_val_anova = anova_selector.transform(
    X_val_scaled
).astype(np.float32)

X_test_anova = anova_selector.transform(
    X_test_scaled
).astype(np.float32)

selected_indices = anova_selector.get_support(
    indices=True
)

selected_features = [
    feature_columns[i]
    for i in selected_indices
]

print("Selected features:")
for i, feature in enumerate(selected_features):
    print(f"{i+1:02d}. {feature}")

print("\nShapes:")
print("Development:", X_dev_anova.shape)
print("Validation :", X_val_anova.shape)
print("Locked test:", X_test_anova.shape)

assert X_dev_anova.shape[1] == N_SELECTED_FEATURES
assert X_val_anova.shape[1] == N_SELECTED_FEATURES
assert X_test_anova.shape[1] == N_SELECTED_FEATURES

print("✓ ANOVA fitted only on development")


# =============================================================================
# 18. CONVERT TO TENSORS
# =============================================================================

X_dev_tensor = torch.tensor(
    X_dev_anova,
    dtype=torch.float32,
    device=DEVICE
)

X_val_tensor = torch.tensor(
    X_val_anova,
    dtype=torch.float32,
    device=DEVICE
)

X_test_tensor = torch.tensor(
    X_test_anova,
    dtype=torch.float32,
    device=DEVICE
)

y_dev_tensor = torch.tensor(
    y_dev,
    dtype=torch.float32,
    device=DEVICE
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.float32,
    device=DEVICE
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32,
    device=DEVICE
)


# =============================================================================
# 19. CLASS WEIGHTS
# =============================================================================

classes = np.array([0, 1])

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_dev
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=DEVICE
)

print("\nClass weights:", class_weights_np)


# =============================================================================
# 20. QUANTUM DEVICE
# =============================================================================

print("\n" + "=" * 100)
print("QUANTUM DEVICE")
print("=" * 100)

# Default.qubit is a CPU simulator.
dev_qnode = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

print("✓ PennyLane default.qubit")
print("✓ CPU-only")
print("✓", N_QUBITS, "qubits")


# =============================================================================
# 21. VQC MODEL
# =============================================================================

class HybridVQC(nn.Module):

    def __init__(
        self,
        input_dim,
        n_qubits=4,
        depth=2,
        encoding="RYRZ",
        entangle=True,
        hidden_dim=16,
        dropout=0.20
    ):

        super().__init__()

        self.input_dim = input_dim
        self.n_qubits = n_qubits
        self.depth = depth
        self.encoding = encoding
        self.entangle = entangle

        # ---------------------------------------------------------------------
        # Classical projection
        # ---------------------------------------------------------------------

        self.projection = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim
            ),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim,
                n_qubits
            )
        )

        # ---------------------------------------------------------------------
        # Classical residual branch
        # ---------------------------------------------------------------------

        self.residual_branch = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim
            ),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim,
                8
            ),

            nn.GELU()
        )

        # ---------------------------------------------------------------------
        # Quantum weights
        # ---------------------------------------------------------------------

        if encoding == "RYRZ":

            self.q_weights = nn.Parameter(
                0.05 * torch.randn(
                    depth,
                    n_qubits,
                    2,
                    dtype=torch.float32
                )
            )

        else:

            self.q_weights = nn.Parameter(
                0.05 * torch.randn(
                    depth,
                    n_qubits,
                    dtype=torch.float32
                )
            )

        # ---------------------------------------------------------------------
        # Fusion
        # ---------------------------------------------------------------------

        self.fusion = nn.Sequential(

            nn.Linear(
                n_qubits + 8,
                16
            ),

            nn.LayerNorm(
                16
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                16,
                1
            )
        )


    def quantum_circuit(
        self,
        angles
    ):

        # Encoding
        if self.encoding == "RYRZ":

            for q in range(self.n_qubits):

                qml.RY(
                    angles[q],
                    wires=q
                )

                qml.RZ(
                    angles[q],
                    wires=q
                )

        else:

            for q in range(self.n_qubits):

                qml.RY(
                    angles[q],
                    wires=q
                )

        # Variational layers
        for d in range(self.depth):

            for q in range(self.n_qubits):

                if self.encoding == "RYRZ":

                    qml.RY(
                        self.q_weights[d, q, 0],
                        wires=q
                    )

                    qml.RZ(
                        self.q_weights[d, q, 1],
                        wires=q
                    )

                else:

                    qml.RY(
                        self.q_weights[d, q],
                        wires=q
                    )

            if self.entangle:

                for q in range(
                    self.n_qubits - 1
                ):

                    qml.CNOT(
                        wires=[q, q + 1]
                    )

        return [
            qml.expval(
                qml.PauliZ(q)
            )
            for q in range(self.n_qubits)
        ]


    def forward(
        self,
        x
    ):

        # ---------------------------------------------------------------------
        # Classical projection
        # ---------------------------------------------------------------------

        projected = self.projection(x)

        # Keep angles bounded
        angles = torch.tanh(
            projected
        ) * np.pi

        # ---------------------------------------------------------------------
        # Quantum branch
        # ---------------------------------------------------------------------

        quantum_outputs = []

        for i in range(
            angles.shape[0]
        ):

            q_out = qml.QNode(
                self.quantum_circuit,
                dev_qnode,
                interface="torch",
                diff_method="backprop"
            )(
                angles[i]
            )

            # IMPORTANT:
            # Explicitly keep this as a torch tensor.
            q_out = torch.stack(
                list(q_out)
            )

            quantum_outputs.append(
                q_out
            )

        quantum = torch.stack(
            quantum_outputs
        )

        quantum = quantum.float()

        # ---------------------------------------------------------------------
        # Residual classical branch
        # ---------------------------------------------------------------------

        residual = self.residual_branch(
            x
        )

        # ---------------------------------------------------------------------
        # Fusion
        # ---------------------------------------------------------------------

        fused = torch.cat(
            [
                quantum,
                residual
            ],
            dim=1
        )

        logits = self.fusion(
            fused
        ).squeeze(1)

        return logits


# =============================================================================
# 22. FASTER QUANTUM MODEL
# =============================================================================
# The previous implementation creates a QNode inside forward().
# For CPU experiments this is acceptable for the relatively small dataset,
# but we also process data in batches.
#
# Batch size is deliberately small to prevent memory / simulator issues.
# =============================================================================


# =============================================================================
# 23. METRIC FUNCTION
# =============================================================================

def calculate_metrics(
    y_true,
    probabilities,
    threshold=0.5
):

    y_true = np.asarray(
        y_true
    ).astype(int)

    probabilities = np.asarray(
        probabilities
    ).reshape(-1)

    predictions = (
        probabilities >= threshold
    ).astype(int)

    metrics = {}

    metrics["Accuracy"] = accuracy_score(
        y_true,
        predictions
    )

    metrics["Balanced_Accuracy"] = balanced_accuracy_score(
        y_true,
        predictions
    )

    metrics["F1"] = f1_score(
        y_true,
        predictions,
        zero_division=0
    )

    metrics["MCC"] = matthews_corrcoef(
        y_true,
        predictions
    )

    if len(np.unique(y_true)) == 2:

        metrics["ROC_AUC"] = roc_auc_score(
            y_true,
            probabilities
        )

        metrics["PR_AUC"] = average_precision_score(
            y_true,
            probabilities
        )

    else:

        metrics["ROC_AUC"] = np.nan
        metrics["PR_AUC"] = np.nan

    return metrics


# =============================================================================
# 24. SAFE NUMPY CONVERSION
# =============================================================================

def safe_numpy(x):

    if isinstance(
        x,
        torch.Tensor
    ):

        return (
            x.detach()
             .cpu()
             .numpy()
        )

    return np.asarray(x)


# =============================================================================
# 25. BATCH PREDICTION
# =============================================================================

@torch.no_grad()
def predict_probabilities(
    model,
    X,
    batch_size=16
):

    model.eval()

    outputs = []

    for start in range(
        0,
        len(X),
        batch_size
    ):

        end = min(
            start + batch_size,
            len(X)
        )

        batch = X[
            start:end
        ]

        logits = model(
            batch
        )

        logits = logits.detach().cpu()

        probabilities = torch.sigmoid(
            logits
        )

        outputs.append(
            probabilities.numpy()
        )

    if len(outputs) == 0:
        return np.empty(
            (0,),
            dtype=np.float32
        )

    return np.concatenate(
        outputs
    ).astype(np.float32)


# =============================================================================
# 26. VALIDATION THRESHOLD SEARCH
# =============================================================================
# Threshold is selected ONLY on validation.
# Test is never involved.
# =============================================================================

def find_best_threshold(
    y_true,
    probabilities
):

    best_threshold = 0.50
    best_score = -np.inf
    best_metrics = None

    thresholds = np.arange(
        0.20,
        0.81,
        0.01
    )

    for threshold in thresholds:

        metrics = calculate_metrics(
            y_true,
            probabilities,
            threshold
        )

        # Balanced Accuracy is primary.
        # F1 is secondary.
        score = (
            0.70 * metrics["Balanced_Accuracy"]
            +
            0.30 * metrics["F1"]
        )

        if score > best_score:

            best_score = score
            best_threshold = float(
                threshold
            )
            best_metrics = metrics

    return (
        best_threshold,
        best_metrics,
        best_score
    )


# =============================================================================
# 27. TRAIN ONE MODEL
# =============================================================================

def train_one_model(
    depth,
    lr,
    weight_decay,
    seed
):

    set_seed(seed)

    print("\n" + "-" * 100)
    print(
        f"CONFIGURATION: "
        f"RYRZ_D{depth}_LR{lr:.3f}_WD{weight_decay}"
        f"_SEED{seed}"
    )
    print("-" * 100)

    model = HybridVQC(
        input_dim=N_SELECTED_FEATURES,
        n_qubits=N_QUBITS,
        depth=depth,
        encoding="RYRZ",
        entangle=True,
        hidden_dim=PROJECTION_HIDDEN,
        dropout=DROPOUT
    ).to(DEVICE)

    # Weighted BCE
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=(
            class_weights[1]
            /
            class_weights[0]
        )
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_state = None
    best_epoch = 1
    best_val_score = -np.inf
    best_val_metrics = None

    no_improvement = 0

    # -------------------------------------------------------------------------
    # TRAINING
    # -------------------------------------------------------------------------

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        model.train()

        # Shuffle development rows ONLY
        permutation = torch.randperm(
            len(X_dev_tensor)
        )

        epoch_losses = []

        for start in range(
            0,
            len(permutation),
            BATCH_SIZE
        ):

            idx = permutation[
                start:
                start + BATCH_SIZE
            ]

            xb = X_dev_tensor[
                idx
            ]

            yb = y_dev_tensor[
                idx
            ]

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                xb
            )

            loss = criterion(
                logits,
                yb
            )

            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            epoch_losses.append(
                float(loss.detach().cpu())
            )

        # ---------------------------------------------------------------------
        # VALIDATION
        # ---------------------------------------------------------------------

        val_prob = predict_probabilities(
            model,
            X_val_tensor,
            batch_size=BATCH_SIZE
        )

        # Temporary 0.5 threshold during epoch selection
        val_metrics = calculate_metrics(
            y_val,
            val_prob,
            threshold=0.5
        )

        selection_score = (
            0.55 *
            val_metrics["Balanced_Accuracy"]
            +
            0.30 *
            val_metrics["F1"]
            +
            0.15 *
            val_metrics["MCC"]
        )

        if (
            selection_score
            >
            best_val_score
        ):

            best_val_score = selection_score

            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            best_val_metrics = (
                val_metrics.copy()
            )

            no_improvement = 0

        else:

            no_improvement += 1

        # Print useful progress
        if (
            epoch == 1
            or epoch % 5 == 0
            or epoch == best_epoch
        ):

            print(
                f"Epoch {epoch:03d} | "
                f"Loss {np.mean(epoch_losses):.5f} | "
                f"Val BA "
                f"{val_metrics['Balanced_Accuracy']:.4f} | "
                f"Val F1 "
                f"{val_metrics['F1']:.4f} | "
                f"Val AUC "
                f"{val_metrics['ROC_AUC']:.4f}"
            )

        if no_improvement >= PATIENCE:

            print(
                f"Early stopping at epoch {epoch}"
            )

            break

    # -------------------------------------------------------------------------
    # RESTORE BEST MODEL
    # -------------------------------------------------------------------------

    if best_state is None:

        raise RuntimeError(
            "No valid model state was produced."
        )

    model.load_state_dict(
        best_state
    )

    model.eval()

    for parameter in model.parameters():

        parameter.requires_grad = False

    # -------------------------------------------------------------------------
    # VALIDATION THRESHOLD
    # -------------------------------------------------------------------------

    val_prob = predict_probabilities(
        model,
        X_val_tensor,
        batch_size=BATCH_SIZE
    )

    threshold, threshold_metrics, threshold_score = (
        find_best_threshold(
            y_val,
            val_prob
        )
    )

    print(
        f"\nBest epoch: {best_epoch}"
    )

    print(
        f"Validation threshold: {threshold:.2f}"
    )

    print(
        f"Validation BA: "
        f"{threshold_metrics['Balanced_Accuracy']:.4f}"
    )

    print(
        f"Validation F1: "
        f"{threshold_metrics['F1']:.4f}"
    )

    print(
        f"Validation ROC-AUC: "
        f"{threshold_metrics['ROC_AUC']:.4f}"
    )

    return {
        "model": model,
        "state_dict": copy.deepcopy(
            model.state_dict()
        ),
        "depth": depth,
        "lr": lr,
        "weight_decay": weight_decay,
        "seed": seed,
        "best_epoch": best_epoch,
        "threshold": threshold,
        "metrics": threshold_metrics,
        "selection_score": threshold_score
    }


# =============================================================================
# 28. HYPERPARAMETER SEARCH
# =============================================================================

print("\n" + "=" * 100)
print("HYBRID VQC DEVELOPMENT / VALIDATION SEARCH")
print("=" * 100)

results = []

best_experiment = None
best_global_score = -np.inf

experiment_number = 0

for depth in DEPTHS:

    for lr in LEARNING_RATES:

        for weight_decay in WEIGHT_DECAYS:

            for seed in SEEDS:

                experiment_number += 1

                try:

                    experiment = train_one_model(
                        depth=depth,
                        lr=lr,
                        weight_decay=weight_decay,
                        seed=seed
                    )

                    metrics = experiment[
                        "metrics"
                    ]

                    row = {

                        "Input":
                            "ANOVA12",

                        "Model":
                            f"RYRZ_D{depth}"
                            f"_LR{lr:.3f}"
                            f"_WD{weight_decay}",

                        "Seed":
                            seed,

                        "Depth":
                            depth,

                        "Learning_Rate":
                            lr,

                        "Weight_Decay":
                            weight_decay,

                        "Best_Epoch":
                            experiment[
                                "best_epoch"
                            ],

                        "Threshold":
                            experiment[
                                "threshold"
                            ],

                        "Accuracy":
                            metrics[
                                "Accuracy"
                            ],

                        "Balanced_Accuracy":
                            metrics[
                                "Balanced_Accuracy"
                            ],

                        "F1":
                            metrics[
                                "F1"
                            ],

                        "MCC":
                            metrics[
                                "MCC"
                            ],

                        "ROC_AUC":
                            metrics[
                                "ROC_AUC"
                            ],

                        "PR_AUC":
                            metrics[
                                "PR_AUC"
                            ],

                        "SelectionScore":
                            experiment[
                                "selection_score"
                            ]
                    }

                    results.append(
                        row
                    )

                    # Primary model selection
                    score = (
                        0.55 *
                        metrics[
                            "Balanced_Accuracy"
                        ]
                        +
                        0.30 *
                        metrics[
                            "F1"
                        ]
                        +
                        0.15 *
                        metrics[
                            "MCC"
                        ]
                    )

                    if score > best_global_score:

                        best_global_score = score

                        best_experiment = (
                            experiment
                        )

                        print(
                            "\n>>> NEW BEST VQC <<<"
                        )

                        print(
                            "Depth:",
                            depth
                        )

                        print(
                            "LR:",
                            lr
                        )

                        print(
                            "WD:",
                            weight_decay
                        )

                        print(
                            "Seed:",
                            seed
                        )

                        print(
                            "Validation BA:",
                            metrics[
                                "Balanced_Accuracy"
                            ]
                        )

                except Exception as e:

                    print(
                        "\nFAILED configuration:"
                    )

                    print(
                        f"D{depth} "
                        f"LR{lr} "
                        f"WD{weight_decay} "
                        f"SEED{seed}"
                    )

                    print(
                        repr(e)
                    )

                    # Clean memory
                    gc.collect()

                    continue


# =============================================================================
# 29. RESULTS TABLE
# =============================================================================

if len(results) == 0:

    raise RuntimeError(
        "All VQC configurations failed."
    )

comparison_df = pd.DataFrame(
    results
)

comparison_df = comparison_df.sort_values(
    by=[
        "Balanced_Accuracy",
        "F1",
        "MCC",
        "ROC_AUC"
    ],
    ascending=False
).reset_index(
    drop=True
)

print("\n" + "=" * 100)
print("HYBRID VQC INTERNAL MODEL COMPARISON")
print("=" * 100)

print(
    comparison_df.to_string(
        index=False
    )
)


# =============================================================================
# 30. SELECT BEST MODEL
# =============================================================================

best_row = comparison_df.iloc[0]

print("\n" + "=" * 100)
print("BEST INTERNAL HYBRID VQC")
print("=" * 100)

print(
    "Input:",
    best_row["Input"]
)

print(
    "Model:",
    best_row["Model"]
)

print(
    "Seed:",
    int(best_row["Seed"])
)

print(
    "Depth:",
    int(best_row["Depth"])
)

print(
    "Learning rate:",
    best_row["Learning_Rate"]
)

print(
    "Weight decay:",
    best_row["Weight_Decay"]
)

print(
    "Best epoch:",
    int(best_row["Best_Epoch"])
)

print(
    "Validation threshold:",
    best_row["Threshold"]
)

print(
    "Validation Accuracy:",
    best_row["Accuracy"]
)

print(
    "Validation Balanced Accuracy:",
    best_row["Balanced_Accuracy"]
)

print(
    "Validation F1:",
    best_row["F1"]
)

print(
    "Validation MCC:",
    best_row["MCC"]
)

print(
    "Validation ROC-AUC:",
    best_row["ROC_AUC"]
)

print(
    "Validation PR-AUC:",
    best_row["PR_AUC"]
)


# =============================================================================
# 31. RECONSTRUCT BEST MODEL CLEANLY
# =============================================================================
# We do NOT pickle the preprocessing objects.
# This avoids the previous:
#
#     ModuleNotFoundError: No module named 'age'
#
# Instead, everything needed for test preprocessing is saved explicitly.
# =============================================================================

BEST_DEPTH = int(
    best_row["Depth"]
)

BEST_LR = float(
    best_row["Learning_Rate"]
)

BEST_WD = float(
    best_row["Weight_Decay"]
)

BEST_SEED = int(
    best_row["Seed"]
)

BEST_THRESHOLD = float(
    best_row["Threshold"]
)

set_seed(
    BEST_SEED
)

best_model = HybridVQC(
    input_dim=N_SELECTED_FEATURES,
    n_qubits=N_QUBITS,
    depth=BEST_DEPTH,
    encoding="RYRZ",
    entangle=True,
    hidden_dim=PROJECTION_HIDDEN,
    dropout=DROPOUT
).to(DEVICE)

best_model.load_state_dict(
    best_experiment[
        "state_dict"
    ]
)

best_model.eval()

for parameter in best_model.parameters():

    parameter.requires_grad = False

print("\n" + "=" * 100)
print("BEST VQC FROZEN")
print("=" * 100)

print("✓ Best model reconstructed")
print("✓ Best development/validation weights loaded")
print("✓ Model in evaluation mode")
print("✓ All parameters frozen")
print("✓ No optimizer")
print("✓ No further training")
print("✓ Locked test has NOT been used")


# =============================================================================
# 32. SAVE MODEL CHECKPOINT
# =============================================================================
# State dictionary only.
# This avoids PyTorch 2.6 pickle/weights-only problems.
# =============================================================================

MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "phase21N_best_hybrid_vqc_state.pt"
)

checkpoint = {

    "model_state_dict":
        best_model.state_dict(),

    "input_dim":
        N_SELECTED_FEATURES,

    "n_qubits":
        N_QUBITS,

    "depth":
        BEST_DEPTH,

    "encoding":
        "RYRZ",

    "entangle":
        True,

    "hidden_dim":
        PROJECTION_HIDDEN,

    "dropout":
        DROPOUT,

    "threshold":
        BEST_THRESHOLD,

    "selected_features":
        selected_features,

    "feature_columns":
        feature_columns,

    "best_info":
        best_row.to_dict()
}

torch.save(
    checkpoint,
    MODEL_PATH
)

print(
    "Saved model:",
    MODEL_PATH
)


# =============================================================================
# 33. SAVE PREPROCESSING PARAMETERS WITHOUT PICKLE
# =============================================================================
# This is intentional.
#
# Instead of pickle.dump(imputer/scaler/selector), save the numerical
# parameters directly.
#
# This completely avoids the previous:
#     ModuleNotFoundError: No module named 'age'
# =============================================================================

PREPROCESS_PATH = os.path.join(
    OUTPUT_DIR,
    "phase21N_preprocessing.npz"
)

np.savez(
    PREPROCESS_PATH,

    imputer_statistics=
        imputer.statistics_,

    scaler_mean=
        scaler.mean_,

    scaler_scale=
        scaler.scale_,

    anova_scores=
        anova_selector.scores_,

    anova_pvalues=
        anova_selector.pvalues_,

    selected_indices=
        selected_indices,

    selected_features=
        np.array(
            selected_features,
            dtype=object
        ),

    feature_columns=
        np.array(
            feature_columns,
            dtype=object
        )
)

print(
    "Saved preprocessing:",
    PREPROCESS_PATH
)


# =============================================================================
# 34. SAVE INTERNAL RESULTS
# =============================================================================

RESULTS_PATH = os.path.join(
    OUTPUT_DIR,
    "phase21N_internal_model_comparison.csv"
)

comparison_df.to_csv(
    RESULTS_PATH,
    index=False
)

print(
    "Saved comparison:",
    RESULTS_PATH
)


# =============================================================================
# 35. IMPORTANT CHECK BEFORE TEST
# =============================================================================

print("\n" + "=" * 100)
print("PRE-TEST INTEGRITY CHECK")
print("=" * 100)

assert len(X_dev_anova) == len(y_dev)
assert len(X_val_anova) == len(y_val)
assert len(X_test_anova) == len(y_test)

assert X_dev_anova.shape[1] == 12
assert X_val_anova.shape[1] == 12
assert X_test_anova.shape[1] == 12

assert len(
    dev_subjects & val_subjects
) == 0

assert len(
    dev_subjects & test_subjects
) == 0

assert len(
    val_subjects & test_subjects
) == 0

print("✓ Development rows:", len(y_dev))
print("✓ Validation rows :", len(y_val))
print("✓ Locked test rows:", len(y_test))

print("✓ ANOVA12 dimensions verified")
print("✓ ZERO subject leakage")
print("✓ Preprocessing fitted only on development")
print("✓ Threshold selected only using validation")
print("✓ Model frozen")
print("✓ Test has not been used")


# =============================================================================
# 36. LOCKED TEST — ONE TIME ONLY
# =============================================================================

print("\n" + "=" * 100)
print("PHASE 21N — ONE-TIME LOCKED TEST EVALUATION")
print("=" * 100)

print("✓ Frozen model")
print("✓ CPU-only")
print("✓ No optimizer")
print("✓ No training")
print("✓ No hyperparameter tuning")
print("✓ Validation-selected threshold:", BEST_THRESHOLD)


# =============================================================================
# 37. TEST INFERENCE
# =============================================================================

test_probabilities = predict_probabilities(
    best_model,
    X_test_tensor,
    batch_size=BATCH_SIZE
)

if len(test_probabilities) != len(y_test):

    raise RuntimeError(
        f"Prediction count mismatch: "
        f"{len(test_probabilities)} vs {len(y_test)}"
    )

print(
    "✓ Prediction count:",
    len(test_probabilities)
)


# =============================================================================
# 38. FINAL TEST METRICS
# =============================================================================

test_metrics = calculate_metrics(
    y_test,
    test_probabilities,
    threshold=BEST_THRESHOLD
)

test_predictions = (
    test_probabilities
    >= BEST_THRESHOLD
).astype(int)


# =============================================================================
# 39. FINAL RESULT
# =============================================================================

print("\n" + "=" * 100)
print("FINAL FROZEN HYBRID VQC LOCKED TEST RESULT")
print("=" * 100)

print(
    f"Threshold          : {BEST_THRESHOLD:.4f}"
)

print(
    f"Accuracy           : "
    f"{test_metrics['Accuracy']:.4f}"
)

print(
    f"Balanced Accuracy  : "
    f"{test_metrics['Balanced_Accuracy']:.4f}"
)

print(
    f"F1                 : "
    f"{test_metrics['F1']:.4f}"
)

print(
    f"MCC                : "
    f"{test_metrics['MCC']:.4f}"
)

print(
    f"ROC-AUC            : "
    f"{test_metrics['ROC_AUC']:.4f}"
)

print(
    f"PR-AUC             : "
    f"{test_metrics['PR_AUC']:.4f}"
)


# =============================================================================
# 40. CONFUSION MATRIX
# =============================================================================

cm = confusion_matrix(
    y_test,
    test_predictions
)

print("\n" + "=" * 100)
print("CONFUSION MATRIX")
print("=" * 100)

print(cm)


# =============================================================================
# 41. CLASSIFICATION REPORT
# =============================================================================

print("\n" + "=" * 100)
print("CLASSIFICATION REPORT")
print("=" * 100)

print(
    classification_report(
        y_test,
        test_predictions,
        digits=4,
        zero_division=0
    )
)


# =============================================================================
# 42. SAVE TEST PREDICTIONS
# =============================================================================

test_predictions_df = locked_test_df[
    [
        SUBJECT_COL
    ]
].copy()

test_predictions_df[
    "true_severity"
] = y_test

test_predictions_df[
    "predicted_probability"
] = test_probabilities

test_predictions_df[
    "predicted_severity"
] = test_predictions

test_predictions_df[
    "threshold"
] = BEST_THRESHOLD


PREDICTIONS_PATH = os.path.join(
    OUTPUT_DIR,
    "phase21N_locked_test_predictions.csv"
)

test_predictions_df.to_csv(
    PREDICTIONS_PATH,
    index=False
)


# =============================================================================
# 43. SAVE TEST METRICS
# =============================================================================

final_metrics = {

    "Model":
        best_row["Model"],

    "Input":
        "ANOVA12",

    "Seed":
        BEST_SEED,

    "Depth":
        BEST_DEPTH,

    "Learning_Rate":
        BEST_LR,

    "Weight_Decay":
        BEST_WD,

    "Best_Epoch":
        int(best_row["Best_Epoch"]),

    "Validation_Threshold":
        BEST_THRESHOLD,

    "Test_Accuracy":
        test_metrics["Accuracy"],

    "Test_Balanced_Accuracy":
        test_metrics["Balanced_Accuracy"],

    "Test_F1":
        test_metrics["F1"],

    "Test_MCC":
        test_metrics["MCC"],

    "Test_ROC_AUC":
        test_metrics["ROC_AUC"],

    "Test_PR_AUC":
        test_metrics["PR_AUC"],

    "Development_Rows":
        len(y_dev),

    "Validation_Rows":
        len(y_val),

    "Locked_Test_Rows":
        len(y_test),

    "Development_Subjects":
        len(dev_subjects),

    "Validation_Subjects":
        len(val_subjects),

    "Locked_Test_Subjects":
        len(test_subjects)
}

metrics_df = pd.DataFrame(
    [final_metrics]
)

METRICS_PATH = os.path.join(
    OUTPUT_DIR,
    "phase21N_locked_test_metrics.csv"
)

metrics_df.to_csv(
    METRICS_PATH,
    index=False
)


# =============================================================================
# 44. SAVE SPLIT INFORMATION
# =============================================================================

development_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "phase21N_development.csv"
    ),
    index=False
)

validation_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "phase21N_validation.csv"
    ),
    index=False
)

locked_test_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "phase21N_locked_test.csv"
    ),
    index=False
)


# =============================================================================
# 45. FINAL INTEGRITY REPORT
# =============================================================================

print("\n" + "=" * 100)
print("PHASE 21N INTEGRITY CHECK")
print("=" * 100)

print("Development subjects:", len(dev_subjects))
print("Validation subjects :", len(val_subjects))
print("Locked-test subjects:", len(test_subjects))

print(
    "Development/validation overlap:",
    dev_subjects & val_subjects
)

print(
    "Development/test overlap:",
    dev_subjects & test_subjects
)

print(
    "Validation/test overlap:",
    val_subjects & test_subjects
)

print("\n✓ ZERO SUBJECT LEAKAGE")
print("✓ total_UPDRS excluded")
print("✓ subject# excluded")
print("✓ severity used only as target")
print("✓ split used only to establish subject-wise partitions")
print("✓ Imputation fitted on development only")
print("✓ Scaling fitted on development only")
print("✓ ANOVA fitted on development only")
print("✓ Validation used for model selection")
print("✓ Validation used for threshold selection")
print("✓ Locked test NOT used for tuning")
print("✓ Model frozen before test")
print("✓ No optimizer during test")
print("✓ No training during test")
print("✓ Locked test evaluated exactly once")


# =============================================================================
# 46. OUTPUTS
# =============================================================================

print("\n" + "=" * 100)
print("SAVED OUTPUTS")
print("=" * 100)

print("✓", MODEL_PATH)
print("✓", PREPROCESS_PATH)
print("✓", RESULTS_PATH)
print("✓", PREDICTIONS_PATH)
print("✓", METRICS_PATH)

print("\n" + "=" * 100)
print("PHASE 21N — COMPLETED")
print("=" * 100)

print(
    "\nFINAL TEST BALANCED ACCURACY:",
    f"{test_metrics['Balanced_Accuracy']:.4f}"
)

print(
    "FINAL TEST F1:",
    f"{test_metrics['F1']:.4f}"
)

print(
    "FINAL TEST ROC-AUC:",
    f"{test_metrics['ROC_AUC']:.4f}"
)

print(
    "\nIMPORTANT:"
)

print(
    "The locked-test result above must now be treated as FINAL "
    "for this experiment."
)

print(
    "Do not change the threshold, preprocessing, architecture, "
    "or hyperparameters based on the locked-test result."
)

PHASE 21N — COMPLETE HYBRID VQC PIPELINE
Device: cpu
CUDA available: False
✓ CUDA deliberately disabled
✓ VQC will run entirely on CPU

LOADING SOURCE DATA
Original dataset: (5875, 22)
Split dataset   : (5875, 3)

Original columns:
['subject#', 'age', 'sex', 'test_time', 'motor_UPDRS', 'total_UPDRS', 'Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP', 'Jitter:PPQ5', 'Jitter:DDP', 'Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'Shimmer:APQ11', 'Shimmer:DDA', 'NHR', 'HNR', 'RPDE', 'DFA', 'PPE']

Split columns:
['subject#', 'severity', 'split']

VALIDATING SUBJECT-WISE SPLIT
✓ Split is constant within every subject
Unique subjects in original: 42
Unique subjects in split: 42

✓ Same number of rows
✓ Subject row alignment verified
✓ Severity attached row-by-row
✓ Split attached row-by-row

Split labels:
split
train    4585
test     1290
Name: count, dtype: int64

Unique severity:
[np.int64(0), np.int64(1)]

Normalized split labels:
split
train    4585
test     1290
Name: count, dtype: int

RuntimeError: Could not identify validation split.

In [26]:
# ============================================================
# PHASE 21 — ROBUST DATA LOADING
# Uses the finalized Phase 21M split directly
# ============================================================

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# 1. DEVICE — CPU ONLY
# ------------------------------------------------------------

DEVICE = torch.device("cpu")

print("=" * 100)
print("PHASE 21 — HYBRID VQC")
print("=" * 100)
print("Device:", DEVICE)
print("✓ CPU-only execution")
print("✓ CUDA will NOT be used")
print()


# ------------------------------------------------------------
# 2. INSTALL / LOAD PENNYLANE
# ------------------------------------------------------------

try:
    import pennylane as qml

except ModuleNotFoundError:

    print("PennyLane not found.")
    print("Installing PennyLane...")

    import subprocess
    import sys

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pennylane"
    ])

    import pennylane as qml

print("PennyLane version:", qml.__version__)
print()


# ------------------------------------------------------------
# 3. EXACT PHASE 21M FILES
# ------------------------------------------------------------

BASE_DIR = "/kaggle/working/phase21M_outputs"

DEV_PATH = os.path.join(
    BASE_DIR,
    "phase21M_development.csv"
)

VAL_PATH = os.path.join(
    BASE_DIR,
    "phase21M_validation.csv"
)

TEST_PATH = os.path.join(
    BASE_DIR,
    "phase21M_locked_test.csv"
)

print("=" * 100)
print("LOADING EXACT PHASE 21M SPLIT")
print("=" * 100)

for path in [DEV_PATH, VAL_PATH, TEST_PATH]:

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\nRequired Phase 21M file not found:\n{path}\n\n"
            "Run the Phase 21M data-preparation cell first."
        )

print("✓ Development file found")
print("✓ Validation file found")
print("✓ Locked-test file found")
print()


# ------------------------------------------------------------
# 4. LOAD DATA
# ------------------------------------------------------------

development_df = pd.read_csv(DEV_PATH)
validation_df = pd.read_csv(VAL_PATH)
locked_test_df = pd.read_csv(TEST_PATH)

print("=" * 100)
print("DATA SHAPES")
print("=" * 100)

print("Development:", development_df.shape)
print("Validation :", validation_df.shape)
print("Locked test:", locked_test_df.shape)
print()


# ------------------------------------------------------------
# 5. REQUIRED COLUMNS
# ------------------------------------------------------------

TARGET_COL = "severity"
SUBJECT_COL = "subject#"

for name, frame in [
    ("Development", development_df),
    ("Validation", validation_df),
    ("Locked test", locked_test_df)
]:

    if TARGET_COL not in frame.columns:
        raise RuntimeError(
            f"{TARGET_COL!r} missing from {name} dataset."
        )

    if SUBJECT_COL not in frame.columns:
        raise RuntimeError(
            f"{SUBJECT_COL!r} missing from {name} dataset."
        )

print("✓ severity found in all datasets")
print("✓ subject# found in all datasets")
print()


# ------------------------------------------------------------
# 6. EXACT EXPECTED ROW COUNTS
# ------------------------------------------------------------

assert len(development_df) == 3602, (
    f"Unexpected development size: {len(development_df)}"
)

assert len(validation_df) == 983, (
    f"Unexpected validation size: {len(validation_df)}"
)

assert len(locked_test_df) == 1290, (
    f"Unexpected locked-test size: {len(locked_test_df)}"
)

print("=" * 100)
print("EXACT PHASE 21M SPLIT")
print("=" * 100)

print("Development rows :", len(development_df))
print("Validation rows  :", len(validation_df))
print("Locked-test rows :", len(locked_test_df))
print()


# ------------------------------------------------------------
# 7. TARGETS
# ------------------------------------------------------------

y_dev = (
    development_df[TARGET_COL]
    .astype(int)
    .to_numpy()
)

y_val = (
    validation_df[TARGET_COL]
    .astype(int)
    .to_numpy()
)

y_test = (
    locked_test_df[TARGET_COL]
    .astype(int)
    .to_numpy()
)

print("Development target:", y_dev.shape)
print("Validation target :", y_val.shape)
print("Test target       :", y_test.shape)

print()
print("Development classes:", np.bincount(y_dev))
print("Validation classes :", np.bincount(y_val))
print("Test classes       :", np.bincount(y_test))
print()


# ------------------------------------------------------------
# 8. SUBJECT LEAKAGE CHECK
# ------------------------------------------------------------

dev_subjects = set(
    development_df[SUBJECT_COL].unique()
)

val_subjects = set(
    validation_df[SUBJECT_COL].unique()
)

test_subjects = set(
    locked_test_df[SUBJECT_COL].unique()
)

print("=" * 100)
print("SUBJECT LEAKAGE CHECK")
print("=" * 100)

print("Development subjects:", len(dev_subjects))
print("Validation subjects :", len(val_subjects))
print("Locked-test subjects:", len(test_subjects))

print(
    "Development / validation overlap:",
    dev_subjects & val_subjects
)

print(
    "Development / test overlap:",
    dev_subjects & test_subjects
)

print(
    "Validation / test overlap:",
    val_subjects & test_subjects
)

assert len(dev_subjects & val_subjects) == 0
assert len(dev_subjects & test_subjects) == 0
assert len(val_subjects & test_subjects) == 0

print("✓ ZERO SUBJECT LEAKAGE")
print()


# ------------------------------------------------------------
# 9. FEATURE COLUMNS
# ------------------------------------------------------------

EXCLUDED_COLUMNS = {
    SUBJECT_COL,
    TARGET_COL,
    "split",
    "total_UPDRS"
}

feature_columns = [
    c for c in development_df.columns
    if c not in EXCLUDED_COLUMNS
]

print("=" * 100)
print("FEATURES")
print("=" * 100)

print("Number of raw features:", len(feature_columns))
print(feature_columns)
print()


# ------------------------------------------------------------
# 10. NUMERIC FEATURES ONLY
# ------------------------------------------------------------

X_dev_raw = development_df[
    feature_columns
].apply(pd.to_numeric, errors="coerce")

X_val_raw = validation_df[
    feature_columns
].apply(pd.to_numeric, errors="coerce")

X_test_raw = locked_test_df[
    feature_columns
].apply(pd.to_numeric, errors="coerce")


# ------------------------------------------------------------
# 11. DEVELOPMENT-ONLY IMPUTATION
# ------------------------------------------------------------

imputer = SimpleImputer(
    strategy="median"
)

X_dev_imp = imputer.fit_transform(
    X_dev_raw
)

X_val_imp = imputer.transform(
    X_val_raw
)

X_test_imp = imputer.transform(
    X_test_raw
)

print("✓ Imputation fitted on development only")


# ------------------------------------------------------------
# 12. DEVELOPMENT-ONLY SCALING
# ------------------------------------------------------------

scaler = StandardScaler()

X_dev_scaled = scaler.fit_transform(
    X_dev_imp
)

X_val_scaled = scaler.transform(
    X_val_imp
)

X_test_scaled = scaler.transform(
    X_test_imp
)

print("✓ Scaling fitted on development only")


# ------------------------------------------------------------
# 13. ANOVA FEATURE SELECTION
# ------------------------------------------------------------

N_ANOVA = 12

N_ANOVA = min(
    N_ANOVA,
    X_dev_scaled.shape[1]
)

anova_selector = SelectKBest(
    score_func=f_classif,
    k=N_ANOVA
)

X_dev_anova = anova_selector.fit_transform(
    X_dev_scaled,
    y_dev
)

X_val_anova = anova_selector.transform(
    X_val_scaled
)

X_test_anova = anova_selector.transform(
    X_test_scaled
)

print()
print("=" * 100)
print("ANOVA FEATURE SELECTION")
print("=" * 100)

print("Selected features:", N_ANOVA)
print("Development:", X_dev_anova.shape)
print("Validation :", X_val_anova.shape)
print("Test       :", X_test_anova.shape)

print()
print("✓ ANOVA fitted on development only")
print("✓ Validation not used for feature selection")
print("✓ Test not used for feature selection")
print()


# ------------------------------------------------------------
# 14. FINAL VQC MATRICES
# ------------------------------------------------------------

X_dev_np = np.asarray(
    X_dev_anova,
    dtype=np.float32
)

X_val_np = np.asarray(
    X_val_anova,
    dtype=np.float32
)

X_test_np = np.asarray(
    X_test_anova,
    dtype=np.float32
)

print("=" * 100)
print("FINAL VQC INPUT")
print("=" * 100)

print("X_dev:", X_dev_np.shape)
print("X_val:", X_val_np.shape)
print("X_test:", X_test_np.shape)

print()
print("✓ Validation split identified directly")
print("✓ No split guessing")
print("✓ No test information used")
print("✓ Ready for VQC training")

PHASE 21 — HYBRID VQC
Device: cpu
✓ CPU-only execution
✓ CUDA will NOT be used

PennyLane version: 0.45.1

LOADING EXACT PHASE 21M SPLIT
✓ Development file found
✓ Validation file found
✓ Locked-test file found

DATA SHAPES
Development: (3602, 25)
Validation : (983, 25)
Locked test: (1290, 25)

✓ severity found in all datasets
✓ subject# found in all datasets

EXACT PHASE 21M SPLIT
Development rows : 3602
Validation rows  : 983
Locked-test rows : 1290

Development target: (3602,)
Validation target : (983,)
Test target       : (1290,)

Development classes: [2348 1254]
Validation classes : [589 394]
Test classes       : [797 493]

SUBJECT LEAKAGE CHECK
Development subjects: 26
Validation subjects : 7
Locked-test subjects: 9
Development / validation overlap: set()
Development / test overlap: set()
Validation / test overlap: set()
✓ ZERO SUBJECT LEAKAGE

FEATURES
Number of raw features: 21
['age', 'sex', 'test_time', 'motor_UPDRS', 'Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP', 'Jitter:PPQ5', '

In [27]:
# ============================================================
# CORRECT FEATURE SELECTION — NO TARGET LEAKAGE
# ============================================================

TARGET_COL = "severity"
SUBJECT_COL = "subject#"

EXCLUDED_COLUMNS = {
    SUBJECT_COL,
    TARGET_COL,
    "split",
    "total_UPDRS",
    "target"          # IMPORTANT: remove accidental target column
}

feature_columns = [
    c for c in development_df.columns
    if c not in EXCLUDED_COLUMNS
]

print("=" * 100)
print("FINAL FEATURE SET — LEAKAGE SAFE")
print("=" * 100)

print("Number of raw features:", len(feature_columns))

for i, col in enumerate(feature_columns, 1):
    print(f"{i:2d}. {col}")

# ------------------------------------------------------------
# RAW MATRICES
# ------------------------------------------------------------

X_dev_raw = development_df[
    feature_columns
].apply(pd.to_numeric, errors="coerce")

X_val_raw = validation_df[
    feature_columns
].apply(pd.to_numeric, errors="coerce")

X_test_raw = locked_test_df[
    feature_columns
].apply(pd.to_numeric, errors="coerce")

# ------------------------------------------------------------
# DEVELOPMENT-ONLY IMPUTATION
# ------------------------------------------------------------

imputer = SimpleImputer(
    strategy="median"
)

X_dev_imp = imputer.fit_transform(X_dev_raw)
X_val_imp = imputer.transform(X_val_raw)
X_test_imp = imputer.transform(X_test_raw)

print()
print("✓ Imputation fitted on development only")

# ------------------------------------------------------------
# DEVELOPMENT-ONLY SCALING
# ------------------------------------------------------------

scaler = StandardScaler()

X_dev_scaled = scaler.fit_transform(X_dev_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

print("✓ Scaling fitted on development only")

# ------------------------------------------------------------
# ANOVA — DEVELOPMENT ONLY
# ------------------------------------------------------------

N_ANOVA = min(12, X_dev_scaled.shape[1])

anova_selector = SelectKBest(
    score_func=f_classif,
    k=N_ANOVA
)

X_dev_anova = anova_selector.fit_transform(
    X_dev_scaled,
    y_dev
)

X_val_anova = anova_selector.transform(
    X_val_scaled
)

X_test_anova = anova_selector.transform(
    X_test_scaled
)

selected_mask = anova_selector.get_support()

selected_features = [
    feature_columns[i]
    for i, selected in enumerate(selected_mask)
    if selected
]

print()
print("=" * 100)
print("ANOVA12")
print("=" * 100)

print("Selected features:")

for i, feature in enumerate(selected_features, 1):
    print(f"{i:2d}. {feature}")

print()
print("Development:", X_dev_anova.shape)
print("Validation :", X_val_anova.shape)
print("Locked test:", X_test_anova.shape)

# ------------------------------------------------------------
# NUMPY
# ------------------------------------------------------------

X_dev_np = np.asarray(
    X_dev_anova,
    dtype=np.float32
)

X_val_np = np.asarray(
    X_val_anova,
    dtype=np.float32
)

X_test_np = np.asarray(
    X_test_anova,
    dtype=np.float32
)

# ------------------------------------------------------------
# FINAL SAFETY CHECK
# ------------------------------------------------------------

assert X_dev_np.shape[0] == len(y_dev)
assert X_val_np.shape[0] == len(y_val)
assert X_test_np.shape[0] == len(y_test)

assert "target" not in feature_columns
assert "severity" not in feature_columns
assert "total_UPDRS" not in feature_columns
assert "subject#" not in feature_columns
assert "split" not in feature_columns

print()
print("=" * 100)
print("FINAL VQC INPUT")
print("=" * 100)

print("X_dev:", X_dev_np.shape)
print("X_val:", X_val_np.shape)
print("X_test:", X_test_np.shape)

print()
print("✓ target excluded")
print("✓ severity excluded")
print("✓ total_UPDRS excluded")
print("✓ subject# excluded")
print("✓ split excluded")
print("✓ ZERO target leakage")
print("✓ Development-only imputation")
print("✓ Development-only scaling")
print("✓ Development-only ANOVA")
print("✓ Locked test untouched")
print()
print("READY FOR VQC TRAINING")

FINAL FEATURE SET — LEAKAGE SAFE
Number of raw features: 20
 1. age
 2. sex
 3. test_time
 4. motor_UPDRS
 5. Jitter(%)
 6. Jitter(Abs)
 7. Jitter:RAP
 8. Jitter:PPQ5
 9. Jitter:DDP
10. Shimmer
11. Shimmer(dB)
12. Shimmer:APQ3
13. Shimmer:APQ5
14. Shimmer:APQ11
15. Shimmer:DDA
16. NHR
17. HNR
18. RPDE
19. DFA
20. PPE

✓ Imputation fitted on development only
✓ Scaling fitted on development only

ANOVA12
Selected features:
 1. age
 2. sex
 3. motor_UPDRS
 4. Jitter(%)
 5. Jitter(Abs)
 6. Jitter:RAP
 7. Jitter:PPQ5
 8. Jitter:DDP
 9. HNR
10. RPDE
11. DFA
12. PPE

Development: (3602, 12)
Validation : (983, 12)
Locked test: (1290, 12)

FINAL VQC INPUT
X_dev: (3602, 12)
X_val: (983, 12)
X_test: (1290, 12)

✓ target excluded
✓ severity excluded
✓ total_UPDRS excluded
✓ subject# excluded
✓ split excluded
✓ ZERO target leakage
✓ Development-only imputation
✓ Development-only scaling
✓ Development-only ANOVA
✓ Locked test untouched

READY FOR VQC TRAINING


In [28]:
# ================================================================
# PHASE 21N — HYBRID VQC TRAINING + VALIDATION
# ================================================================
#
# INPUT:
#   X_dev_np, y_dev
#   X_val_np, y_val
#
# TEST:
#   X_test_np, y_test
#   --> NOT USED IN THIS CELL
#
# GOAL:
#   Train multiple leakage-safe Hybrid VQC configurations,
#   select the best one using DEVELOPMENT + VALIDATION only,
#   freeze it, and save the checkpoint.
#
# ================================================================

import os
import copy
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

import pennylane as qml

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score
)

# ================================================================
# 1. CPU ONLY
# ================================================================

DEVICE = torch.device("cpu")

# Prevent accidental CUDA usage
os.environ["CUDA_VISIBLE_DEVICES"] = ""

print("=" * 100)
print("PHASE 21N — HYBRID VQC TRAINING / VALIDATION")
print("=" * 100)
print("Device:", DEVICE)
print("✓ CPU-only VQC")
print("✓ CUDA will NOT be used")
print("✓ Locked test will NOT be used")
print()


# ================================================================
# 2. VERIFY INPUTS
# ================================================================

required = [
    "X_dev_np",
    "y_dev",
    "X_val_np",
    "y_val"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required variables: "
        + str(missing)
        + "\nRun the corrected ANOVA12 preparation cell first."
    )


# ================================================================
# 3. FORCE NUMPY / CPU FLOAT32
# ================================================================

X_dev_np = np.asarray(
    X_dev_np,
    dtype=np.float32
)

X_val_np = np.asarray(
    X_val_np,
    dtype=np.float32
)

y_dev = np.asarray(
    y_dev,
    dtype=np.int64
)

y_val = np.asarray(
    y_val,
    dtype=np.int64
)


# ================================================================
# 4. SAFETY CHECKS
# ================================================================

assert X_dev_np.ndim == 2
assert X_val_np.ndim == 2

assert len(X_dev_np) == len(y_dev)
assert len(X_val_np) == len(y_val)

assert X_dev_np.shape[1] == 12
assert X_val_np.shape[1] == 12

assert set(np.unique(y_dev)).issubset({0, 1})
assert set(np.unique(y_val)).issubset({0, 1})

print("=" * 100)
print("INPUT VERIFICATION")
print("=" * 100)

print("Development:", X_dev_np.shape)
print("Validation :", X_val_np.shape)

print("Development classes:", np.bincount(y_dev))
print("Validation classes :", np.bincount(y_val))

print("✓ ANOVA12 input confirmed")
print("✓ Binary target confirmed")
print("✓ No locked-test data used")
print()


# ================================================================
# 5. REPRODUCIBILITY
# ================================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Explicitly avoid CUDA
    if torch.cuda.is_available():
        pass


# ================================================================
# 6. TENSORS
# ================================================================

X_dev = torch.tensor(
    X_dev_np,
    dtype=torch.float32,
    device=DEVICE
)

y_dev_tensor = torch.tensor(
    y_dev,
    dtype=torch.float32,
    device=DEVICE
)

X_val = torch.tensor(
    X_val_np,
    dtype=torch.float32,
    device=DEVICE
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.float32,
    device=DEVICE
)


# ================================================================
# 7. QUANTUM DEVICE
# ================================================================

N_QUBITS = 4

qdev = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

print("=" * 100)
print("QUANTUM DEVICE")
print("=" * 100)
print("Backend: PennyLane default.qubit")
print("Qubits :", N_QUBITS)
print("✓ CPU quantum simulation")
print()


# ================================================================
# 8. QUANTUM CIRCUIT FACTORY
# ================================================================

def build_quantum_circuit(
    depth=2,
    encoding="RYRZ",
    entangle=True
):

    @qml.qnode(qdev, interface="torch")
    def circuit(inputs, weights):

        # --------------------------------------------------------
        # Initial encoding
        # --------------------------------------------------------

        if encoding == "RY":

            for q in range(N_QUBITS):
                qml.RY(
                    inputs[q],
                    wires=q
                )

        elif encoding == "RYRZ":

            for q in range(N_QUBITS):

                qml.RY(
                    inputs[q],
                    wires=q
                )

                qml.RZ(
                    inputs[q],
                    wires=q
                )

        else:
            raise ValueError(
                f"Unknown encoding: {encoding}"
            )

        # --------------------------------------------------------
        # Variational layers
        # --------------------------------------------------------

        for d in range(depth):

            for q in range(N_QUBITS):

                qml.RY(
                    weights[d, q, 0],
                    wires=q
                )

                if encoding == "RYRZ":

                    qml.RZ(
                        weights[d, q, 1],
                        wires=q
                    )

            # ----------------------------------------------------
            # Ring entanglement
            # ----------------------------------------------------

            if entangle:

                for q in range(N_QUBITS - 1):

                    qml.CNOT(
                        wires=[q, q + 1]
                    )

                qml.CNOT(
                    wires=[N_QUBITS - 1, 0]
                )

        # --------------------------------------------------------
        # Measurements
        # --------------------------------------------------------

        return [
            qml.expval(
                qml.PauliZ(q)
            )
            for q in range(N_QUBITS)
        ]

    return circuit


# ================================================================
# 9. HYBRID VQC MODEL
# ================================================================

class HybridVQC(nn.Module):

    def __init__(
        self,
        input_dim=12,
        n_qubits=4,
        depth=2,
        encoding="RYRZ",
        entangle=True,
        dropout=0.10
    ):

        super().__init__()

        self.input_dim = input_dim
        self.n_qubits = n_qubits
        self.depth = depth
        self.encoding = encoding
        self.entangle = entangle

        # --------------------------------------------------------
        # Learnable quantum projection
        # 12 -> 4
        # --------------------------------------------------------

        self.quantum_projection = nn.Sequential(

            nn.Linear(
                input_dim,
                16
            ),

            nn.LayerNorm(16),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                16,
                n_qubits
            ),

            nn.Tanh()
        )

        # --------------------------------------------------------
        # Quantum circuit
        # --------------------------------------------------------

        self.qnode = build_quantum_circuit(
            depth=depth,
            encoding=encoding,
            entangle=entangle
        )

        weight_dim = 2 if encoding == "RYRZ" else 1

        self.q_weights = nn.Parameter(
            0.05 * torch.randn(
                depth,
                n_qubits,
                weight_dim,
                dtype=torch.float32
            )
        )

        # --------------------------------------------------------
        # Classical residual branch
        # --------------------------------------------------------

        self.residual_branch = nn.Sequential(

            nn.Linear(
                input_dim,
                16
            ),

            nn.LayerNorm(16),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                16,
                8
            ),

            nn.GELU()
        )

        # --------------------------------------------------------
        # Quantum branch processing
        # --------------------------------------------------------

        self.quantum_branch = nn.Sequential(

            nn.Linear(
                n_qubits,
                8
            ),

            nn.LayerNorm(8),

            nn.GELU()
        )

        # --------------------------------------------------------
        # Quantum + classical fusion
        # --------------------------------------------------------

        self.fusion = nn.Sequential(

            nn.Linear(
                8 + 8,
                16
            ),

            nn.LayerNorm(16),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                16,
                1
            )
        )


    def forward(self, x):

        # ========================================================
        # Quantum input
        # ========================================================

        q_input = self.quantum_projection(x)

        # ========================================================
        # Quantum circuit
        # ========================================================

        quantum_outputs = []

        for sample in q_input:

            q_out = self.qnode(
                sample,
                self.q_weights
            )

            q_out = torch.stack(
                list(q_out)
            )

            quantum_outputs.append(q_out)

        q_out = torch.stack(
            quantum_outputs
        )

        # ========================================================
        # Branches
        # ========================================================

        q_features = self.quantum_branch(
            q_out
        )

        residual_features = self.residual_branch(
            x
        )

        # ========================================================
        # Fusion
        # ========================================================

        fused = torch.cat(
            [
                q_features,
                residual_features
            ],
            dim=1
        )

        logits = self.fusion(
            fused
        ).squeeze(1)

        return logits


# ================================================================
# 10. METRICS
# ================================================================

def calculate_metrics(
    y_true,
    logits
):

    y_true = np.asarray(
        y_true
    )

    logits = np.asarray(
        logits
    )

    probabilities = 1.0 / (
        1.0 + np.exp(
            -np.clip(
                logits,
                -50,
                50
            )
        )
    )

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    metrics = {}

    metrics["Accuracy"] = accuracy_score(
        y_true,
        predictions
    )

    metrics["Balanced_Accuracy"] = balanced_accuracy_score(
        y_true,
        predictions
    )

    metrics["F1"] = f1_score(
        y_true,
        predictions,
        zero_division=0
    )

    metrics["MCC"] = matthews_corrcoef(
        y_true,
        predictions
    )

    try:
        metrics["ROC_AUC"] = roc_auc_score(
            y_true,
            probabilities
        )
    except Exception:
        metrics["ROC_AUC"] = 0.5

    try:
        metrics["PR_AUC"] = average_precision_score(
            y_true,
            probabilities
        )
    except Exception:
        metrics["PR_AUC"] = 0.5

    return metrics, probabilities, predictions


# ================================================================
# 11. CLASS WEIGHT
# ================================================================

n0 = np.sum(y_dev == 0)
n1 = np.sum(y_dev == 1)

pos_weight_value = n0 / max(
    n1,
    1
)

print("=" * 100)
print("CLASS WEIGHTING")
print("=" * 100)

print("Class 0:", n0)
print("Class 1:", n1)
print("Positive weight:", pos_weight_value)
print()


# ================================================================
# 12. TRAINING FUNCTION
# ================================================================

def train_one_configuration(
    config,
    seed
):

    set_seed(seed)

    model = HybridVQC(
        input_dim=12,
        n_qubits=4,
        depth=config["depth"],
        encoding=config["encoding"],
        entangle=config["entangle"],
        dropout=config["dropout"]
    ).to(DEVICE)

    # ------------------------------------------------------------
    # Weighted BCE
    # ------------------------------------------------------------

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            pos_weight_value,
            dtype=torch.float32,
            device=DEVICE
        )
    )

    # ------------------------------------------------------------
    # AdamW
    # ------------------------------------------------------------

    optimizer = optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    # ------------------------------------------------------------
    # Scheduler
    # ------------------------------------------------------------

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=4
    )

    best_score = -np.inf
    best_epoch = 0
    best_state = None

    patience_counter = 0

    history = []

    for epoch in range(
        1,
        config["max_epochs"] + 1
    ):

        # ========================================================
        # TRAIN
        # ========================================================

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            X_dev
        )

        loss = criterion(
            logits,
            y_dev_tensor
        )

        loss.backward()

        # --------------------------------------------------------
        # Gradient clipping
        # --------------------------------------------------------

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        # ========================================================
        # VALIDATION
        # ========================================================

        model.eval()

        with torch.no_grad():

            val_logits = model(
                X_val
            )

            val_loss = criterion(
                val_logits,
                y_val_tensor
            )

        val_metrics, _, _ = calculate_metrics(
            y_val,
            val_logits.detach().cpu().numpy()
        )

        # --------------------------------------------------------
        # Selection score
        #
        # Balanced accuracy is primary.
        # F1 + ROC-AUC provide secondary stability.
        # --------------------------------------------------------

        selection_score = (
            0.45 * val_metrics["Balanced_Accuracy"]
            +
            0.30 * val_metrics["F1"]
            +
            0.25 * val_metrics["ROC_AUC"]
        )

        history.append({

            "epoch": epoch,

            "train_loss":
                float(loss.detach().cpu().item()),

            "val_loss":
                float(val_loss.detach().cpu().item()),

            "accuracy":
                val_metrics["Accuracy"],

            "balanced_accuracy":
                val_metrics["Balanced_Accuracy"],

            "f1":
                val_metrics["F1"],

            "mcc":
                val_metrics["MCC"],

            "roc_auc":
                val_metrics["ROC_AUC"],

            "pr_auc":
                val_metrics["PR_AUC"],

            "selection_score":
                selection_score
        })

        scheduler.step(
            selection_score
        )

        # --------------------------------------------------------
        # BEST MODEL
        # --------------------------------------------------------

        if selection_score > best_score:

            best_score = selection_score

            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        # --------------------------------------------------------
        # Progress
        # --------------------------------------------------------

        if (
            epoch == 1
            or epoch % 5 == 0
            or selection_score >= best_score
        ):

            print(
                f"Epoch {epoch:03d} | "
                f"Train Loss {loss.item():.5f} | "
                f"Val Loss {val_loss.item():.5f} | "
                f"BA {val_metrics['Balanced_Accuracy']:.4f} | "
                f"F1 {val_metrics['F1']:.4f} | "
                f"ROC-AUC {val_metrics['ROC_AUC']:.4f}"
            )

        # --------------------------------------------------------
        # Early stopping
        # --------------------------------------------------------

        if patience_counter >= config["patience"]:

            print(
                f"Early stopping at epoch {epoch}"
            )

            break

    # ============================================================
    # RESTORE BEST VALIDATION MODEL
    # ============================================================

    if best_state is None:

        raise RuntimeError(
            "No best model state was produced."
        )

    model.load_state_dict(
        best_state
    )

    model.eval()

    for parameter in model.parameters():

        parameter.requires_grad = False

    # ============================================================
    # FINAL VALIDATION METRICS
    # ============================================================

    with torch.no_grad():

        best_val_logits = model(
            X_val
        )

    final_metrics, _, _ = calculate_metrics(
        y_val,
        best_val_logits.cpu().numpy()
    )

    result = {

        "Input": "ANOVA12",

        "Model":
            config["name"],

        "Seed":
            seed,

        "Depth":
            config["depth"],

        "Encoding":
            config["encoding"],

        "LearningRate":
            config["lr"],

        "WeightDecay":
            config["weight_decay"],

        "Entangle":
            config["entangle"],

        "Dropout":
            config["dropout"],

        "Best_Epoch":
            best_epoch,

        "Accuracy":
            final_metrics["Accuracy"],

        "Balanced_Accuracy":
            final_metrics["Balanced_Accuracy"],

        "F1":
            final_metrics["F1"],

        "MCC":
            final_metrics["MCC"],

        "ROC_AUC":
            final_metrics["ROC_AUC"],

        "PR_AUC":
            final_metrics["PR_AUC"],

        "SelectionScore":
            best_score
    }

    return (
        model,
        result,
        pd.DataFrame(history)
    )


# ================================================================
# 13. CONFIGURATIONS
# ================================================================

CONFIGURATIONS = [

    {
        "name":
            "A_RYRZ_D2_LR001",

        "depth":
            2,

        "encoding":
            "RYRZ",

        "lr":
            0.001,

        "weight_decay":
            0.0000,

        "entangle":
            True,

        "dropout":
            0.10,

        "max_epochs":
            60,

        "patience":
            10
    },

    {
        "name":
            "B_RYRZ_D2_LR003",

        "depth":
            2,

        "encoding":
            "RYRZ",

        "lr":
            0.003,

        "weight_decay":
            0.0000,

        "entangle":
            True,

        "dropout":
            0.10,

        "max_epochs":
            60,

        "patience":
            10
    },

    {
        "name":
            "C_RYRZ_D2_LR005",

        "depth":
            2,

        "encoding":
            "RYRZ",

        "lr":
            0.005,

        "weight_decay":
            0.0000,

        "entangle":
            True,

        "dropout":
            0.15,

        "max_epochs":
            60,

        "patience":
            10
    },

    {
        "name":
            "D_RYRZ_D3_LR001",

        "depth":
            3,

        "encoding":
            "RYRZ",

        "lr":
            0.001,

        "weight_decay":
            0.0000,

        "entangle":
            True,

        "dropout":
            0.10,

        "max_epochs":
            60,

        "patience":
            10
    },

    {
        "name":
            "E_RYRZ_D3_LR003",

        "depth":
            3,

        "encoding":
            "RYRZ",

        "lr":
            0.003,

        "weight_decay":
            0.0000,

        "entangle":
            True,

        "dropout":
            0.15,

        "max_epochs":
            60,

        "patience":
            10
    },

    {
        "name":
            "F_RY_D2_LR003",

        "depth":
            2,

        "encoding":
            "RY",

        "lr":
            0.003,

        "weight_decay":
            0.0000,

        "entangle":
            True,

        "dropout":
            0.10,

        "max_epochs":
            60,

        "patience":
            10
    },

    {
        "name":
            "G_RYRZ_D2_LR001_NOENT",

        "depth":
            2,

        "encoding":
            "RYRZ",

        "lr":
            0.001,

        "weight_decay":
            0.0001,

        "entangle":
            False,

        "dropout":
            0.10,

        "max_epochs":
            60,

        "patience":
            10
    }
]


# ================================================================
# 14. SEEDS
# ================================================================

SEEDS = [
    123,
    2024
]


# ================================================================
# 15. OUTPUT DIRECTORY
# ================================================================

OUTPUT_DIR = (
    "/kaggle/working/phase21N_outputs"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ================================================================
# 16. TRAIN ALL CONFIGURATIONS
# ================================================================

results = []

best_model = None
best_result = None
best_history = None

global_best_score = -np.inf


print("=" * 100)
print("STARTING HYBRID VQC SEARCH")
print("=" * 100)

print(
    f"Configurations: {len(CONFIGURATIONS)}"
)

print(
    f"Seeds: {SEEDS}"
)

print(
    f"Total runs: "
    f"{len(CONFIGURATIONS) * len(SEEDS)}"
)

print()


for config in CONFIGURATIONS:

    print()
    print("-" * 100)

    print(
        "CONFIGURATION:",
        config["name"]
    )

    print("-" * 100)

    for seed in SEEDS:

        print()
        print(
            f"Seed {seed}"
        )

        try:

            model, result, history = (
                train_one_configuration(
                    config,
                    seed
                )
            )

            results.append(
                result
            )

            print()
            print(
                f"✓ Completed {config['name']} "
                f"Seed {seed}"
            )

            print(
                f"Validation BA: "
                f"{result['Balanced_Accuracy']:.4f}"
            )

            print(
                f"Validation F1: "
                f"{result['F1']:.4f}"
            )

            print(
                f"Validation ROC-AUC: "
                f"{result['ROC_AUC']:.4f}"
            )

            print(
                f"Selection score: "
                f"{result['SelectionScore']:.4f}"
            )

            # ----------------------------------------------------
            # GLOBAL BEST
            # ----------------------------------------------------

            if (
                result["SelectionScore"]
                > global_best_score
            ):

                global_best_score = (
                    result["SelectionScore"]
                )

                best_model = copy.deepcopy(
                    model
                )

                best_result = copy.deepcopy(
                    result
                )

                best_history = (
                    history.copy()
                )

                print()
                print(
                    "★ NEW BEST VQC ★"
                )

        except Exception as e:

            print()
            print(
                f"FAILED "
                f"{config['name']} "
                f"Seed {seed}"
            )

            print(
                repr(e)
            )

            continue


# ================================================================
# 17. CHECK RESULTS
# ================================================================

if len(results) == 0:

    raise RuntimeError(
        "All VQC configurations failed."
    )


# ================================================================
# 18. COMPARISON TABLE
# ================================================================

comparison_df = pd.DataFrame(
    results
)

comparison_df = comparison_df.sort_values(
    by="SelectionScore",
    ascending=False
).reset_index(
    drop=True
)


print()
print()
print("=" * 100)
print("HYBRID VQC INTERNAL MODEL COMPARISON")
print("=" * 100)

print(
    comparison_df[
        [
            "Input",
            "Model",
            "Seed",
            "Depth",
            "Encoding",
            "LearningRate",
            "WeightDecay",
            "Entangle",
            "Best_Epoch",
            "Accuracy",
            "Balanced_Accuracy",
            "F1",
            "MCC",
            "ROC_AUC",
            "PR_AUC",
            "SelectionScore"
        ]
    ].to_string(
        index=False
    )
)


# ================================================================
# 19. BEST MODEL
# ================================================================

print()
print()
print("=" * 100)
print("BEST INTERNAL HYBRID VQC")
print("=" * 100)

print(
    "Input:",
    best_result["Input"]
)

print(
    "Model:",
    best_result["Model"]
)

print(
    "Seed:",
    best_result["Seed"]
)

print(
    "Depth:",
    best_result["Depth"]
)

print(
    "Encoding:",
    best_result["Encoding"]
)

print(
    "Learning rate:",
    best_result["LearningRate"]
)

print(
    "Weight decay:",
    best_result["WeightDecay"]
)

print(
    "Entanglement:",
    best_result["Entangle"]
)

print(
    "Best epoch:",
    best_result["Best_Epoch"]
)

print()

print(
    f"Accuracy: "
    f"{best_result['Accuracy']:.4f}"
)

print(
    f"Balanced Accuracy: "
    f"{best_result['Balanced_Accuracy']:.4f}"
)

print(
    f"F1: "
    f"{best_result['F1']:.4f}"
)

print(
    f"MCC: "
    f"{best_result['MCC']:.4f}"
)

print(
    f"ROC-AUC: "
    f"{best_result['ROC_AUC']:.4f}"
)

print(
    f"PR-AUC: "
    f"{best_result['PR_AUC']:.4f}"
)

print(
    f"Selection Score: "
    f"{best_result['SelectionScore']:.4f}"
)


# ================================================================
# 20. FINAL FREEZE
# ================================================================

best_model.eval()

for parameter in best_model.parameters():

    parameter.requires_grad = False


# ================================================================
# 21. SAVE CHECKPOINT
# ================================================================

MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "phase21N_best_hybrid_vqc.pt"
)

COMPARISON_PATH = os.path.join(
    OUTPUT_DIR,
    "phase21N_vqc_internal_model_comparison.csv"
)

HISTORY_PATH = os.path.join(
    OUTPUT_DIR,
    "phase21N_best_training_history.csv"
)


checkpoint = {

    "model_state_dict":
        best_model.state_dict(),

    "best_info":
        best_result,

    "input_dim":
        12,

    "n_qubits":
        4,

    "depth":
        best_result["Depth"],

    "encoding":
        best_result["Encoding"],

    "entangle":
        best_result["Entangle"],

    "dropout":
        best_result["Dropout"],

    "selected_features":
        [
            "age",
            "sex",
            "motor_UPDRS",
            "Jitter(%)",
            "Jitter(Abs)",
            "Jitter:RAP",
            "Jitter:PPQ5",
            "Jitter:DDP",
            "HNR",
            "RPDE",
            "DFA",
            "PPE"
        ]
}


torch.save(
    checkpoint,
    MODEL_PATH
)

comparison_df.to_csv(
    COMPARISON_PATH,
    index=False
)

best_history.to_csv(
    HISTORY_PATH,
    index=False
)


# ================================================================
# 22. FINAL STATUS
# ================================================================

print()
print()
print("=" * 100)
print("HYBRID VQC FROZEN")
print("=" * 100)

print("✓ Best model selected using development + validation")
print("✓ ANOVA12 selected using development only")
print("✓ Imputation fitted using development only")
print("✓ Scaling fitted using development only")
print("✓ Multiple VQC architectures compared")
print("✓ Multiple random seeds compared")
print("✓ Learnable quantum projection used")
print("✓ Classical residual branch used")
print("✓ Quantum + classical fusion used")
print("✓ Class-weighted loss used")
print("✓ AdamW optimization used")
print("✓ Gradient clipping used")
print("✓ Early stopping used")
print("✓ Locked test NOT used")
print("✓ Locked test NOT evaluated")
print("✓ Best model frozen")
print("✓ No optimizer remains")
print()

print("Saved:")
print("✓", MODEL_PATH)
print("✓", COMPARISON_PATH)
print("✓", HISTORY_PATH)

print()
print("=" * 100)
print("PHASE 21N TRAINING / VALIDATION COMPLETED")
print("=" * 100)

print()
print("IMPORTANT:")
print(
    "DO NOT evaluate X_test_np / y_test in this cell."
)
print(
    "The next cell should load the frozen checkpoint "
    "and evaluate the locked test exactly once."
)

PHASE 21N — HYBRID VQC TRAINING / VALIDATION
Device: cpu
✓ CPU-only VQC
✓ CUDA will NOT be used
✓ Locked test will NOT be used

INPUT VERIFICATION
Development: (3602, 12)
Validation : (983, 12)
Development classes: [2348 1254]
Validation classes : [589 394]
✓ ANOVA12 input confirmed
✓ Binary target confirmed
✓ No locked-test data used

QUANTUM DEVICE
Backend: PennyLane default.qubit
Qubits : 4
✓ CPU quantum simulation

CLASS WEIGHTING
Class 0: 2348
Class 1: 1254
Positive weight: 1.872408293460925

STARTING HYBRID VQC SEARCH
Configurations: 7
Seeds: [123, 2024]
Total runs: 14


----------------------------------------------------------------------------------------------------
CONFIGURATION: A_RYRZ_D2_LR001
----------------------------------------------------------------------------------------------------

Seed 123

FAILED A_RYRZ_D2_LR001 Seed 123
RuntimeError('mat1 and mat2 must have the same dtype, but got Double and Float')

Seed 2024

FAILED A_RYRZ_D2_LR001 Seed 2024
RuntimeError('

RuntimeError: All VQC configurations failed.

In [29]:
# =============================================================================
# PHASE 21 — VQC DTYPE-SAFE TRAINING
# Fixes: mat1 and mat2 must have the same dtype, but got Double and Float
# =============================================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score
)

# =============================================================================
# DEVICE — CPU ONLY
# =============================================================================

DEVICE = torch.device("cpu")

print("=" * 100)
print("PHASE 21 — DTYPE-SAFE CPU-ONLY VQC")
print("=" * 100)
print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())
print("✓ VQC will use CPU only")
print("✓ CUDA will NOT be used")
print("✓ float32 enforced throughout")
print("=" * 100)


# =============================================================================
# REPRODUCIBILITY
# =============================================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# =============================================================================
# IMPORTANT:
# This cell expects these arrays from your previous preparation cell:
#
# X_dev_np, y_dev_np
# X_val_np, y_val_np
# X_test_np, y_test_np
#
# OR:
# X_dev, y_dev
# X_val, y_val
# X_test, y_test
# =============================================================================

def get_array(name1, name2=None):
    if name1 in globals():
        return globals()[name1]

    if name2 is not None and name2 in globals():
        return globals()[name2]

    return None


X_dev_np = get_array("X_dev_np", "X_dev")
y_dev_np = get_array("y_dev_np", "y_dev")

X_val_np = get_array("X_val_np", "X_val")
y_val_np = get_array("y_val_np", "y_val")

X_test_np = get_array("X_test_np", "X_test")
y_test_np = get_array("y_test_np", "y_test")


missing = []

for name, value in [
    ("X_dev", X_dev_np),
    ("y_dev", y_dev_np),
    ("X_val", X_val_np),
    ("y_val", y_val_np),
    ("X_test", X_test_np),
    ("y_test", y_test_np),
]:
    if value is None:
        missing.append(name)

if missing:
    raise RuntimeError(
        "Missing required variables: " + str(missing) +
        "\nRun your leakage-safe ANOVA12 preparation cell first."
    )


# =============================================================================
# FORCE NUMPY TYPES
# =============================================================================

X_dev_np = np.asarray(X_dev_np, dtype=np.float32)
X_val_np = np.asarray(X_val_np, dtype=np.float32)
X_test_np = np.asarray(X_test_np, dtype=np.float32)

y_dev_np = np.asarray(y_dev_np, dtype=np.int64)
y_val_np = np.asarray(y_val_np, dtype=np.int64)
y_test_np = np.asarray(y_test_np, dtype=np.int64)


print("\n" + "=" * 100)
print("INPUT DTYPE CHECK")
print("=" * 100)

print("X_dev:", X_dev_np.shape, X_dev_np.dtype)
print("X_val:", X_val_np.shape, X_val_np.dtype)
print("X_test:", X_test_np.shape, X_test_np.dtype)

print("y_dev:", y_dev_np.shape, y_dev_np.dtype)
print("y_val:", y_val_np.shape, y_val_np.dtype)
print("y_test:", y_test_np.shape, y_test_np.dtype)

assert X_dev_np.dtype == np.float32
assert X_val_np.dtype == np.float32
assert X_test_np.dtype == np.float32

print("✓ All feature matrices are float32")
print("✓ All targets are int64")


# =============================================================================
# TENSOR CONVERSION
# =============================================================================

X_dev = torch.tensor(
    X_dev_np,
    dtype=torch.float32,
    device=DEVICE
)

X_val = torch.tensor(
    X_val_np,
    dtype=torch.float32,
    device=DEVICE
)

X_test = torch.tensor(
    X_test_np,
    dtype=torch.float32,
    device=DEVICE
)

y_dev = torch.tensor(
    y_dev_np,
    dtype=torch.long,
    device=DEVICE
)

y_val = torch.tensor(
    y_val_np,
    dtype=torch.long,
    device=DEVICE
)

y_test = torch.tensor(
    y_test_np,
    dtype=torch.long,
    device=DEVICE
)


print("\n" + "=" * 100)
print("FINAL TENSOR CHECK")
print("=" * 100)

print("X_dev:", X_dev.shape, X_dev.dtype)
print("X_val:", X_val.shape, X_val.dtype)
print("X_test:", X_test.shape, X_test.dtype)

assert X_dev.dtype == torch.float32
assert X_val.dtype == torch.float32
assert X_test.dtype == torch.float32

print("✓ Tensor dtype problem FIXED")


# =============================================================================
# VQC CONFIGURATIONS
# =============================================================================

CONFIGS = {

    # Your problematic configuration
    "G_RYRZ_D2_LR001_NOENT": {
        "depth": 2,
        "encoding": "RYRZ",
        "lr": 0.001,
        "entangle": False,
    },

    # Other useful configurations
    "A_RYRZ_D2_LR001": {
        "depth": 2,
        "encoding": "RYRZ",
        "lr": 0.001,
        "entangle": True,
    },

    "B_RYRZ_D2_LR003": {
        "depth": 2,
        "encoding": "RYRZ",
        "lr": 0.003,
        "entangle": True,
    },

    "C_RYRZ_D2_LR005": {
        "depth": 2,
        "encoding": "RYRZ",
        "lr": 0.005,
        "entangle": True,
    },

    "D_RYRZ_D3_LR001": {
        "depth": 3,
        "encoding": "RYRZ",
        "lr": 0.001,
        "entangle": True,
    },

    "E_RYRZ_D3_LR003": {
        "depth": 3,
        "encoding": "RYRZ",
        "lr": 0.003,
        "entangle": True,
    },

    "F_RY_D2_LR003": {
        "depth": 2,
        "encoding": "RY",
        "lr": 0.003,
        "entangle": True,
    },
}


# =============================================================================
# QUANTUM DEVICE
# =============================================================================

N_QUBITS = 4


dev = qml.device(
    "default.qubit",
    wires=N_QUBITS
)


# =============================================================================
# QUANTUM CIRCUIT
# =============================================================================

def make_qnode(depth=2, encoding="RYRZ", entangle=True):

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):

        # ---------------------------------------------------------------------
        # INPUT ENCODING
        # ---------------------------------------------------------------------

        for q in range(N_QUBITS):

            x = inputs[q]

            # VERY IMPORTANT:
            # Keep quantum input as torch.float32.
            x = x.to(torch.float32)

            if encoding == "RY":
                qml.RY(x, wires=q)

            elif encoding == "RYRZ":
                qml.RY(x, wires=q)
                qml.RZ(x, wires=q)

            else:
                qml.RY(x, wires=q)

        # ---------------------------------------------------------------------
        # VARIATIONAL LAYERS
        # ---------------------------------------------------------------------

        for d in range(depth):

            for q in range(N_QUBITS):

                qml.RY(
                    weights[d, q, 0],
                    wires=q
                )

                if encoding == "RYRZ":

                    qml.RZ(
                        weights[d, q, 1],
                        wires=q
                    )

            # -----------------------------------------------------------------
            # ENTANGLEMENT
            # -----------------------------------------------------------------

            if entangle:

                for q in range(N_QUBITS - 1):

                    qml.CNOT(
                        wires=[q, q + 1]
                    )

        return [
            qml.expval(qml.PauliZ(q))
            for q in range(N_QUBITS)
        ]

    return circuit


# =============================================================================
# HYBRID VQC MODEL
# =============================================================================

class VQC(nn.Module):

    def __init__(
        self,
        input_dim,
        depth=2,
        encoding="RYRZ",
        entangle=True
    ):

        super().__init__()

        self.input_dim = input_dim
        self.depth = depth
        self.encoding = encoding
        self.entangle = entangle

        # ---------------------------------------------------------------------
        # Classical projection
        # 12 ANOVA FEATURES -> 4 QUANTUM FEATURES
        # ---------------------------------------------------------------------

        self.projection = nn.Sequential(

            nn.Linear(
                input_dim,
                8,
                dtype=torch.float32
            ),

            nn.LayerNorm(
                8
            ),

            nn.GELU(),

            nn.Linear(
                8,
                N_QUBITS,
                dtype=torch.float32
            ),

        )

        # ---------------------------------------------------------------------
        # Quantum weights
        # ---------------------------------------------------------------------

        n_params_per_qubit = 2 if encoding == "RYRZ" else 1

        self.q_weights = nn.Parameter(
            torch.randn(
                depth,
                N_QUBITS,
                n_params_per_qubit,
                dtype=torch.float32
            ) * 0.05
        )

        # ---------------------------------------------------------------------
        # Classical output head
        # ---------------------------------------------------------------------

        self.classifier = nn.Sequential(

            nn.Linear(
                N_QUBITS,
                8,
                dtype=torch.float32
            ),

            nn.GELU(),

            nn.Dropout(0.10),

            nn.Linear(
                8,
                1,
                dtype=torch.float32
            )
        )

        self.qnode = make_qnode(
            depth=depth,
            encoding=encoding,
            entangle=entangle
        )


    def forward(self, x):

        # ABSOLUTELY IMPORTANT
        x = x.to(
            device=DEVICE,
            dtype=torch.float32
        )

        # Classical projection
        z = self.projection(x)

        # Stable quantum angle
        z = torch.tanh(z) * np.pi

        # Quantum circuit
        q_out = self.qnode(
            z,
            self.q_weights
        )

        # PennyLane can return tuple/list
        if isinstance(q_out, (list, tuple)):

            q_out = torch.stack(
                list(q_out)
            )

        q_out = q_out.to(
            dtype=torch.float32
        )

        # For batched inputs PennyLane may produce
        # [batch, qubits]
        if q_out.ndim == 1:

            q_out = q_out.unsqueeze(0)

        return self.classifier(q_out).squeeze(-1)


# =============================================================================
# METRIC FUNCTION
# =============================================================================

def evaluate_model(model, X, y):

    model.eval()

    with torch.no_grad():

        logits = model(X)

        probs = torch.sigmoid(
            logits
        ).detach().cpu().numpy()

        preds = (
            probs >= 0.5
        ).astype(int)

    y_np = y.detach().cpu().numpy()

    acc = accuracy_score(
        y_np,
        preds
    )

    bal = balanced_accuracy_score(
        y_np,
        preds
    )

    f1 = f1_score(
        y_np,
        preds,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_np,
        preds
    )

    try:
        roc = roc_auc_score(
            y_np,
            probs
        )
    except:
        roc = np.nan

    try:
        pr = average_precision_score(
            y_np,
            probs
        )
    except:
        pr = np.nan

    return {
        "Accuracy": acc,
        "Balanced_Accuracy": bal,
        "F1": f1,
        "MCC": mcc,
        "ROC_AUC": roc,
        "PR_AUC": pr
    }


# =============================================================================
# TRAINING FUNCTION
# =============================================================================

def train_vqc(
    config_name,
    config,
    seed,
    epochs=20,
    patience=6
):

    print("\n" + "-" * 100)
    print("CONFIGURATION:", config_name)
    print("Seed:", seed)
    print("-" * 100)

    set_seed(seed)

    model = VQC(
        input_dim=X_dev.shape[1],
        depth=config["depth"],
        encoding=config["encoding"],
        entangle=config["entangle"]
    ).to(DEVICE)

    # FINAL SAFETY CHECK
    model = model.float()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=1e-4
    )

    criterion = nn.BCEWithLogitsLoss()

    best_state = None
    best_score = -np.inf
    best_epoch = 0

    wait = 0

    for epoch in range(1, epochs + 1):

        model.train()

        optimizer.zero_grad()

        logits = model(X_dev)

        loss = criterion(
            logits,
            y_dev.float()
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        # ---------------------------------------------------------------
        # Validation
        # ---------------------------------------------------------------

        val_metrics = evaluate_model(
            model,
            X_val,
            y_val
        )

        # Main selection metric
        score = (
            0.50 * val_metrics["Balanced_Accuracy"]
            +
            0.30 * val_metrics["F1"]
            +
            0.20 * val_metrics["ROC_AUC"]
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Loss {loss.item():.5f} | "
            f"Val BalAcc {val_metrics['Balanced_Accuracy']:.4f} | "
            f"Val F1 {val_metrics['F1']:.4f} | "
            f"Val ROC {val_metrics['ROC_AUC']:.4f}"
        )

        if score > best_score:

            best_score = score
            best_epoch = epoch
            wait = 0

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        else:

            wait += 1

            if wait >= patience:

                print(
                    f"Early stopping at epoch {epoch}"
                )

                break

    # -------------------------------------------------------------------
    # RESTORE BEST VALIDATION MODEL
    # -------------------------------------------------------------------

    model.load_state_dict(
        best_state
    )

    model.eval()

    val_metrics = evaluate_model(
        model,
        X_val,
        y_val
    )

    result = {
        "Input": "ANOVA12",
        "Model": config_name,
        "Seed": seed,
        "Best_Epoch": best_epoch,
        **val_metrics,
        "SelectionScore": best_score
    }

    print("\nBEST VALIDATION RESULT")

    for k, v in result.items():

        if isinstance(v, (float, np.floating)):

            print(
                f"{k:22s}: {v:.6f}"
            )

        else:

            print(
                f"{k:22s}: {v}"
            )

    return model, result


# =============================================================================
# RUN CONFIGURATIONS
# =============================================================================

results = []

best_model = None
best_result = None

for config_name, config in CONFIGS.items():

    for seed in [123, 2024]:

        try:

            model, result = train_vqc(
                config_name,
                config,
                seed,
                epochs=20,
                patience=6
            )

            results.append(result)

            if (
                best_result is None
                or
                result["SelectionScore"]
                >
                best_result["SelectionScore"]
            ):

                best_model = model
                best_result = result

        except Exception as e:

            print(
                f"\nFAILED {config_name} Seed {seed}"
            )

            print(
                repr(e)
            )

            # Continue instead of destroying
            # the entire experiment.
            continue


# =============================================================================
# RESULTS
# =============================================================================

if len(results) == 0:

    raise RuntimeError(
        "All VQC configurations failed."
    )


results_df = (
    pd.DataFrame(results)
    .sort_values(
        "SelectionScore",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 100)
print("VQC INTERNAL MODEL COMPARISON")
print("=" * 100)

print(
    results_df.to_string(
        index=False
    )
)


# =============================================================================
# BEST MODEL
# =============================================================================

print("\n" + "=" * 100)
print("BEST VQC")
print("=" * 100)

print(
    results_df.iloc[0].to_string()
)


# =============================================================================
# DO NOT USE LOCKED TEST FOR MODEL SELECTION
# =============================================================================

print("\n" + "=" * 100)
print("LOCKED TEST")
print("=" * 100)

print(
    "✓ Locked test has NOT been used for tuning"
)

print(
    "✓ Locked test has NOT been used for model selection"
)

print(
    "✓ Best model selected using validation only"
)


# =============================================================================
# SAVE INTERNAL RESULTS
# =============================================================================

OUTPUT_DIR = "/kaggle/working/phase21_vqc_outputs"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

results_path = os.path.join(
    OUTPUT_DIR,
    "phase21_vqc_internal_comparison.csv"
)

results_df.to_csv(
    results_path,
    index=False
)


# =============================================================================
# SAVE BEST MODEL
# =============================================================================

best_config = CONFIGS[
    best_result["Model"]
]

checkpoint = {

    "model_state_dict":
        {
            k: v.detach().cpu()
            for k, v in best_model.state_dict().items()
        },

    "best_info":
        best_result,

    "input_dim":
        int(X_dev.shape[1]),

    "n_qubits":
        N_QUBITS,

    "depth":
        best_config["depth"],

    "encoding":
        best_config["encoding"],

    "entangle":
        best_config["entangle"],
}


model_path = os.path.join(
    OUTPUT_DIR,
    "best_vqc_model.pt"
)

torch.save(
    checkpoint,
    model_path
)


# =============================================================================
# FINAL
# =============================================================================

print("\n" + "=" * 100)
print("PHASE 21 VQC TRAINING COMPLETED")
print("=" * 100)

print(
    "Best model:",
    best_result["Model"]
)

print(
    "Seed:",
    best_result["Seed"]
)

print(
    "Best epoch:",
    best_result["Best_Epoch"]
)

print(
    "Validation Balanced Accuracy:",
    f"{best_result['Balanced_Accuracy']:.4f}"
)

print(
    "Validation F1:",
    f"{best_result['F1']:.4f}"
)

print(
    "Validation ROC-AUC:",
    f"{best_result['ROC_AUC']:.4f}"
)

print("\nSaved:")
print("✓", results_path)
print("✓", model_path)

print("\n" + "=" * 100)
print("IMPORTANT")
print("=" * 100)

print(
    "Do NOT tune using the locked test."
)

print(
    "Once the VQC is finalized, evaluate the locked test separately."
)

print("=" * 100)

PHASE 21 — DTYPE-SAFE CPU-ONLY VQC
Device: cpu
CUDA available: False
✓ VQC will use CPU only
✓ CUDA will NOT be used
✓ float32 enforced throughout

INPUT DTYPE CHECK
X_dev: (3602, 12) float32
X_val: (983, 12) float32
X_test: (1290, 12) float32
y_dev: (3602,) int64
y_val: (983,) int64
y_test: (1290,) int64
✓ All feature matrices are float32
✓ All targets are int64

FINAL TENSOR CHECK
X_dev: torch.Size([3602, 12]) torch.float32
X_val: torch.Size([983, 12]) torch.float32
X_test: torch.Size([1290, 12]) torch.float32
✓ Tensor dtype problem FIXED

----------------------------------------------------------------------------------------------------
CONFIGURATION: G_RYRZ_D2_LR001_NOENT
Seed: 123
----------------------------------------------------------------------------------------------------

FAILED G_RYRZ_D2_LR001_NOENT Seed 123
ValueError('Target size (torch.Size([3602])) must be the same as input size (torch.Size([4]))')

-------------------------------------------------------------------

RuntimeError: All VQC configurations failed.

In [30]:
# ================================================================
# PHASE 21 — CORRECTED CPU-ONLY VQC TRAINING
# ================================================================

import os
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------
# CPU ONLY
# ------------------------------------------------

DEVICE = torch.device("cpu")

print("=" * 100)
print("PHASE 21 — CORRECTED VQC TRAINING")
print("=" * 100)
print("Device:", DEVICE)
print("✓ CPU-only")
print("✓ CUDA will NOT be used")


# ================================================================
# 1. REPRODUCIBILITY
# ================================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ================================================================
# 2. CHECK REQUIRED DATA
# ================================================================

required = [
    "X_dev",
    "X_val",
    "y_dev",
    "y_val"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Missing variables: " + str(missing) +
        "\nExpected X_dev, X_val, y_dev and y_val."
    )


# ================================================================
# 3. FORCE CORRECT NUMPY TYPES
# ================================================================

X_dev_np = np.asarray(X_dev, dtype=np.float32)
X_val_np = np.asarray(X_val, dtype=np.float32)

y_dev_np = np.asarray(y_dev, dtype=np.int64).reshape(-1)
y_val_np = np.asarray(y_val, dtype=np.int64).reshape(-1)


print()
print("=" * 100)
print("INPUT CHECK")
print("=" * 100)

print("X_dev:", X_dev_np.shape, X_dev_np.dtype)
print("y_dev:", y_dev_np.shape, y_dev_np.dtype)

print("X_val:", X_val_np.shape, X_val_np.dtype)
print("y_val:", y_val_np.shape, y_val_np.dtype)


# ================================================================
# 4. SANITY CHECK
# ================================================================

if X_dev_np.ndim != 2:
    raise RuntimeError(
        f"X_dev must be 2D. Got {X_dev_np.shape}"
    )

if X_val_np.ndim != 2:
    raise RuntimeError(
        f"X_val must be 2D. Got {X_val_np.shape}"
    )

if X_dev_np.shape[1] != 12:
    raise RuntimeError(
        f"Expected ANOVA12 = 12 features. "
        f"Got {X_dev_np.shape[1]}"
    )

if X_val_np.shape[1] != 12:
    raise RuntimeError(
        f"Expected ANOVA12 = 12 features. "
        f"Got {X_val_np.shape[1]}"
    )

if len(X_dev_np) != len(y_dev_np):
    raise RuntimeError(
        f"Development mismatch: "
        f"X={len(X_dev_np)}, y={len(y_dev_np)}"
    )

if len(X_val_np) != len(y_val_np):
    raise RuntimeError(
        f"Validation mismatch: "
        f"X={len(X_val_np)}, y={len(y_val_np)}"
    )

print("✓ Development shapes consistent")
print("✓ Validation shapes consistent")
print("✓ ANOVA12 = 12 features")


# ================================================================
# 5. TORCH DATA
# ================================================================

X_dev_t = torch.tensor(
    X_dev_np,
    dtype=torch.float32,
    device=DEVICE
)

y_dev_t = torch.tensor(
    y_dev_np,
    dtype=torch.float32,
    device=DEVICE
)

X_val_t = torch.tensor(
    X_val_np,
    dtype=torch.float32,
    device=DEVICE
)

y_val_t = torch.tensor(
    y_val_np,
    dtype=torch.float32,
    device=DEVICE
)


print()
print("=" * 100)
print("TORCH TENSOR CHECK")
print("=" * 100)

print("X_dev:", X_dev_t.shape, X_dev_t.dtype)
print("y_dev:", y_dev_t.shape, y_dev_t.dtype)

print("X_val:", X_val_t.shape, X_val_t.dtype)
print("y_val:", y_val_t.shape, y_val_t.dtype)


# ================================================================
# 6. QUANTUM SETTINGS
# ================================================================

N_QUBITS = 4
N_FEATURES = 12

BATCH_SIZE = 32

EPOCHS = 60

PATIENCE = 10

SEEDS = [123, 2024]

LEARNING_RATES = [
    0.001,
    0.003,
    0.005
]

DEPTHS = [
    2,
    3
]


# ================================================================
# 7. FEATURE -> QUANTUM ANGLE PROJECTION
# ================================================================

class QuantumProjection(nn.Module):

    def __init__(self, input_dim, n_qubits):

        super().__init__()

        self.projection = nn.Sequential(

            nn.Linear(input_dim, 32),

            nn.LayerNorm(32),

            nn.Tanh(),

            nn.Linear(32, n_qubits),

            nn.Tanh()
        )

    def forward(self, x):

        # IMPORTANT:
        # x = [batch, 12]
        # output = [batch, 4]

        angles = self.projection(x)

        # keep angles numerically stable

        return angles * np.pi


# ================================================================
# 8. CORRECT QUANTUM CIRCUIT
# ================================================================

def make_qnode(
    n_qubits=4,
    depth=2,
    encoding="RYRZ",
    entangle=True
):

    dev = qml.device(
        "default.qubit",
        wires=n_qubits
    )

    @qml.qnode(dev, interface="torch")
    def circuit(inputs, weights):

        # inputs shape = [4]

        if encoding == "RYRZ":

            for q in range(n_qubits):

                qml.RY(
                    inputs[q],
                    wires=q
                )

                qml.RZ(
                    inputs[q],
                    wires=q
                )

        elif encoding == "RY":

            for q in range(n_qubits):

                qml.RY(
                    inputs[q],
                    wires=q
                )

        else:

            raise ValueError(
                f"Unknown encoding: {encoding}"
            )


        for d in range(depth):

            for q in range(n_qubits):

                qml.RY(
                    weights[d, q, 0],
                    wires=q
                )

                qml.RZ(
                    weights[d, q, 1],
                    wires=q
                )

            if entangle:

                for q in range(n_qubits - 1):

                    qml.CNOT(
                        wires=[q, q + 1]
                    )

        # ONE scalar prediction
        #
        # This is critical.
        #
        # Do NOT return all 4 qubits.

        return qml.expval(
            qml.PauliZ(0)
        )

    return circuit


# ================================================================
# 9. VQC MODEL
# ================================================================

class VQC(nn.Module):

    def __init__(
        self,
        input_dim=12,
        n_qubits=4,
        depth=2,
        encoding="RYRZ",
        entangle=True
    ):

        super().__init__()

        self.input_dim = input_dim

        self.n_qubits = n_qubits

        self.depth = depth

        self.encoding = encoding

        self.entangle = entangle

        # Classical projection
        self.projection = QuantumProjection(
            input_dim,
            n_qubits
        )

        # Quantum circuit
        self.qnode = make_qnode(
            n_qubits=n_qubits,
            depth=depth,
            encoding=encoding,
            entangle=entangle
        )

        # Trainable quantum parameters
        self.q_weights = nn.Parameter(
            0.05 * torch.randn(
                depth,
                n_qubits,
                2,
                dtype=torch.float32
            )
        )

        # Classical output calibration
        self.output_scale = nn.Parameter(
            torch.tensor(
                1.0,
                dtype=torch.float32
            )
        )

        self.output_bias = nn.Parameter(
            torch.tensor(
                0.0,
                dtype=torch.float32
            )
        )


    def forward(self, x):

        # ========================================================
        # EXPECTED:
        #
        # x = [batch, 12]
        #
        # projection = [batch, 4]
        # ========================================================

        x = x.float()

        angles = self.projection(x)

        batch_outputs = []

        # --------------------------------------------------------
        # CRITICAL FIX
        #
        # Quantum circuit processes ONE SAMPLE at a time.
        #
        # Each circuit call returns ONE scalar.
        # --------------------------------------------------------

        for i in range(angles.shape[0]):

            q_out = self.qnode(
                angles[i],
                self.q_weights
            )

            # q_out = scalar tensor

            batch_outputs.append(q_out)

        # --------------------------------------------------------
        # Convert:
        #
        # [scalar, scalar, ...]
        #
        # into:
        #
        # [batch]
        # --------------------------------------------------------

        q_out = torch.stack(
            batch_outputs
        ).float()

        # Classical calibration

        logits = (
            self.output_scale * q_out
            + self.output_bias
        )

        # IMPORTANT:
        # output shape = [batch]

        return logits


# ================================================================
# 10. TEST MODEL SHAPE BEFORE TRAINING
# ================================================================

print()
print("=" * 100)
print("MODEL SHAPE TEST")
print("=" * 100)

test_model = VQC(
    input_dim=12,
    n_qubits=4,
    depth=2,
    encoding="RYRZ",
    entangle=True
).to(DEVICE)

test_batch = X_dev_t[:8]

with torch.no_grad():

    test_output = test_model(
        test_batch
    )

print("Input :", test_batch.shape)
print("Output:", test_output.shape)

if test_output.shape != torch.Size([8]):

    raise RuntimeError(
        f"VQC output shape is WRONG: "
        f"{test_output.shape}\n"
        f"Expected: torch.Size([8])"
    )

print("✓ VQC output shape is correct")
print("✓ One prediction per sample")


# ================================================================
# 11. CLASS WEIGHTS
# ================================================================

n0 = np.sum(y_dev_np == 0)

n1 = np.sum(y_dev_np == 1)

pos_weight = n0 / max(n1, 1)

print()
print("=" * 100)
print("CLASS BALANCE")
print("=" * 100)

print("Class 0:", n0)
print("Class 1:", n1)
print("Positive weight:", pos_weight)


# ================================================================
# 12. METRICS
# ================================================================

def calculate_metrics(
    y_true,
    logits
):

    y_true = np.asarray(
        y_true
    ).astype(int)

    logits = np.asarray(
        logits
    )

    probs = 1.0 / (
        1.0 + np.exp(
            -np.clip(logits, -30, 30)
        )
    )

    preds = (
        probs >= 0.5
    ).astype(int)

    metrics = {}

    metrics["Accuracy"] = accuracy_score(
        y_true,
        preds
    )

    metrics["Balanced_Accuracy"] = (
        balanced_accuracy_score(
            y_true,
            preds
        )
    )

    metrics["F1"] = f1_score(
        y_true,
        preds,
        zero_division=0
    )

    metrics["MCC"] = matthews_corrcoef(
        y_true,
        preds
    )

    try:

        metrics["ROC_AUC"] = roc_auc_score(
            y_true,
            probs
        )

    except:

        metrics["ROC_AUC"] = np.nan

    try:

        metrics["PR_AUC"] = average_precision_score(
            y_true,
            probs
        )

    except:

        metrics["PR_AUC"] = np.nan

    return metrics, probs, preds


# ================================================================
# 13. TRAIN ONE MODEL
# ================================================================

def train_one_model(
    seed,
    depth,
    learning_rate,
    encoding="RYRZ",
    entangle=True
):

    set_seed(seed)

    print()
    print("-" * 100)
    print(
        f"Seed={seed} | "
        f"Depth={depth} | "
        f"LR={learning_rate} | "
        f"Encoding={encoding} | "
        f"Entangle={entangle}"
    )
    print("-" * 100)


    model = VQC(
        input_dim=N_FEATURES,
        n_qubits=N_QUBITS,
        depth=depth,
        encoding=encoding,
        entangle=entangle
    ).to(DEVICE)


    # ------------------------------------------------------------
    # ALL PARAMETERS MUST BE FLOAT32
    # ------------------------------------------------------------

    for p in model.parameters():

        p.data = p.data.float()


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=0.0
    )


    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            pos_weight,
            dtype=torch.float32,
            device=DEVICE
        )
    )


    best_state = None

    best_score = -np.inf

    best_epoch = 0

    patience_counter = 0


    # ------------------------------------------------------------
    # TRAINING
    # ------------------------------------------------------------

    for epoch in range(
        1,
        EPOCHS + 1
    ):

        model.train()

        # Shuffle development data

        permutation = torch.randperm(
            len(X_dev_t)
        )

        epoch_loss = 0.0

        n_batches = 0


        for start in range(
            0,
            len(X_dev_t),
            BATCH_SIZE
        ):

            idx = permutation[
                start:start + BATCH_SIZE
            ]

            xb = X_dev_t[idx]

            yb = y_dev_t[idx]


            optimizer.zero_grad(
                set_to_none=True
            )


            logits = model(xb)


            # ------------------------------------------------
            # CRITICAL SHAPE CHECK
            # ------------------------------------------------

            if logits.shape != yb.shape:

                raise RuntimeError(
                    "\nShape mismatch inside training:\n"
                    f"logits = {logits.shape}\n"
                    f"target = {yb.shape}\n"
                    "\nExpected both to be [batch]."
                )


            loss = criterion(
                logits,
                yb
            )


            if not torch.isfinite(loss):

                raise RuntimeError(
                    f"Non-finite loss at epoch {epoch}"
                )


            loss.backward()


            # Gradient clipping

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )


            optimizer.step()


            epoch_loss += loss.item()

            n_batches += 1


        # --------------------------------------------------------
        # VALIDATION
        # --------------------------------------------------------

        model.eval()

        with torch.no_grad():

            val_logits = []

            for start in range(
                0,
                len(X_val_t),
                BATCH_SIZE
            ):

                xb = X_val_t[
                    start:start + BATCH_SIZE
                ]

                out = model(xb)

                val_logits.append(
                    out.cpu()
                )

            val_logits = torch.cat(
                val_logits
            ).numpy()


        val_metrics, _, _ = calculate_metrics(
            y_val_np,
            val_logits
        )


        # Selection score:
        # prioritize balanced accuracy + F1 + ROC-AUC

        selection_score = (
            0.40 * val_metrics["Balanced_Accuracy"]
            + 0.30 * val_metrics["F1"]
            + 0.20 * val_metrics["ROC_AUC"]
            + 0.10 * val_metrics["PR_AUC"]
        )


        avg_loss = (
            epoch_loss /
            max(n_batches, 1)
        )


        if selection_score > best_score:

            best_score = selection_score

            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1


        # Print progress

        if (
            epoch == 1
            or epoch % 5 == 0
            or epoch == best_epoch
        ):

            print(
                f"Epoch {epoch:03d} | "
                f"Loss={avg_loss:.5f} | "
                f"Val BA={val_metrics['Balanced_Accuracy']:.4f} | "
                f"Val F1={val_metrics['F1']:.4f} | "
                f"Val AUC={val_metrics['ROC_AUC']:.4f}"
            )


        if patience_counter >= PATIENCE:

            print(
                f"Early stopping at epoch {epoch}"
            )

            break


    # ------------------------------------------------------------
    # RESTORE BEST MODEL
    # ------------------------------------------------------------

    if best_state is None:

        raise RuntimeError(
            "No valid model checkpoint was produced."
        )


    model.load_state_dict(
        best_state
    )

    model.eval()


    # ------------------------------------------------------------
    # FINAL VALIDATION
    # ------------------------------------------------------------

    with torch.no_grad():

        val_logits = []

        for start in range(
            0,
            len(X_val_t),
            BATCH_SIZE
        ):

            xb = X_val_t[
                start:start + BATCH_SIZE
            ]

            val_logits.append(
                model(xb).cpu()
            )

        val_logits = torch.cat(
            val_logits
        ).numpy()


    val_metrics, _, _ = calculate_metrics(
        y_val_np,
        val_logits
    )


    result = {

        "Input": "ANOVA12",

        "Model":
            f"VQC_RYRZ_D{depth}"
            f"_LR{str(learning_rate).replace('.', '')}",

        "Seed": seed,

        "Depth": depth,

        "Encoding": encoding,

        "Entangle": entangle,

        "Best_Epoch": best_epoch,

        "Accuracy":
            val_metrics["Accuracy"],

        "Balanced_Accuracy":
            val_metrics["Balanced_Accuracy"],

        "F1":
            val_metrics["F1"],

        "MCC":
            val_metrics["MCC"],

        "ROC_AUC":
            val_metrics["ROC_AUC"],

        "PR_AUC":
            val_metrics["PR_AUC"],

        "SelectionScore":
            best_score
    }


    return model, result


# ================================================================
# 14. RUN CONFIGURATIONS
# ================================================================

results = []

best_model = None

best_result = None

global_best_score = -np.inf


for depth in DEPTHS:

    for lr in LEARNING_RATES:

        for entangle in [True, False]:

            config_name = (
                f"RYRZ_D{depth}_"
                f"LR{str(lr).replace('.', '')}_"
                f"{'ENT' if entangle else 'NOENT'}"
            )

            print()
            print("=" * 100)
            print(
                "CONFIGURATION:",
                config_name
            )
            print("=" * 100)


            for seed in SEEDS:

                try:

                    model, result = train_one_model(
                        seed=seed,
                        depth=depth,
                        learning_rate=lr,
                        encoding="RYRZ",
                        entangle=entangle
                    )


                    results.append(
                        result
                    )


                    print(
                        f"✓ Completed seed {seed}"
                    )

                    print(
                        f"  BA  = "
                        f"{result['Balanced_Accuracy']:.4f}"
                    )

                    print(
                        f"  F1  = "
                        f"{result['F1']:.4f}"
                    )

                    print(
                        f"  AUC = "
                        f"{result['ROC_AUC']:.4f}"
                    )


                    if (
                        result["SelectionScore"]
                        > global_best_score
                    ):

                        global_best_score = (
                            result["SelectionScore"]
                        )

                        best_model = copy.deepcopy(
                            model
                        )

                        best_result = result.copy()


                except Exception as e:

                    print()
                    print(
                        f"FAILED {config_name} "
                        f"Seed {seed}"
                    )

                    print(
                        repr(e)
                    )


# ================================================================
# 15. RESULTS
# ================================================================

if len(results) == 0:

    raise RuntimeError(
        "All VQC configurations failed."
    )


results_df = (
    pd.DataFrame(results)
    .sort_values(
        "SelectionScore",
        ascending=False
    )
    .reset_index(drop=True)
)


print()
print("=" * 100)
print("HYBRID / VQC INTERNAL MODEL COMPARISON")
print("=" * 100)

print(
    results_df.to_string(
        index=False
    )
)


# ================================================================
# 16. BEST MODEL
# ================================================================

print()
print("=" * 100)
print("BEST INTERNAL VQC")
print("=" * 100)

for k, v in best_result.items():

    print(
        f"{k}: {v}"
    )


# ================================================================
# 17. FREEZE MODEL
# ================================================================

best_model.eval()

for param in best_model.parameters():

    param.requires_grad = False


print()
print("=" * 100)
print("VQC MODEL FROZEN")
print("=" * 100)

print("✓ Best model selected using validation")
print("✓ Development used for training")
print("✓ Validation used for model selection")
print("✓ Locked test NOT used")
print("✓ Model frozen")
print("✓ No optimizer retained")


# ================================================================
# 18. SAVE
# ================================================================

OUTPUT_DIR = (
    "/kaggle/working/"
    "phase21_vqc_outputs"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


results_path = os.path.join(
    OUTPUT_DIR,
    "phase21_vqc_model_comparison.csv"
)

checkpoint_path = os.path.join(
    OUTPUT_DIR,
    "phase21_vqc_best_model.pt"
)


results_df.to_csv(
    results_path,
    index=False
)


checkpoint = {

    "model_state_dict":
        best_model.state_dict(),

    "best_info":
        best_result,

    "input_dim":
        N_FEATURES,

    "n_qubits":
        N_QUBITS,

    "depth":
        best_result["Depth"],

    "encoding":
        best_result["Encoding"],

    "entangle":
        best_result["Entangle"]
}


torch.save(
    checkpoint,
    checkpoint_path
)


print()
print("=" * 100)
print("SAVED")
print("=" * 100)

print(
    "✓",
    results_path
)

print(
    "✓",
    checkpoint_path
)


# ================================================================
# 19. FINAL SANITY CHECK
# ================================================================

best_model.eval()

with torch.no_grad():

    sanity_output = best_model(
        X_val_t[:32]
    )


print()
print("=" * 100)
print("FINAL OUTPUT SHAPE CHECK")
print("=" * 100)

print(
    "Input:",
    X_val_t[:32].shape
)

print(
    "Output:",
    sanity_output.shape
)

assert sanity_output.shape == torch.Size([32])

print(
    "✓ 32 samples -> 32 predictions"
)

print(
    "✓ VQC shape problem fixed"
)

print()
print("=" * 100)
print("PHASE 21 VQC TRAINING COMPLETED")
print("=" * 100)

print(
    f"Best validation BA: "
    f"{best_result['Balanced_Accuracy']:.4f}"
)

print(
    f"Best validation F1: "
    f"{best_result['F1']:.4f}"
)

print(
    f"Best validation ROC-AUC: "
    f"{best_result['ROC_AUC']:.4f}"
)

PHASE 21 — CORRECTED VQC TRAINING
Device: cpu
✓ CPU-only
✓ CUDA will NOT be used

INPUT CHECK
X_dev: (3602, 12) float32
y_dev: (3602,) int64
X_val: (983, 12) float32
y_val: (983,) int64
✓ Development shapes consistent
✓ Validation shapes consistent
✓ ANOVA12 = 12 features

TORCH TENSOR CHECK
X_dev: torch.Size([3602, 12]) torch.float32
y_dev: torch.Size([3602]) torch.float32
X_val: torch.Size([983, 12]) torch.float32
y_val: torch.Size([983]) torch.float32

MODEL SHAPE TEST
Input : torch.Size([8, 12])
Output: torch.Size([8])
✓ VQC output shape is correct
✓ One prediction per sample

CLASS BALANCE
Class 0: 2348
Class 1: 1254
Positive weight: 1.872408293460925

CONFIGURATION: RYRZ_D2_LR0001_ENT

----------------------------------------------------------------------------------------------------
Seed=123 | Depth=2 | LR=0.001 | Encoding=RYRZ | Entangle=True
----------------------------------------------------------------------------------------------------
Epoch 001 | Loss=0.67835 | Val BA=0

KeyboardInterrupt: 

In [32]:
# ================================================================
# SAVE ALL COMPLETED RESULTS + CURRENT BEST MODEL
# ================================================================

import os
import pandas as pd
import torch

OUTPUT_DIR = "/kaggle/working/phase21_vqc_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------
# 1. SAVE ALL COMPLETED RUNS
# ------------------------------------------------

results_df = (
    pd.DataFrame(results)
    .sort_values(
        "SelectionScore",
        ascending=False
    )
    .reset_index(drop=True)
)

results_path = os.path.join(
    OUTPUT_DIR,
    "phase21_vqc_completed_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

print("✓ Results saved:")
print(results_path)


# ------------------------------------------------
# 2. SAVE BEST MODEL ALREADY COMPLETED
# ------------------------------------------------

if best_model is not None and best_result is not None:

    checkpoint_path = os.path.join(
        OUTPUT_DIR,
        "phase21_vqc_best_model_so_far.pt"
    )

    checkpoint = {
        "model_state_dict":
            best_model.state_dict(),

        "best_info":
            best_result,

        "input_dim":
            N_FEATURES,

        "n_qubits":
            N_QUBITS,

        "depth":
            best_result["Depth"],

        "encoding":
            best_result["Encoding"],

        "entangle":
            best_result["Entangle"]
    }

    torch.save(
        checkpoint,
        checkpoint_path
    )

    print("✓ Best model saved:")
    print(checkpoint_path)

else:

    print("⚠ No best model available.")


# ------------------------------------------------
# 3. DISPLAY BEST RESULT
# ------------------------------------------------

print()
print("=" * 100)
print("BEST COMPLETED VQC RESULT")
print("=" * 100)

if best_result is not None:

    for k, v in best_result.items():
        print(f"{k}: {v}")

print()
print("=" * 100)
print("SAVE COMPLETED")
print("=" * 100)

✓ Results saved:
/kaggle/working/phase21_vqc_outputs/phase21_vqc_completed_results.csv
✓ Best model saved:
/kaggle/working/phase21_vqc_outputs/phase21_vqc_best_model_so_far.pt

BEST COMPLETED VQC RESULT
Input: ANOVA12
Model: VQC_RYRZ_D2_LR0003
Seed: 2024
Depth: 2
Encoding: RYRZ
Entangle: False
Best_Epoch: 37
Accuracy: 0.9928789420142421
Balanced_Accuracy: 0.9911167512690355
F1: 0.9910371318822023
MCC: 0.9852396678025186
ROC_AUC: 0.9999655270483396
PR_AUC: 0.9999488057408779
SelectionScore: 0.9937458260560307

SAVE COMPLETED


In [34]:
# ================================================================
# PHASE 21 — FINAL LOCKED TEST EVALUATION
# ================================================================

import os
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------
# CPU ONLY
# ------------------------------------------------

DEVICE = torch.device("cpu")

print("=" * 100)
print("PHASE 21 — FINAL LOCKED TEST EVALUATION")
print("=" * 100)
print("Device:", DEVICE)


# ================================================================
# 1. CHECK LOCKED TEST DATA
# ================================================================

required = [
    "X_test",
    "y_test"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:

    raise RuntimeError(
        "Missing variables: "
        + str(missing)
        + "\nExpected X_test and y_test."
    )


# ================================================================
# 2. CONVERT TEST DATA
# ================================================================

X_test_np = np.asarray(
    X_test,
    dtype=np.float32
)

y_test_np = np.asarray(
    y_test,
    dtype=np.int64
).reshape(-1)


print()
print("=" * 100)
print("LOCKED TEST DATA CHECK")
print("=" * 100)

print(
    "X_test:",
    X_test_np.shape,
    X_test_np.dtype
)

print(
    "y_test:",
    y_test_np.shape,
    y_test_np.dtype
)


# ================================================================
# 3. SANITY CHECK
# ================================================================

if X_test_np.ndim != 2:

    raise RuntimeError(
        f"X_test must be 2D. "
        f"Got {X_test_np.shape}"
    )


if X_test_np.shape[1] != 12:

    raise RuntimeError(
        f"Expected ANOVA12 = 12 features. "
        f"Got {X_test_np.shape[1]}"
    )


if len(X_test_np) != len(y_test_np):

    raise RuntimeError(
        f"Test mismatch: "
        f"X={len(X_test_np)}, "
        f"y={len(y_test_np)}"
    )


print("✓ Locked test shapes correct")
print("✓ ANOVA12 confirmed")


# ================================================================
# 4. TORCH TEST TENSOR
# ================================================================

X_test_t = torch.tensor(
    X_test_np,
    dtype=torch.float32,
    device=DEVICE
)

print()
print("X_test tensor:", X_test_t.shape)
print("✓ Test tensor ready")


# ================================================================
# 5. QUANTUM SETTINGS
# ================================================================

N_QUBITS = 4
N_FEATURES = 12

BATCH_SIZE = 32


# ================================================================
# 6. FEATURE → QUANTUM PROJECTION
# ================================================================

class QuantumProjection(nn.Module):

    def __init__(
        self,
        input_dim,
        n_qubits
    ):

        super().__init__()

        self.projection = nn.Sequential(

            nn.Linear(
                input_dim,
                32
            ),

            nn.LayerNorm(32),

            nn.Tanh(),

            nn.Linear(
                32,
                n_qubits
            ),

            nn.Tanh()
        )


    def forward(self, x):

        angles = self.projection(x)

        return angles * np.pi


# ================================================================
# 7. QUANTUM CIRCUIT
# ================================================================

def make_qnode(
    n_qubits=4,
    depth=2,
    encoding="RYRZ",
    entangle=True
):

    dev = qml.device(
        "default.qubit",
        wires=n_qubits
    )


    @qml.qnode(
        dev,
        interface="torch"
    )

    def circuit(
        inputs,
        weights
    ):

        # --------------------------------------------------------
        # INPUT ENCODING
        # --------------------------------------------------------

        if encoding == "RYRZ":

            for q in range(n_qubits):

                qml.RY(
                    inputs[q],
                    wires=q
                )

                qml.RZ(
                    inputs[q],
                    wires=q
                )


        elif encoding == "RY":

            for q in range(n_qubits):

                qml.RY(
                    inputs[q],
                    wires=q
                )


        else:

            raise ValueError(
                f"Unknown encoding: {encoding}"
            )


        # --------------------------------------------------------
        # VARIATIONAL LAYERS
        # --------------------------------------------------------

        for d in range(depth):

            for q in range(n_qubits):

                qml.RY(
                    weights[d, q, 0],
                    wires=q
                )

                qml.RZ(
                    weights[d, q, 1],
                    wires=q
                )


            if entangle:

                for q in range(
                    n_qubits - 1
                ):

                    qml.CNOT(
                        wires=[
                            q,
                            q + 1
                        ]
                    )


        # --------------------------------------------------------
        # SINGLE OUTPUT
        # --------------------------------------------------------

        return qml.expval(
            qml.PauliZ(0)
        )


    return circuit


# ================================================================
# 8. VQC MODEL
# ================================================================

class VQC(nn.Module):

    def __init__(
        self,
        input_dim=12,
        n_qubits=4,
        depth=2,
        encoding="RYRZ",
        entangle=True
    ):

        super().__init__()

        self.input_dim = input_dim

        self.n_qubits = n_qubits

        self.depth = depth

        self.encoding = encoding

        self.entangle = entangle


        # --------------------------------------------------------
        # CLASSICAL PROJECTION
        # --------------------------------------------------------

        self.projection = QuantumProjection(
            input_dim,
            n_qubits
        )


        # --------------------------------------------------------
        # QUANTUM CIRCUIT
        # --------------------------------------------------------

        self.qnode = make_qnode(
            n_qubits=n_qubits,
            depth=depth,
            encoding=encoding,
            entangle=entangle
        )


        # --------------------------------------------------------
        # QUANTUM PARAMETERS
        # --------------------------------------------------------

        self.q_weights = nn.Parameter(

            0.05 * torch.randn(

                depth,
                n_qubits,
                2,

                dtype=torch.float32
            )
        )


        # --------------------------------------------------------
        # OUTPUT CALIBRATION
        # --------------------------------------------------------

        self.output_scale = nn.Parameter(

            torch.tensor(
                1.0,
                dtype=torch.float32
            )
        )


        self.output_bias = nn.Parameter(

            torch.tensor(
                0.0,
                dtype=torch.float32
            )
        )


    def forward(self, x):

        x = x.float()

        angles = self.projection(x)

        batch_outputs = []


        for i in range(
            angles.shape[0]
        ):

            q_out = self.qnode(
                angles[i],
                self.q_weights
            )

            batch_outputs.append(
                q_out
            )


        q_out = torch.stack(
            batch_outputs
        ).float()


        logits = (
            self.output_scale * q_out
            + self.output_bias
        )


        return logits


# ================================================================
# 9. CHECKPOINT PATH
# ================================================================

checkpoint_path = (
    "/kaggle/working/"
    "phase21_vqc_outputs/"
    "phase21_vqc_best_model_so_far.pt"
)


print()
print("=" * 100)
print("CHECKPOINT")
print("=" * 100)

print(
    "Path:",
    checkpoint_path
)


if not os.path.exists(
    checkpoint_path
):

    raise FileNotFoundError(
        "Checkpoint not found:\n"
        + checkpoint_path
    )


print("✓ Checkpoint exists")


# ================================================================
# 10. LOAD CHECKPOINT
# ================================================================
#
# IMPORTANT:
#
# PyTorch 2.6 changed torch.load()
# default behavior to weights_only=True.
#
# Our checkpoint contains numpy/scalar metadata,
# so we explicitly use weights_only=False.
#
# This is appropriate because the checkpoint was
# generated by our own trusted Kaggle training code.
# ================================================================

checkpoint = torch.load(

    checkpoint_path,

    map_location=DEVICE,

    weights_only=False
)


print("✓ Checkpoint loaded successfully")


# ================================================================
# 11. DISPLAY SAVED CONFIGURATION
# ================================================================

print()
print("=" * 100)
print("SAVED VQC CONFIGURATION")
print("=" * 100)

best_info = checkpoint[
    "best_info"
]

for key, value in best_info.items():

    print(
        f"{key}: {value}"
    )


# ================================================================
# 12. RECREATE EXACT BEST MODEL
# ================================================================

saved_input_dim = checkpoint[
    "input_dim"
]

saved_n_qubits = checkpoint[
    "n_qubits"
]

saved_depth = checkpoint[
    "depth"
]

saved_encoding = checkpoint[
    "encoding"
]

saved_entangle = checkpoint[
    "entangle"
]


print()
print("=" * 100)
print("RECREATING BEST VQC")
print("=" * 100)

print(
    "Input dimension:",
    saved_input_dim
)

print(
    "Qubits:",
    saved_n_qubits
)

print(
    "Depth:",
    saved_depth
)

print(
    "Encoding:",
    saved_encoding
)

print(
    "Entanglement:",
    saved_entangle
)


model = VQC(

    input_dim=saved_input_dim,

    n_qubits=saved_n_qubits,

    depth=saved_depth,

    encoding=saved_encoding,

    entangle=saved_entangle

).to(DEVICE)


# ================================================================
# 13. LOAD TRAINED WEIGHTS
# ================================================================

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


model.eval()


print()
print("✓ Model architecture recreated")
print("✓ Trained weights loaded")
print("✓ Model set to evaluation mode")


# ================================================================
# 14. TEST OUTPUT SHAPE CHECK
# ================================================================

print()
print("=" * 100)
print("TEST OUTPUT SHAPE CHECK")
print("=" * 100)


with torch.no_grad():

    test_sample_output = model(
        X_test_t[:8]
    )


print(
    "Input:",
    X_test_t[:8].shape
)

print(
    "Output:",
    test_sample_output.shape
)


if test_sample_output.shape != torch.Size([8]):

    raise RuntimeError(
        "VQC output shape is incorrect:\n"
        + str(test_sample_output.shape)
    )


print(
    "✓ 8 samples → 8 predictions"
)


# ================================================================
# 15. LOCKED TEST INFERENCE
# ================================================================

print()
print("=" * 100)
print("RUNNING LOCKED TEST EVALUATION")
print("=" * 100)


model.eval()

test_logits_list = []


with torch.no_grad():

    for start in range(

        0,

        len(X_test_t),

        BATCH_SIZE

    ):

        xb = X_test_t[
            start:
            start + BATCH_SIZE
        ]


        out = model(
            xb
        )


        test_logits_list.append(
            out.cpu()
        )


test_logits = torch.cat(
    test_logits_list
).numpy()


print(
    "✓ Test inference completed"
)

print(
    "Number of predictions:",
    len(test_logits)
)


# ================================================================
# 16. CONVERT LOGITS → PROBABILITIES
# ================================================================

test_probs = 1.0 / (

    1.0
    + np.exp(
        -np.clip(
            test_logits,
            -30,
            30
        )
    )
)


# ================================================================
# 17. PREDICTIONS
# ================================================================

test_preds = (

    test_probs >= 0.5

).astype(int)


# ================================================================
# 18. FINAL TEST METRICS
# ================================================================

accuracy = accuracy_score(
    y_test_np,
    test_preds
)


balanced_accuracy = (
    balanced_accuracy_score(
        y_test_np,
        test_preds
    )
)


f1 = f1_score(
    y_test_np,
    test_preds,
    zero_division=0
)


mcc = matthews_corrcoef(
    y_test_np,
    test_preds
)


try:

    roc_auc = roc_auc_score(
        y_test_np,
        test_probs
    )

except:

    roc_auc = np.nan


try:

    pr_auc = average_precision_score(
        y_test_np,
        test_probs
    )

except:

    pr_auc = np.nan


# ================================================================
# 19. PRINT FINAL LOCKED TEST RESULTS
# ================================================================

print()
print("=" * 100)
print("FINAL LOCKED TEST RESULTS — VQC")
print("=" * 100)

print(
    f"Accuracy           : {accuracy:.6f}"
)

print(
    f"Balanced Accuracy  : {balanced_accuracy:.6f}"
)

print(
    f"F1 Score           : {f1:.6f}"
)

print(
    f"MCC                : {mcc:.6f}"
)

print(
    f"ROC-AUC            : {roc_auc:.6f}"
)

print(
    f"PR-AUC             : {pr_auc:.6f}"
)


# ================================================================
# 20. CONFUSION MATRIX
# ================================================================

cm = confusion_matrix(
    y_test_np,
    test_preds
)


print()
print("=" * 100)
print("CONFUSION MATRIX")
print("=" * 100)

print(
    cm
)


# ================================================================
# 21. CLASSIFICATION REPORT
# ================================================================

print()
print("=" * 100)
print("CLASSIFICATION REPORT")
print("=" * 100)

print(
    classification_report(
        y_test_np,
        test_preds,
        digits=4,
        zero_division=0
    )
)


# ================================================================
# 22. TEST CLASS DISTRIBUTION
# ================================================================

print()
print("=" * 100)
print("TEST CLASS DISTRIBUTION")
print("=" * 100)

print(
    "Actual Class 0:",
    np.sum(y_test_np == 0)
)

print(
    "Actual Class 1:",
    np.sum(y_test_np == 1)
)

print(
    "Predicted Class 0:",
    np.sum(test_preds == 0)
)

print(
    "Predicted Class 1:",
    np.sum(test_preds == 1)
)


# ================================================================
# 23. SAVE FINAL TEST RESULTS
# ================================================================

OUTPUT_DIR = (
    "/kaggle/working/"
    "phase21_vqc_outputs"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


test_results = {

    "Input":
        "ANOVA12",

    "Model":
        best_info["Model"],

    "Seed":
        best_info["Seed"],

    "Depth":
        best_info["Depth"],

    "Encoding":
        best_info["Encoding"],

    "Entangle":
        best_info["Entangle"],

    "Best_Epoch":
        best_info["Best_Epoch"],

    "Test_Accuracy":
        accuracy,

    "Test_Balanced_Accuracy":
        balanced_accuracy,

    "Test_F1":
        f1,

    "Test_MCC":
        mcc,

    "Test_ROC_AUC":
        roc_auc,

    "Test_PR_AUC":
        pr_auc
}


test_results_df = pd.DataFrame(
    [test_results]
)


test_results_path = os.path.join(

    OUTPUT_DIR,

    "phase21_vqc_final_locked_test_results.csv"
)


test_results_df.to_csv(

    test_results_path,

    index=False
)


print()
print("=" * 100)
print("FINAL TEST RESULTS SAVED")
print("=" * 100)

print(
    "✓",
    test_results_path
)


# ================================================================
# 24. FINAL SUMMARY
# ================================================================

print()
print("=" * 100)
print("PHASE 21 — VQC FINAL TEST EVALUATION COMPLETED")
print("=" * 100)

print()
print("FINAL LOCKED TEST PERFORMANCE")
print("----------------------------------------")

print(
    f"Accuracy          : {accuracy:.4f}"
)

print(
    f"Balanced Accuracy : {balanced_accuracy:.4f}"
)

print(
    f"F1                : {f1:.4f}"
)

print(
    f"MCC               : {mcc:.4f}"
)

print(
    f"ROC-AUC           : {roc_auc:.4f}"
)

print(
    f"PR-AUC             : {pr_auc:.4f}"
)

print()
print("✓ Locked test was used only for final evaluation")
print("✓ No retraining performed")
print("✓ No hyperparameter tuning performed on test")
print("✓ Best completed VQC checkpoint evaluated")
print()
print("=" * 100)

PHASE 21 — FINAL LOCKED TEST EVALUATION
Device: cpu

LOCKED TEST DATA CHECK
X_test: (1290, 12) float32
y_test: (1290,) int64
✓ Locked test shapes correct
✓ ANOVA12 confirmed

X_test tensor: torch.Size([1290, 12])
✓ Test tensor ready

CHECKPOINT
Path: /kaggle/working/phase21_vqc_outputs/phase21_vqc_best_model_so_far.pt
✓ Checkpoint exists
✓ Checkpoint loaded successfully

SAVED VQC CONFIGURATION
Input: ANOVA12
Model: VQC_RYRZ_D2_LR0003
Seed: 2024
Depth: 2
Encoding: RYRZ
Entangle: False
Best_Epoch: 37
Accuracy: 0.9928789420142421
Balanced_Accuracy: 0.9911167512690355
F1: 0.9910371318822023
MCC: 0.9852396678025186
ROC_AUC: 0.9999655270483396
PR_AUC: 0.9999488057408779
SelectionScore: 0.9937458260560307

RECREATING BEST VQC
Input dimension: 12
Qubits: 4
Depth: 2
Encoding: RYRZ
Entanglement: False

✓ Model architecture recreated
✓ Trained weights loaded
✓ Model set to evaluation mode

TEST OUTPUT SHAPE CHECK
Input: torch.Size([8, 12])
Output: torch.Size([8])
✓ 8 samples → 8 predictions

RUN

In [24]:
# ============================================================
# INSTALL REQUIRED QUANTUM LIBRARIES
# ============================================================

!pip install -q pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 49.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 49.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 88.4 MB/s eta 0:00:00:00:0100:01
